
# Chapter 5 - Flow Matching and Similar Techniques

After this chapter you should be able to:

1. Understand that the general idea and intuition behind **flow matching** 
2. Describe how flowmatching **obviates** estimation issues faced by **VAE** and **Diffusion**
3. Understand the **inituition** behind flow matching
4. Describe and **rationalize** the use of various **neural networks** to implement **flow mathing**
5. Understand **classifier guided** and **classifier free** ideas applied to **flow matching**
6. Describe and implement **latent flow matching**
7. Understand and describe how **Optimtal Transport** is applied to **flow matching**
8. Understand the **idea** of **consistency** models 
9. Describe why **consistency** models are difficult to train
10. Describe the **various** design dimensions or **variants** of **flow matching** solutions 
11. Building towards a **transport-based view**
12. Some **publishable ideas**

## License

**Text, figures, and explanations:**  
© 2026 Imran Zualkernan. Licensed under **CC BY 4.0**.

**Code cells:**  
© 2026 Imran Zualkernan. Licensed under the **MIT License**.

You are free to reuse, modify, and redistribute with attribution.

## Introduction to Flow Matching

### The Basic Idea

Flow matching learns a **time-dependent velocity field**
$
v_\theta(x,t)
$
that tells you **how to move particles** from a simple base distribution (noise) to a complex target distribution (data).

- Start distribution (easy to sample):
$
x_0 \sim p_0(x)\quad \text{(e.g., } \mathcal{N}(0,I)\text{)}.
$

- Target distribution (data):
$
x_1 \sim p_1(x)\quad \text{(unknown density, but we have samples)}.
$

If we can learn a good velocity field, then we can **sample from the data** by:
1) sample $$x_0 \sim p_0$$  
2) integrate an ODE forward in time:
$
\frac{dx}{dt} = v_\theta(x,t),\qquad t:0 \to 1,
$
to obtain $x_1 \approx x(1)$ which should follow $p_1$.

### The Mathematics

If particles follow the ODE
$
\frac{dx}{dt} = v(x,t),
$
the density $p_t(x)$ evolves according to the **continuity equation**
$
\frac{\partial p_t(x)}{\partial t} + \nabla \cdot \big(p_t(x)\,v(x,t)\big) = 0.
$

So if we learn a velocity field $v_\theta$ that matches the **true** velocity field generating a bridge from $p_0$ to $p_1$, then integrating the ODE transports samples correctly.

Flow matching avoids directly computing densities or likelihoods by learning **velocities** from samples.

### How flow matching defines “the correct velocity”

Flow matching chooses a **coupling** between base samples and data samples:
- Sample $x_0 \sim p_0$
- Sample $x_1 \sim p_1$
- Pair them via some rule (often independent pairing in simple demos, or an optimal-transport coupling in stronger methods)

Then define an **intermediate state** $x_t$ along a path from $x_0$ to $x_1$.

#### The simplest path: linear interpolation
Define:
$
x_t = (1-t)\,x_0 + t\,x_1,\qquad t\in[0,1].
$

For this path, the **true velocity along the path** is:
$
\frac{d x_t}{dt} = x_1 - x_0.
$

So for any sampled triplet $(x_0, x_1, t)$, we can compute:
- the point $x_t$ where we “query” the velocity field
- the target velocity $u_t$ we want the network to output at that point

Specifically:
$
u_t = x_1 - x_0.
$

Then we train:
$
v_\theta(x_t,t) \approx u_t.
$

### Simple Example

Consider a 1D example:

- A base (noise) sample: $x_0 = -1.5$
- A target (data) sample: $x_1 = 2.0$
- Pick a time: $t = 0.3$

**Point on the path**
$
x_t = (1-0.3)(-1.5) + 0.3(2.0) = 0.7(-1.5) + 0.6 = -1.05 + 0.6 = -0.45.
$

**Velocity target**
$
u_t = x_1 - x_0 = 2.0 - (-1.5) = 3.5.
$

So the training supervision says:

> “When the particle is at $x=-0.45$ at time $t=0.3$, the correct velocity (for this paired path) is $3.5$.”

During training we sample many pairs and times, building a dataset of constraints:
$
\big(x_t, t\big)\ \mapsto\ u_t.
$

The learned field $v_\theta(x,t)$ becomes a smooth function that approximates the average correct direction of transport at each space-time location.

### Learning stage and training objective

#### Flow matching regression loss
A standard flow matching objective is:
$$
\mathcal{L}(\theta)
=
\mathbb{E}_{x_0\sim p_0,\ x_1\sim p_1,\ t\sim \mathrm{Uniform}[0,1]}
\left[
\left\| v_\theta(x_t,t) - u_t \right\|_2^2
\right],
$$
where
$
x_t = (1-t)x_0 + t x_1,\qquad u_t = x_1 - x_0.
$

This is supervised learning of a vector field.

#### Sampling stage (ODE integration)

After training, generate a sample from the target by:

1. Sample base noise:
$
x(0) = x_0 \sim p_0.
$

2. Integrate:
$
\frac{dx}{dt} = v_\theta(x,t),\qquad t:0\to 1.
$

3. Output:
$
\hat{x}_1 = x(1).
$

#### Why integration gives the point we need
In the ideal case where $v_\theta = v^\star$ (the true velocity field), the ODE flow map $\Phi_{0\to 1}$ transports the distribution:
$$
x_1 = \Phi_{0\to 1}(x_0),
\qquad
x_0 \sim p_0
\quad\Rightarrow\quad
x_1 \sim p_1.
$$

Even when $v_\theta$ is approximate, numerical integration often produces high-quality samples.

### Pseudocode with Mathematics

#### Learning (train the velocity field)
**Inputs:** dataset samples from $p_1$, base sampler for $p_0$, model $v_\theta(x,t)$.

Repeat SGD steps:

1. Sample:
$
x_0 \sim p_0,\qquad x_1 \sim p_1,\qquad t \sim \mathrm{Uniform}[0,1].
$

2. Build bridge point:
$
x_t = (1-t)x_0 + t x_1.
$

3. Build velocity target:
$
u_t = x_1 - x_0.
$

4. Loss:
$
\mathcal{L} = \left\| v_\theta(x_t,t) - u_t \right\|_2^2.
$

5. Update:
$
\theta \leftarrow \theta - \eta\nabla_\theta \mathcal{L}.
$

#### Sampling (integrate the learned ODE)
**Inputs:** trained $v_\theta$, ODE solver with step size $\Delta t$.

1. Initialize:
$
x \leftarrow x_0,\quad x_0\sim p_0,\quad t\leftarrow 0.
$

2. For $n=0,1,\dots,N-1$ with $\Delta t = 1/N$ (Euler as example):
$$
x \leftarrow x + \Delta t\, v_\theta(x,t),
\qquad
t \leftarrow t + \Delta t.
$$

3. Output:
$
\hat{x}_1 \leftarrow x.
$

### Extentions 

- **Coupling:** Instead of pairing $x_0$ and $x_1$ independently, you can use an optimal-transport coupling to reduce crossing paths and improve quality.
- **Path:** You can use non-linear bridges (e.g., variance-preserving stochastic interpolation) where $u_t$ depends on $t$.
- **Solver:** Use better ODE solvers (Heun / RK4 / adaptive) for better samples at similar compute.

### Summary 

Flow matching learns $v_\theta(x,t)$ by supervising it with “correct” velocities on synthetic paths from $p_0$ to $p_1$, then generates samples by integrating:
$$
\frac{dx}{dt} = v_\theta(x,t)
$$
from $t=0$ (noise) to $t=1$(data).

## The continuity equation

The **continuity equation** is the mathematical statement of a simple idea:

> **Probability mass can move around, but it can’t appear out of nowhere or vanish.**

It’s the same conservation law used for fluids (“mass conservation”), except here the “fluid” is **probability density**.

Let $p_t(x)$ be the probability density of a random variable $x\in\mathbb{R}^d$ at time $t$.  
Let $v(x,t)\in\mathbb{R}^d$ be a velocity (wind) field that moves particles.

Then the continuity equation is:

$$
\frac{\partial p_t(x)}{\partial t} + \nabla \cdot \big(p_t(x)\,v(x,t)\big) = 0.
$$

- $p_t(x)$ = “how much probability is at location $x$ at time $t$”
- $p_t(x)\,v(x,t)$ = **probability flux** (how fast probability is flowing through space)
- $\nabla\cdot(\cdot)$ = divergence, measuring **net outflow** from a tiny region

### Definition of divergence

For a vector field $F(x) = (F_1(x),\dots,F_d(x))\in\mathbb{R}^d$, the **divergence** is the scalar field:

$$
\nabla\cdot F(x) = \sum_{i=1}^d \frac{\partial F_i(x)}{\partial x_i}.
$$

Intuition:

- $\nabla\cdot F(x) > 0$ means the field is *expanding outward* near $x$ (net outflow).
- $\nabla\cdot F(x) < 0$ means the field is *converging inward* near $x$ (net inflow).

In the continuity equation, we use $F(x,t)=p_t(x)\,v(x,t)$.

### Derivation

Pick any region $A \subset \mathbb{R}^d$.  
The total probability inside $A$ at time $t$ is:

$$
P_t(A) = \int_A p_t(x)\,dx.
$$

**Conservation idea:** the only way $P_t(A)$ can change is if probability **flows across the boundary** $\partial A$.

Let $n(x)$ be the outward unit normal on $\partial A$. The outward flux through the boundary is:

$$
\int_{\partial A} p_t(x)\,v(x,t)\cdot n(x)\,dS.
$$

So conservation says:

$$
\frac{d}{dt}\int_A p_t(x)\,dx
=
-\int_{\partial A} p_t(x)\,v(x,t)\cdot n(x)\,dS.
$$

Now apply the divergence theorem:

$$
\int_{\partial A} F(x)\cdot n(x)\,dS = \int_A \nabla\cdot F(x)\,dx,
\quad \text{with } F(x)=p_t(x)\,v(x,t).
$$

So:

$$
\frac{d}{dt}\int_A p_t(x)\,dx
=
-\int_A \nabla\cdot\big(p_t(x)\,v(x,t)\big)\,dx.
$$

Bring the derivative inside the integral:

$$
\int_A \frac{\partial p_t(x)}{\partial t}\,dx
=
-\int_A \nabla\cdot\big(p_t(x)\,v(x,t)\big)\,dx.
$$

Since this holds for **every** region $A$, the integrands must match:

$$
\frac{\partial p_t(x)}{\partial t} + \nabla \cdot \big(p_t(x)\,v(x,t)\big) = 0.
$$

That’s the continuity equation.

### Intuition

Look at a tiny neighborhood around $x$.

- If $\nabla\cdot(p_tv) > 0$, there is **net outflow**: more probability leaves than enters, so $p_t(x)$ must **decrease**.
- If $\nabla\cdot(p_tv) < 0$, there is **net inflow**: probability accumulates, so $p_t(x)$ must **increase**.

So the equation:

$$
\frac{\partial p_t}{\partial t} = -\nabla\cdot(p_tv)
$$

literally says:

> **Density increases where flow converges, and decreases where flow diverges.**

### Relevance for flow matching / generative ODEs

Flow matching defines sampling by the ODE:

$\frac{dx}{dt} = v_\theta(x,t).$

That ODE moves **particles**. The continuity equation tells you what happens to the **distribution** of those particles.

So if $v_\theta$ is learned correctly, the density $p_t$ that evolves under this PDE will satisfy:

- $p_{t=0} = p_0$ (noise)
- $p_{t=1} = p_1$ (data)

Meaning: integrating the ODE transports the whole probability mass from $p_0$ to $p_1$.

### Trajectory-level View

Along a particle path $x(t)$ following $dx/dt=v(x,t)$, the density satisfies:

$$
\frac{d}{dt}\log p_t(x(t)) = -\nabla\cdot v(x(t),t),
$$

which is a trajectory-level view of how the flow compresses or expands probability.

### Why the Euler update works for integrating $\frac{dx}{dt}=v(x,t)$

#### The goal: follow the wind field

In flow matching, sampling solves the ODE:
$$
\frac{dx}{dt} = v_\theta(x,t), \qquad t\in[0,1],
$$
starting from $x(0)\sim p_0$ and ending at $x(1)$.

We cannot usually solve this ODE in closed form, so we approximate the trajectory numerically.

#### Where Euler’s formula comes from (first-order Taylor expansion)

Consider the true solution $x(t)$. For a small time step $\Delta t$, expand $x(t+\Delta t)$ with a Taylor series:

$$
x(t+\Delta t) = x(t) + \Delta t\,\frac{dx}{dt}(t) + \frac{(\Delta t)^2}{2}\,\frac{d^2x}{dt^2}(\xi)
$$

for some $\xi\in(t,t+\Delta t)$.

Because the ODE says:
$$
\frac{dx}{dt}(t)=v_\theta(x(t),t),
$$
we get:
$$
x(t+\Delta t)
=
x(t) + \Delta t\,v_\theta(x(t),t) \;+\; O((\Delta t)^2).
$$

If $\Delta t$ is small, the $O((\Delta t)^2)$ term is small, so a natural approximation is:

$$
x(t+\Delta t)\approx x(t) + \Delta t\,v_\theta(x(t),t).
$$

This is exactly the **(forward) Euler method**.

#### The Euler update rule (discrete integration)

Choose $N$ steps and let:
$$
\Delta t = \frac{1}{N},\qquad t_n = n\Delta t.
$$

Euler integration applies the update:
$$
x_{n+1} = x_n + \Delta t\,v_\theta(x_n,t_n),
\qquad n=0,1,\dots,N-1,
$$
with $$x_0 = x(0)$$.

Interpretation:

- At time $t_n$, look at the wind (velocity) at your current location $x_n$.
- Assume that wind is roughly constant for the next tiny interval $[t_n,t_{n+1}]$.
- Move by “wind × time” to get $x_{n+1}$.

#### Why this matches the integral definition of the solution

The exact solution satisfies:
$$
x(1) = x(0) + \int_0^1 v_\theta(x(t),t)\,dt.
$$

Euler replaces the integral by a Riemann sum approximation:
$$
\int_0^1 v_\theta(x(t),t)\,dt
\approx
\sum_{n=0}^{N-1} v_\theta(x_n,t_n)\,\Delta t.
$$

So Euler’s method is literally:

$$
x_N
=
x_0 + \sum_{n=0}^{N-1} v_\theta(x_n,t_n)\,\Delta t
\approx
x(1).
$$

As $N\to\infty$ (so $\Delta t\to 0$), the Riemann sum converges to the integral under standard smoothness conditions.


#### In flow matching sampling (what this means practically)

Given trained $v_\theta$:

1. Sample $x_0\sim p_0$.
2. Run Euler (or a higher-order solver) to approximate the ODE path.
3. Output $x_N \approx x(1)$.

Euler works because it is a **first-order** approximation to the true continuous-time motion: it follows the wind direction over many tiny steps, and the total motion approximates the time integral of the wind.

> If you need better accuracy with the same number of steps, use a higher-order solver (e.g., Heun/RK2 or RK4), but Euler is the simplest and often sufficient for demonstrations.

## $v_\theta$ is a vector

In flow matching, $v_\theta(x,t)$ is a **velocity vector**.

For data points $x\in\mathbb{R}^d$, the model is a **vector field**:
$$
v_\theta:\ \mathbb{R}^d \times [0,1] \to \mathbb{R}^d,
\qquad
v_\theta(x,t) = \begin{bmatrix} v_{\theta,1}(x,t) \\ \vdots \\ v_{\theta,d}(x,t) \end{bmatrix}.
$$
- The **direction** of $v_\theta(x,t)$ tells the particle which way to move in space.
- The **magnitude** (speed) is the vector norm:
$$
\|v_\theta(x,t)\|_2,
$$
which tells how fast it moves.

The particle trajectory follows the ODE:
$
\frac{dx}{dt} = v_\theta(x,t),
$
so each component of the vector controls motion along one coordinate axis.

**Special case (1D):** when $d=1$, the “vector” reduces to a scalar (signed speed):
$
v_\theta(x,t)\in\mathbb{R},
$
where the sign indicates direction (left/right) and the absolute value indicates magnitude (speed).

## Flow Matching Intuition

### The wind metaphor

Imagine a huge field full of tiny balloons (particles).  
At **time** $t=0$ the balloons are scattered according to an easy distribution $p_0$ (like “random noise”).  
At **time** $t=1$ we want the balloons to end up scattered like our data distribution $p_1$.

Flow matching learns a **wind field**:

$
v_\theta(x,t)
$

- $x$ is the balloon’s current position.
- $t$ is the time.
- $v_\theta(x,t)$ tells you **which way to blow** (direction) and **how hard** (speed).

If the wind field is correct, each balloon moves by obeying the ODE:

$
\frac{dx}{dt} = v_\theta(x,t),\qquad t\in[0,1].
$

So sampling is:

1. Throw balloons randomly: $x(0)\sim p_0$  
2. Turn on the learned wind: integrate to $t=1$. Numerical **integration** simply adds up these infinitesimal pushes:

  $
    x(t+\Delta t)\approx x(t)+\Delta t\,v_\theta(x(t),t),
  $

and in the limit,

  $
    x(1)=x(0)+\int_0^1 v_\theta(x(t),t)\,dt,
  $
so the morphing is completed by accumulating the wind’s effect over time.

3. The final positions $x(1)$ follow $p_1$

### What flow matching learns (wind supervision)

To learn the wind, flow matching creates “training stories”:

- Pick a starting balloon position $x_0 \sim p_0$  
- Pick a target balloon position $x_1 \sim p_1$  
- Choose a time $t\sim \mathrm{Uniform}[0,1]$ 
- Define where the balloon is *supposed to be* at time $t$:
  
$
x_t = (1-t)\,x_0 + t\,x_1
$

On this straight-line story, the correct wind is just the slope:

$
u_t = \frac{dx_t}{dt} = x_1 - x_0.
$

So the training signal is:

> “When you see a balloon at $x_t$ at time $t$, the wind should be $u_t$.”

The learning objective becomes supervised regression:

$
\mathcal{L}(\theta)=
\mathbb{E}\left[\left\|v_\theta(x_t,t)-u_t\right\|_2^2\right].
$

### A 1D example: from one Binomial to another (wind in action)

We want to move a 1D random variable from:

- Start distribution:
$
X_0 \sim \mathrm{Binom}(N,p_0)
$

- Target distribution:
$
X_1 \sim \mathrm{Binom}(N,p_1)
$

This is a **distribution over counts** $\{0,1,\dots,N\}$.  
To use an ODE (continuous movement), we treat counts as points on the number line and let the flow move them continuously (later you can round/clamp to $[0,N]$ if you want discrete counts).

#### Setup
Let:
$
N=20,\quad p_0=0.2,\quad p_1=0.8.
$

Then:
$
\mathbb{E}[X_0]=Np_0=4,\qquad \mathbb{E}[X_1]=Np_1=16.
$

So the target distribution is shifted to the right.

#### Wind story for one particle (pointwise velocity)

Take one sample from each distribution:

- Suppose we draw:
$$
x_0 = 3 \sim \mathrm{Binom}(20,0.2),
\qquad
x_1 = 17 \sim \mathrm{Binom}(20,0.8).
$$

Pick time:
$
t = 0.25.
$

#### Where should the particle be at time t?
$
x_t = (1-0.25)\cdot 3 + 0.25\cdot 17 = 0.75\cdot 3 + 4.25 = 2.25 + 4.25 = 6.5.
$

#### What is the “wind” (velocity target)?
For the straight-line story:
$
u_t = x_1 - x_0 = 17 - 3 = 14.
$

**Interpretation (wind):**

> At time $t=0.25$, if the balloon is around $x=6.5$, the wind should blow **to the right** with speed about **14**.

Different pairs $(x_0,x_1)$ produce different winds; the model learns the *average correct wind pattern* across many such stories.

## How Flow Matching Avoids Score Calculation

### The “score” bottleneck in diffusion-style methods

In many diffusion models, the key object is the **score function** of the noisy distribution:
$
\nabla_x \log p_t(x),
$
i.e., the gradient of the log-density at time $t$ which is related to the noise.

A common training objective (in one parameterization) learns a network that is equivalent to the score by predicting noise:
$
x_t = \alpha(t)x_0 + \sigma(t)\varepsilon,\qquad \varepsilon\sim \mathcal{N}(0,I),
$
and training
$
\varepsilon_\theta(x_t,t)\approx \varepsilon.
$

From this, one can recover a score estimate (up to schedule factors) of the form
$
\nabla_x \log p_t(x)\ \approx\ -\frac{1}{\sigma(t)}\,\varepsilon_\theta(x_t,t).
$

#### Why “score” is hard / fragile in practice
Even though diffusion training is stable, conceptually the method depends on estimating a derivative of a log-density:
- $\nabla_x \log p_t(x)$ can be **highly sensitive** where $p_t(x)$ is small.
- It is not directly observable; you learn it indirectly via denoising objectives.
- It is tightly coupled to a **stochastic process** (SDE) and its associated reverse-time dynamics.

### The “posterior / likelihood” bottleneck in VAEs

A VAE introduces latent variables $z$ and maximizes an ELBO:
$$
\log p_\theta(x)
\ge
\mathbb{E}_{q_\phi(z\mid x)}[\log p_\theta(x\mid z)]
-
\mathrm{KL}(q_\phi(z\mid x)\,\|\,p(z)).
$$

#### What makes this “hard”
- You must define and train an **encoder** $q_\phi(z\mid x)$ to approximate a posterior.
- You must choose a decoder likelihood model $p_\theta(x\mid z)$ (e.g., Gaussian, Bernoulli), which can be mismatched to real data.
- The objective involves a **KL term** and a **reconstruction likelihood**, both of which can create issues like posterior collapse and mismatch in perceptual quality.

### Flow matching: A shift in viewpoint

Flow matching does **not** try to estimate:
- a score $\nabla_x \log p_t(x)$, or
- a latent posterior $q_\phi(z\mid x)$.

Instead, flow matching learns a **velocity field**:
$$
v_\theta(x,t),
$$
which defines a deterministic transport ODE:
$$
\frac{dx}{dt} = v_\theta(x,t),\qquad t\in[0,1].
$$

If $x(0)\sim p_0$ (easy base, e.g., Gaussian) and we choose the right $v_\theta$, then the ODE flow transports the distribution to the data:
$
x(1)\sim p_1.
$

#### What flow matching *does* optimize: supervised vector-field regression

Flow matching constructs training pairs by sampling:
$
x_0 \sim p_0,\qquad x_1 \sim p_1,\qquad t \sim \mathrm{Uniform}[0,1],
$
then defining an intermediate point along a chosen bridge, e.g. linear interpolation:
$
x_t = (1-t)x_0 + t x_1.
$

The “ground-truth” velocity along this bridge is known analytically:
$
u_t = \frac{d x_t}{dt}.
$

For linear interpolation:
$
u_t = x_1 - x_0.
$

Then the learning problem is simple supervised regression:
$$
\mathcal{L}(\theta)
=
\mathbb{E}\left[\left\|v_\theta(x_t,t) - u_t\right\|_2^2\right].
$$

#### What this avoids
- No need to compute or approximate $\nabla_x \log p_t(x)$.
- No need to specify or approximate $q_\phi(z\mid x)$.
- No reverse-time SDE derivation is required to define the target (the target velocity is explicit from the bridge).

### Sampling: deterministic integration vs stochastic sampling

#### Flow matching sampling
1. Sample base noise:
$
x(0)\sim p_0.
$
2. Integrate ODE:
$
x(1) = x(0) + \int_0^1 v_\theta(x(t),t)\,dt.
$

#### Diffusion sampling
Often requires many stochastic (or deterministic probability-flow) steps, but still tied to the score-trained model.

Flow matching is frequently compatible with fewer solver steps because it is an ODE transport by design.

### Summary

#### Diffusion
- Learns: score-like object $$\nabla_x\log p_t(x)$$ (often via noise prediction)
- Sampling: reverse diffusion / SDE/ODE
- Key math object: log-density gradient

#### VAE
- Learns: encoder posterior $$q_\phi(z\mid x)$$ and decoder likelihood $$p_\theta(x\mid z)$$
- Sampling: sample $$z\sim p(z)$$, decode
- Key math object: KL between posteriors + likelihood model assumptions

#### Flow matching
- Learns: velocity field $$v_\theta(x,t)$$
- Sampling: integrate $$dx/dt=v_\theta(x,t)$$
- Key math object: supervised vector regression to bridge-defined velocity targets

Flow matching avoids the explicit or implicit need to estimate **scores** $\nabla_x\log p_t(x)$ (diffusion) and avoids **latent posterior approximation** $q_\phi(z\mid x)$ (VAE) by instead learning a **deterministic velocity field** $v_\theta(x,t)$ via direct supervision from a chosen transport bridge, then sampling by ODE integration.


# Flow Matching in 1D — Normal Shift and Bimodal Demo

This notebook demonstrates **Flow Matching** in **1D** with two targets:

1. **Normal shift**: $p_{\text{data}}(x)=\mathcal{N}(\mu,1)$ with $\mu=3$  
2. **Bimodal**: $p_{\text{data}}(x)=\tfrac12\mathcal{N}(-2,0.5^2)+\tfrac12\mathcal{N}(2,0.5^2)$

We start from a base distribution:

$$
x_0 \sim p_0 = \mathcal{N}(0,1).

$$

We define a simple **straight-line coupling** by sampling pairs $(x_0,x_1)$ and interpolating:

$$
x_t = (1-t)x_0 + t x_1.

$$

The corresponding **target velocity** is:

$$
\frac{d x_t}{dt} = x_1 - x_0.

$$

We train a neural net $v_\theta(x,t)$ by regression:

$$
\mathcal{L}(\theta)
=
\mathbb{E}_{x_0,x_1,t}\left[\left(v_\theta(x_t,t) - (x_1-x_0)\right)^2\right],
\quad t \sim \mathrm{Unif}(0,1).

$$

After training, we **generate** by integrating the ODE:

$$
\frac{dx}{dt} = v_\theta(x,t), \quad t\in[0,1].

$$

## What one should expect
- In the **Normal shift** case, the learned velocity should become **nearly constant** ($\approx \mu$).
- In the **Bimodal** case, the learned velocity becomes **nonlinear** in $x$ and evolves with $t$, pushing mass toward $-2$ and $+2$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Example endpoints
x0 = -0.7
x1 = 2.3

# t grid
t = np.linspace(0, 1, 200)
xt = (1 - t) * x0 + t * x1

# Mark a few discrete steps
t_marks = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
xt_marks = (1 - t_marks) * x0 + t_marks * x1

# Plot 1: trajectory in (t, x) plane
plt.figure(figsize=(7.5, 4.5))
plt.plot(t, xt, linewidth=2)
plt.scatter(t_marks, xt_marks, s=60)
for tm, xm in zip(t_marks, xt_marks):
    plt.text(tm, xm, f"  t={tm:.2f}, x={xm:.2f}", va="center")
plt.xlabel("t")
plt.ylabel("x_t")
plt.title("Geometric view: x_t traces a straight line in (t, x) space")
plt.grid(True, alpha=0.3)
plt.show()

# Plot 2: points on the 1D line (x-axis) for the same t values
plt.figure(figsize=(7.5, 2.2))
plt.scatter(xt_marks, np.zeros_like(xt_marks), s=80)
plt.yticks([])
plt.xlabel("x")
plt.title("Same x_t points on the 1D axis (t increases left → right along the path)")
plt.grid(True, axis="x", alpha=0.3)
for xm, tm in zip(xt_marks, t_marks):
    plt.text(xm, 0.0, f"\n t={tm:.2f}", ha="center", va="top")
plt.show()



## Example: Normal to another Normal Distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- "v_theta" for Normal->Normal shift ---
# Base: N(0,1), Target: N(mu,1)
mu = 3.0

def vtheta(x, t):
    # For this specific task (same variance), an ideal flow is constant drift by mu
    # so x(t) = x0 + mu*t and x(1) ~ N(mu,1).
    return mu * np.ones_like(x)

# --- Euler integration for many particles ---
N = 400  # number of particles to visualize
steps = 40
dt = 1.0 / steps
t_grid = np.linspace(0, 1, steps + 1)

# initial samples
x = np.random.randn(N)  # x(0) ~ N(0,1)

# store trajectory
traj = np.zeros((steps + 1, N))
traj[0] = x

for k in range(steps):
    t = t_grid[k]
    x = x + dt * vtheta(x, t)
    traj[k + 1] = x

# --- Plot 1: many particle trajectories in (t, x) space ---
plt.figure(figsize=(8.5, 4.8))
for i in range(N):
    plt.plot(t_grid, traj[:, i], alpha=0.08)
plt.xlabel("t")
plt.ylabel("x(t)")
plt.title(r"Many particles transported by $dx/dt = v_\theta(x,t)$ (Normal $\to$ shifted Normal)")
plt.grid(True, alpha=0.25)
plt.show()

# --- Plot 2: distribution snapshots over time ---
snap_idx = [0, int(0.25*steps), int(0.5*steps), int(0.75*steps), steps]
plt.figure(figsize=(8.5, 4.8))
for k in snap_idx:
    plt.hist(traj[k], bins=80, density=True, alpha=0.35, label=f"t={t_grid[k]:.2f}")
plt.xlabel("x")
plt.ylabel("density")
plt.title("Distribution evolution under the same learned wind field")
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

# --- Plot 3: final distribution vs true target ---
true_target = mu + np.random.randn(50000)
plt.figure(figsize=(8.5, 4.8))
plt.hist(traj[-1], bins=80, density=True, alpha=0.5, label="generated x(1)")
plt.hist(true_target, bins=80, density=True, alpha=0.5, label=r"true target $N(\mu,1)$")
plt.xlabel("x")
plt.ylabel("density")
plt.title("Final distribution comparison")
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()



## Example: Single to Trimodal Distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

# ----------------------------
# Setup
# ----------------------------
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Target: tri-modal mixture
modes = torch.tensor([-3.0, 0.0, 3.0], device=device)

def sample_base(n: int) -> torch.Tensor:
    # x0 ~ N(0,1)
    return torch.randn(n, 1, device=device)

def sample_trimodal(n: int, sigma: float = 0.6) -> torch.Tensor:
    # x1 ~ (1/3) N(-3,sigma^2) + (1/3) N(0,sigma^2) + (1/3) N(3,sigma^2)
    comp = torch.randint(0, 3, size=(n, 1), device=device)
    means = modes[comp]
    return means + sigma * torch.randn(n, 1, device=device)

def uniform_t(n: int) -> torch.Tensor:
    return torch.rand(n, 1, device=device)

def make_xt_and_u(x0: torch.Tensor, x1: torch.Tensor, t: torch.Tensor):
    # Straight line path and its true velocity
    xt = (1.0 - t) * x0 + t * x1
    u  = x1 - x0
    return xt, u

# ----------------------------
# v_theta model
# ----------------------------
class VelocityMLP(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))

def count_params(m):
    return sum(p.numel() for p in m.parameters())

# ----------------------------
# Helpers for plotting v_theta
# ----------------------------
@torch.no_grad()
def velocity_curves(model, x_grid_np, t_list):
    model.eval()
    x = torch.from_numpy(x_grid_np.astype(np.float32)).view(-1,1).to(device)
    curves = []
    for tval in t_list:
        t = torch.full_like(x, float(tval))
        v = model(x, t).detach().cpu().numpy().reshape(-1)
        curves.append(v)
    return curves

def plot_velocity_snapshot(x_grid_np, curves, t_list, title):
    plt.figure(figsize=(8.8,4.6))
    for v, tval in zip(curves, t_list):
        plt.plot(x_grid_np, v, label=f"t={tval:.2f}")
    plt.axhline(0.0, linewidth=1)
    plt.xlabel("x")
    plt.ylabel("v_theta(x,t)")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.show()

# ----------------------------
# Training (Flow Matching)
# ----------------------------
def train_flow_matching_trimodal(
    epochs=2000,
    batch_size=2048,
    lr=2e-3,
    sigma=0.6,
    snapshot_epochs=(0, 50, 200, 800, 1999)
):
    model = VelocityMLP(hidden=128).to(device)
    print("params:", count_params(model))

    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    losses = []
    snaps = {}

    # for velocity snapshots
    x_grid_np = np.linspace(-6, 6, 500).astype(np.float32)
    t_list = [0.0, 0.25, 0.5, 0.75, 1.0]

    for ep in range(epochs):
        model.train()

        x0 = sample_base(batch_size)
        x1 = sample_trimodal(batch_size, sigma=sigma)
        t  = uniform_t(batch_size)

        xt, u = make_xt_and_u(x0, x1, t)

        pred = model(xt, t)
        loss = loss_fn(pred, u)

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(float(loss.item()))

        if ep in snapshot_epochs:
            curves = velocity_curves(model, x_grid_np, t_list)
            snaps[int(ep)] = (x_grid_np.copy(), curves, list(t_list))

        if (ep + 1) % 250 == 0:
            print(f"epoch {ep+1}/{epochs}  loss={loss.item():.5f}")

    return model, np.array(losses), snaps

# ----------------------------
# ODE sampling with Euler
# ----------------------------
@torch.no_grad()
def euler_sample_trajectory(model, n_samples=20000, n_steps=80):
    model.eval()
    x = sample_base(n_samples)  # x(0)
    dt = 1.0 / n_steps

    record_steps = [0, int(0.25*n_steps), int(0.5*n_steps), int(0.75*n_steps), n_steps]
    traj = []
    times = []

    for k in range(n_steps + 1):
        if k in record_steps:
            traj.append(x.detach().cpu().numpy().reshape(-1))
            times.append(k * dt)

        if k == n_steps:
            break

        t = torch.full_like(x, float(k * dt))
        v = model(x, t)
        x = x + dt * v

    return np.array(times), traj

def plot_distribution_evolution(times, traj, title, bins=120, xlim=(-6,6)):
    plt.figure(figsize=(8.8,4.6))
    for t, xs in zip(times, traj):
        plt.hist(xs, bins=bins, density=True, alpha=0.35, range=xlim, label=f"t={t:.2f}")
    plt.xlim(*xlim)
    plt.xlabel("x")
    plt.ylabel("density")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.show()

# ----------------------------
# Particle trajectory plot (many lines)
# ----------------------------
@torch.no_grad()
def plot_many_particle_paths(model, n_paths=400, n_steps=80, title=""):
    model.eval()
    dt = 1.0 / n_steps
    t_grid = np.linspace(0, 1, n_steps+1)

    x = sample_base(n_paths)  # (n_paths,1)
    traj = torch.zeros(n_steps+1, n_paths, device=device)
    traj[0] = x[:,0]

    for k in range(n_steps):
        t = torch.full_like(x, float(k*dt))
        v = model(x,t)
        x = x + dt*v
        traj[k+1] = x[:,0]

    traj_np = traj.detach().cpu().numpy()

    plt.figure(figsize=(8.8,4.6))
    for i in range(n_paths):
        plt.plot(t_grid, traj_np[:,i], alpha=0.08)
    for m in [-3, 0, 3]:
        plt.axhline(m, linestyle="--", linewidth=1)
    plt.xlabel("t")
    plt.ylabel("x(t)")
    plt.title(title)
    plt.grid(True, alpha=0.2)
    plt.show()

# ----------------------------
# RUN: train + plot everything
# ----------------------------
sigma = 0.6
model, losses, snaps = train_flow_matching_trimodal(
    epochs=2000,
    batch_size=2048,
    lr=2e-3,
    sigma=sigma,
    snapshot_epochs=(0, 50, 200, 800, 1999)
)

# 1) Loss curve
plt.figure(figsize=(8.8,4.2))
plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.title("Training loss — Flow Matching (tri-modal target)")
plt.grid(True, alpha=0.2)
plt.show()

# 2) Velocity field snapshots across training
for ep in sorted(snaps.keys()):
    xg, curves, t_list = snaps[ep]
    plot_velocity_snapshot(xg, curves, t_list, title=f"v_theta(x,t) snapshots — epoch {ep}")

# 3) Many particle trajectories in (t,x)
plot_many_particle_paths(
    model,
    n_paths=400,
    n_steps=80,
    title=r"Particle transport under learned $dx/dt=v_\theta(x,t)$ (Normal $\to$ tri-modal)"
)

# 4) Distribution evolution snapshots
times, traj = euler_sample_trajectory(model, n_samples=25000, n_steps=80)
plot_distribution_evolution(
    times, traj,
    title="Distribution evolution under learned transport (snapshots)",
    xlim=(-6,6),
    bins=140
)

# 5) Final distribution vs true tri-modal target
true_target = sample_trimodal(60000, sigma=sigma).detach().cpu().numpy().reshape(-1)

plt.figure(figsize=(8.8,4.6))
plt.hist(traj[-1], bins=140, density=True, alpha=0.5, range=(-6,6), label="generated x(1)")
plt.hist(true_target, bins=140, density=True, alpha=0.5, range=(-6,6), label="true target tri-modal mixture")
for m in [-3, 0, 3]:
    plt.axvline(m, linestyle="--", linewidth=1)
plt.xlabel("x")
plt.ylabel("density")
plt.title("Final distribution comparison (generated vs target)")
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

# Flow Matching for MNIST: Design Rationale

## Goal
Train a model to generate MNIST digits by learning a time-dependent velocity field (a “wind field”):

$
\frac{dx}{dt} = v_\theta(x,t), \qquad t\in[0,1].
$

If we start from noise $x(0)\sim \mathcal{N}(0,I)$ and integrate the ODE to $t=1$, then $x(1)$ should look like an MNIST digit.

### Architecture choice: small U-Net
MNIST has spatial structure (28×28), so convolutional networks work much better than MLPs.
A small U-Net is ideal because:
- it’s the standard backbone for diffusion/flow image models,
- skip connections help preserve local structure,
- it’s small enough to run quickly.

### Time conditioning
We need $v_\theta(x,t)$ to depend on time. We use a diffusion-style sinusoidal embedding of $t$, then inject it as a bias into feature maps.

### Sampling = numerical integration
At generation time, we integrate the ODE from $t=0$ to $t=1$.
Euler’s method is the simplest solver:

$
x_{k+1} = x_k + \Delta t\, v_\theta(x_k,t_k), \qquad \Delta t=\frac{1}{N}.
$

More steps (or better solvers like Heun/RK2) typically improve quality.

### Data scaling
We scale MNIST images to $[-1,1]$ (instead of $[0,1]$) to make them more symmetric and closer in scale to standard Gaussian noise.

In [ ]:
!pip install matplotlib

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Notebook-safe transform: [0,1] -> [-1,1]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # (x-0.5)/0.5
])

# Download MNIST
train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Create loaders (num_workers=0 avoids PicklingError in Jupyter on macOS/Windows)
batch_size = 256

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False
)

# Sanity check
x, y = next(iter(train_loader))
print("x:", x.shape, "y:", y.shape, "range:", (x.min().item(), x.max().item()))

In [ ]:
# Flow Matching on MNIST

import math
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


# -------------------------
# Setup
# -------------------------
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# -------------------------
# Data: MNIST in [-1,1]
# -------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # [0,1] -> [-1,1]
])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(
    train_ds,
    batch_size=256,
    shuffle=True,
    num_workers=0,  # IMPORTANT for Jupyter on macOS/Windows
    pin_memory=torch.cuda.is_available(),
    drop_last=True
)

# -------------------------
# Visualization helpers
# -------------------------
@torch.no_grad()
def show_grid(imgs, title="", nrow=8):
    """
    imgs: [N,1,28,28] in [-1,1]
    """
    imgs = imgs.detach().cpu()
    imgs = (imgs + 1) / 2  # [-1,1] -> [0,1]
    N = imgs.shape[0]
    ncol = nrow
    nrow_actual = int(math.ceil(N / ncol))
    plt.figure(figsize=(ncol, nrow_actual))
    for i in range(N):
        plt.subplot(nrow_actual, ncol, i + 1)
        plt.imshow(imgs[i, 0], cmap="gray")
        plt.axis("off")
    plt.suptitle(title)
    plt.show()

@torch.no_grad()
def trajectory_snapshots(model, n=8, steps=30, times=(0.0, 0.25, 0.5, 0.75, 1.0)):
    """
    Show intermediate x(t) snapshots during ODE integration for a few samples.
    """
    model.eval()
    x = torch.randn(n, 1, 28, 28, device=device)
    dt = 1.0 / steps
    snaps = {}

    for k in range(steps + 1):
        tval = k * dt
        for st in times:
            if (st not in snaps) and abs(tval - st) < 0.5 * dt:
                snaps[st] = x.detach().clone()

        if k == steps:
            break
        t = torch.full((n, 1), tval, device=device)
        v = model(x, t)
        x = x + dt * v

    rows = len(times)
    cols = n
    plt.figure(figsize=(cols * 1.0, rows * 1.0))
    for r, st in enumerate(times):
        imgs = snaps.get(st, x)
        imgs = (imgs + 1) / 2
        for c in range(cols):
            plt.subplot(rows, cols, r * cols + c + 1)
            plt.imshow(imgs[c, 0].detach().cpu(), cmap="gray")
            plt.axis("off")
            if c == 0:
                plt.ylabel(f"t={st:.2f}", rotation=0, labelpad=30, va="center")
    plt.suptitle("Trajectory snapshots: noise → digit")
    plt.show()

# -------------------------
# Time embedding
# -------------------------
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        t: [B,1] in [0,1]
        returns: [B, dim]
        """
        if t.ndim == 2:
            t = t[:, 0]
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(0, half, device=t.device).float() / (half - 1)
        )
        args = t[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
        if self.dim % 2 == 1:
            emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=1)
        return emb


# -------------------------
# Velocity U-Net v_theta(x,t)
# -------------------------
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)

        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)

        self.t_proj = nn.Sequential(nn.SiLU(), nn.Linear(t_dim, out_ch))
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv2d(in_ch, out_ch, 1)

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.t_proj(t_emb)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class Down(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.Conv2d(ch, ch, 4, stride=2, padding=1)  # 28->14->7

    def forward(self, x):
        return self.op(x)


class Up(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.ConvTranspose2d(ch, ch, 4, stride=2, padding=1)

    def forward(self, x):
        return self.op(x)


class VelocityUNet(nn.Module):
    def __init__(self, base_ch=64, t_dim=128):
        super().__init__()
        self.time_emb = nn.Sequential(
            SinusoidalTimeEmbedding(t_dim),
            nn.Linear(t_dim, t_dim),
            nn.SiLU(),
            nn.Linear(t_dim, t_dim),
        )

        self.in_conv = nn.Conv2d(1, base_ch, 3, padding=1)

        # Down
        self.rb1 = ResBlock(base_ch, base_ch, t_dim)
        self.down1 = Down(base_ch)
        self.rb2 = ResBlock(base_ch, base_ch * 2, t_dim)
        self.down2 = Down(base_ch * 2)

        # Mid
        self.mid1 = ResBlock(base_ch * 2, base_ch * 2, t_dim)
        self.mid2 = ResBlock(base_ch * 2, base_ch * 2, t_dim)

        # Up
        self.up2 = Up(base_ch * 2)
        self.rb_up2 = ResBlock(base_ch * 2 + base_ch * 2, base_ch * 2, t_dim)
        self.up1 = Up(base_ch * 2)
        self.rb_up1 = ResBlock(base_ch * 2 + base_ch, base_ch, t_dim)

        self.out_norm = nn.GroupNorm(8, base_ch)
        self.out_conv = nn.Conv2d(base_ch, 1, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_emb(t)

        x0 = self.in_conv(x)
        h1 = self.rb1(x0, t_emb)
        d1 = self.down1(h1)

        h2 = self.rb2(d1, t_emb)
        d2 = self.down2(h2)

        m = self.mid1(d2, t_emb)
        m = self.mid2(m, t_emb)

        u2 = self.up2(m)
        u2 = torch.cat([u2, h2], dim=1)
        u2 = self.rb_up2(u2, t_emb)

        u1 = self.up1(u2)
        u1 = torch.cat([u1, h1], dim=1)
        u1 = self.rb_up1(u1, t_emb)

        out = self.out_conv(F.silu(self.out_norm(u1)))
        return out


model = VelocityUNet(base_ch=64, t_dim=128).to(device)
print("params:", sum(p.numel() for p in model.parameters()))

# -------------------------
# Flow matching batch
# -------------------------
def flow_matching_batch(x1):
    """
    x1: [B,1,28,28] data in [-1,1]
    Create:
      x0 ~ N(0,I)
      t ~ Uniform(0,1)
      xt = (1-t)x0 + t x1
      u  = x1 - x0
    """
    B = x1.shape[0]
    x0 = torch.randn_like(x1)
    t = torch.rand(B, 1, device=x1.device)
    t_img = t[:, :, None, None]
    xt = (1 - t_img) * x0 + t_img * x1
    u = x1 - x0
    return xt, u, t


@torch.no_grad()
def sample_euler(model, n=64, steps=50):
    """
    Integrate dx/dt = v_theta(x,t) from t=0->1 using Euler.
    """
    model.eval()
    x = torch.randn(n, 1, 28, 28, device=device)
    dt = 1.0 / steps
    for k in range(steps):
        t = torch.full((n, 1), k * dt, device=device)
        v = model(x, t)
        x = x + dt * v
    return x


# -------------------------
# Train loop (run this cell to train)
# -------------------------
epochs = 50          # increase to 20–50 on GPU
lr = 2e-4
sample_steps = 50

opt = optim.AdamW(model.parameters(), lr=lr)

use_amp = (device.type == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

loss_history = []
start = time.time()

for epoch in range(1, epochs + 1):
    model.train()
    epoch_losses = []

    for x1, _ in train_loader:
        x1 = x1.to(device)
        xt, u, t = flow_matching_batch(x1)

        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            v = model(xt, t)
            loss = F.mse_loss(v, u)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        epoch_losses.append(loss.item())

    mean_loss = float(np.mean(epoch_losses))
    loss_history.append(mean_loss)
    elapsed = (time.time() - start) / 60.0
    print(f"epoch {epoch:02d}/{epochs}  loss={mean_loss:.5f}  elapsed={elapsed:.1f} min")

    # Visualizations every 10 epochs
    if epoch % 10 == 0 or epoch == epochs:
        xgen = sample_euler(model, n=64, steps=sample_steps)
        show_grid(xgen, title=f"Generated samples (epoch {epoch})", nrow=8)
        trajectory_snapshots(model, n=8, steps=30)

# Loss curve
plt.figure(figsize=(7.5, 4))
plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.title("Flow Matching training loss (MNIST)")
plt.grid(True, alpha=0.25)
plt.show()

# Save (optional)
torch.save({"model": model.state_dict()}, "mnist_flow_matching_vtheta.pt")
print("Saved: mnist_flow_matching_vtheta.pt")

## Visualizing Velocity

### What does each pixel in the velocity map represent in a 2D example?

Each pixel in the velocity map is the **instantaneous rate of change** of that image pixel at that moment.

Formally, your state is the whole image:
$$
x(t)\in\mathbb{R}^{H\times W}.
$$

The flow-matching sampling ODE is:
$$
\frac{dx}{dt}=v_\theta(x,t).
$$

This equation is **elementwise**. For a pixel $(i,j)$:
$$
\frac{d}{dt}x_{ij}(t)=\big[v_\theta(x(t),t)\big]_{ij}.
$$

So:

- $\big[v_\theta\big]_{ij} > 0$: that pixel should **increase** (get brighter / move toward $+1$ if you use $[-1,1]$ normalization).
- $\big[v_\theta\big]_{ij} < 0$: that pixel should **decrease** (get darker / move toward $-1$).
- $\left|\big[v_\theta\big]_{ij}\right|$: **how fast** it should change.

With **Euler integration**, the discrete update at pixel $(i,j)$ is:
$$
x_{ij}^{(k+1)} \approx x_{ij}^{(k)} + \Delta t\;\big[v_\theta(x^{(k)},t_k)\big]_{ij}.
$$

### Important Constraint

Even though it’s “per pixel,” it is **not independent per pixel**. The network computes $v_\theta(x,t)$ using the **whole image context**, so the velocity at one pixel depends on nearby strokes and global structure.

Intuitively, it is asking:

> “At this stage $t$, given what the whole image currently looks like, should this pixel get a bit brighter or darker, and by how much, to move toward a realistic digit?”

### If we had multiple channels (RGB)

For RGB images, the velocity has shape:
$$
v_\theta(x,t)\in\mathbb{R}^{3\times H\times W}.
$$

Then each pixel has a **3D velocity vector**:
$$
\big[v_\theta(x,t)\big]_{ij}\in\mathbb{R}^3,
$$

one component per color channel (how R, G, and B should change per unit time).

In [ ]:
!pip install matplotlib

In [ ]:
# ---  MNIST conditional Flow Matching for visualizing velocity ---
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# -----------------------
# 0) Device + seeds
# -----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0); np.random.seed(0)

# -----------------------
# Data: MNIST in [-1,1]
# -----------------------
tfm = transforms.Compose([
    transforms.ToTensor(),                       # [0,1]
    transforms.Normalize((0.5,), (0.5,)),        # [-1,1]
])

train_ds = datasets.MNIST("./data_mnist", train=True, download=True, transform=tfm)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2, pin_memory=(device.type=="cuda"))

# -----------------------
# 2) Time embedding + conditional velocity model
# -----------------------
def sinusoidal_time_embedding(t, dim=64):
    # t: (B,1) in [0,1]
    if t.ndim == 2:
        t = t[:,0]
    half = dim // 2
    freqs = torch.exp(-math.log(10000.0) * torch.arange(0, half, device=t.device).float() / max(half-1, 1))
    args = t[:, None] * freqs[None, :]
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
    if dim % 2 == 1:
        emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=1)
    return emb  # (B,dim)

class CondFlowCNN(nn.Module):
    """
    v_theta(x,t,y): predicts velocity field same shape as x (B,1,28,28)
    Conditioning:
      - time embedding
      - label embedding
    Both are injected as per-pixel feature maps and concatenated with x.
    """
    def __init__(self, tdim=64, ydim=32, base=64, num_classes=10):
        super().__init__()
        self.tdim = tdim
        self.ydim = ydim
        self.yemb = nn.Embedding(num_classes, ydim)
        self.tproj = nn.Sequential(nn.Linear(tdim, base), nn.SiLU(), nn.Linear(base, base))
        self.yproj = nn.Sequential(nn.Linear(ydim, base), nn.SiLU(), nn.Linear(base, base))

        # Input channels: x(1) + tmap(base) + ymap(base) => 1 + base + base
        in_ch = 1 + base + base
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, base, 3, padding=1), nn.SiLU(),
            nn.Conv2d(base, base, 3, padding=1), nn.SiLU(),
            nn.Conv2d(base, base, 3, padding=1), nn.SiLU(),
            nn.Conv2d(base, 1, 3, padding=1)  # output velocity (1 channel)
        )

    def forward(self, x, t, y):
        # x: (B,1,28,28), t: (B,1), y: (B,)
        B, _, H, W = x.shape
        te = sinusoidal_time_embedding(t, self.tdim)          # (B,tdim)
        ye = self.yemb(y)                                     # (B,ydim)
        tfeat = self.tproj(te).view(B, -1, 1, 1).expand(B, -1, H, W)
        yfeat = self.yproj(ye).view(B, -1, 1, 1).expand(B, -1, H, W)
        inp = torch.cat([x, tfeat, yfeat], dim=1)
        return self.net(inp)

model = CondFlowCNN(tdim=64, ydim=32, base=64, num_classes=10).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-4)

print("Params:", sum(p.numel() for p in model.parameters()))

# -----------------------
# Training: Flow matching
#    x0 ~ N(0,1), x1 ~ data, x_t = (1-t)x0 + t x1, target u = x1 - x0
# -----------------------
def train_flow_matching(model, loader, epochs=5):
    model.train()
    hist = []
    t0 = time.time()
    for ep in range(1, epochs+1):
        losses = []
        for x1, y in loader:
            x1 = x1.to(device)          # (B,1,28,28) in [-1,1]
            y  = y.to(device)           # (B,)
            B = x1.shape[0]

            x0 = torch.randn_like(x1)   # base noise
            t  = torch.rand(B, 1, device=device)

            xt = (1.0 - t.view(B,1,1,1)) * x0 + t.view(B,1,1,1) * x1
            u  = x1 - x0

            opt.zero_grad(set_to_none=True)
            pred = model(xt, t, y)
            loss = F.mse_loss(pred, u)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            losses.append(loss.item())

        m = float(np.mean(losses))
        hist.append(m)
        print(f"[FM] epoch {ep}/{epochs}  loss={m:.5f}  min={(time.time()-t0)/60:.1f}")
    return hist

hist = train_flow_matching(model, train_loader, epochs=5)

plt.figure(figsize=(6,3))
plt.plot(hist)
plt.title("Flow matching training loss")
plt.xlabel("epoch")
plt.grid(True, alpha=0.2)
plt.show()

# -----------------------
# Sampling: Euler integrate dx/dt = v_theta(x,t,y)
#    Also record snapshots of x and velocity magnitude
# -----------------------
@torch.no_grad()
def sample_trajectory(model, y_digit=3, steps=60, snaps=8):
    model.eval()
    y = torch.tensor([y_digit], device=device).long()  # condition on one digit
    x = torch.randn(1, 1, 28, 28, device=device)       # start from noise
    dt = 1.0 / steps

    snap_idx = np.linspace(0, steps, snaps, dtype=int)  # include final
    xs, vms, ts = [], [], []

    for k in range(steps+1):
        t = torch.full((1,1), k*dt if k < steps else 1.0, device=device)

        if k in snap_idx:
            # velocity magnitude at this state/time
            v = model(x, t, y)
            vm = torch.abs(v)  # (1,1,28,28) magnitude in 1D-per-pixel sense
            xs.append(x.detach().cpu())
            vms.append(vm.detach().cpu())
            ts.append(float(t.item()))

        if k == steps:
            break

        v = model(x, t, y)
        x = x + dt * v
        x = torch.clamp(x, -1.0, 1.0)  # keep in range for display (optional)

    return xs, vms, ts

def show_row(images, title, cmap="gray", vmin=-1, vmax=1):
    # images: list of (1,1,28,28) CPU tensors
    n = len(images)
    plt.figure(figsize=(1.6*n, 2.2))
    for i, im in enumerate(images):
        plt.subplot(1, n, i+1)
        plt.imshow(im[0,0].numpy(), cmap=cmap, vmin=vmin, vmax=vmax)
        plt.axis("off")
    plt.suptitle(title, y=1.05)
    plt.show()

# Choose a target digit label
target_digit = 3

xs, vms, ts = sample_trajectory(model, y_digit=target_digit, steps=80, snaps=9)

# Show noise→path→final
show_row(xs, title=f"State x(t): noise → digit {target_digit} (times: {['%.2f'%t for t in ts]})", cmap="gray", vmin=-1, vmax=1)

# Show velocity magnitude along the same path (what the wind is doing)
# Velocity can be large early; we autoscale for visibility
show_row(vms, title=f"Velocity magnitude |v_theta(x(t),t,y)| along the path", cmap="magma", vmin=None, vmax=None)

# Final image alone
plt.figure(figsize=(2,2))
plt.imshow(xs[-1][0,0].numpy(), cmap="gray", vmin=-1, vmax=1)
plt.title(f"Final sample (digit {target_digit}) at t=1")
plt.axis("off")
plt.show()

In [ ]:
# Two-row trajectory visualization: row 1 = x(t), row 2 = signed v_theta(x(t),t,y)
# Drop-in replacement for the plotting part after you compute xs, vms/velocities.

import numpy as np
import matplotlib.pyplot as plt
import torch

@torch.no_grad()
def sample_trajectory_with_velocity(model, y_digit=3, steps=80, snaps=9):
    model.eval()
    y = torch.tensor([y_digit], device=device).long()
    x = torch.randn(1, 1, 28, 28, device=device)
    dt = 1.0 / steps

    snap_idx = np.linspace(0, steps, snaps, dtype=int)
    xs, vs, ts = [], [], []

    for k in range(steps + 1):
        t = torch.full((1,1), (k * dt) if k < steps else 1.0, device=device)

        v = model(x, t, y)

        if k in snap_idx:
            xs.append(x.detach().cpu())
            vs.append(v.detach().cpu())
            ts.append(float(t.item()))

        if k == steps:
            break

        x = x + dt * v
        x = torch.clamp(x, -1.0, 1.0)

    return xs, vs, ts

def show_trajectory_two_rows(xs, vs, ts, title_prefix=""):
    """
    xs: list of (1,1,28,28) CPU tensors
    vs: list of (1,1,28,28) CPU tensors (SIGNED velocities)
    ts: list of floats
    """
    n = len(xs)

    # Choose a symmetric color range for velocity based on robust percentile across all snapshots
    all_v = torch.cat([v.reshape(-1) for v in vs], dim=0).numpy()
    # We do this because a few extreme pixels can have very large velocities that would make the colormap 
    # less informative for the majority of pixels. So we are just clip
    vmax = np.percentile(np.abs(all_v), 99)  # robust; avoids a few extreme pixels dominating
    if vmax < 1e-6:
        vmax = 1e-6

    fig, axes = plt.subplots(2, n, figsize=(1.6*n, 3.6))

    for i in range(n):
        # Row 1: state x(t)
        ax = axes[0, i]
        ax.imshow(xs[i][0,0].numpy(), cmap="gray", vmin=-1, vmax=1)
        ax.set_title(f"t={ts[i]:.2f}", fontsize=9)
        ax.axis("off")

        # Row 2: signed velocity v(x(t),t)
        ax = axes[1, i]
        ax.imshow(vs[i][0,0].numpy(), cmap="seismic", vmin=-vmax, vmax=vmax)
        ax.axis("off")

    axes[0, 0].set_ylabel("x(t)", rotation=0, labelpad=20, fontsize=11, va="center")
    axes[1, 0].set_ylabel("v(t) (signed)", rotation=0, labelpad=20, fontsize=11, va="center")

    plt.suptitle(f"{title_prefix}Two-row snapshots: state and signed velocity", y=1.02)
    plt.tight_layout()
    plt.show()

# Example usage
target_digit = 3
xs, vs, ts = sample_trajectory_with_velocity(model, y_digit=target_digit, steps=80, snaps=9)
show_trajectory_two_rows(xs, vs, ts, title_prefix=f"MNIST y={target_digit} — ")

## Oxford Flowers II Example

In [ ]:
!pip install matplotlib

In [ ]:
import math
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

from torchvision import transforms
from torchvision.datasets import INaturalist

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
!pip install scipy
import scipy
from scipy import integrate, stats

In [ ]:

import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import Flowers102

# ----------------------------
# Oxford Flowers 102: download + labeled DataLoaders (notebook-safe)
# ----------------------------
root = "./data_flowers102"
img_size = 64        # you can increase to 96/128 if you have GPU
batch_size = 64
num_workers = 0      # notebook-safe

transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),                                # [0,1]
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # -> [-1,1]
])

train_ds = Flowers102(
    root=root,
    split="train",
    download=True,
    transform=transform
)

val_ds = Flowers102(
    root=root,
    split="val",
    download=True,
    transform=transform
)

test_ds = Flowers102(
    root=root,
    split="test",
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=False
)

test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=False
)

# Sanity check
x, y = next(iter(train_loader))
print("x:", x.shape, "y:", y.shape, "range:", (x.min().item(), x.max().item()))
print("num classes:", 102, "labels min/max:", int(y.min()), int(y.max()))

In [ ]:
@torch.no_grad()
def show_grid_any(imgs, title="", nrow=8):
    imgs = imgs.detach().cpu()
    if imgs.min().item() < -0.1:
        imgs = (imgs + 1) / 2
    imgs = imgs.clamp(0, 1)

    N, C, H, W = imgs.shape
    ncol = nrow
    nrow_actual = int(math.ceil(N / ncol))

    plt.figure(figsize=(ncol, nrow_actual))
    for i in range(N):
        plt.subplot(nrow_actual, ncol, i + 1)
        if C == 1:
            plt.imshow(imgs[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
        else:
            plt.imshow(imgs[i].permute(1, 2, 0).numpy())
        plt.axis("off")
    plt.suptitle(title)
    plt.show()

# Show some samples
x, y = next(iter(train_loader))
print("Batch shape:", x.shape)
show_grid_any(x[:64], title="Oxford Flowers 102 samples", nrow=8)

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        # t: [B,1] in [0,1]  ->  [B,dim]
        if t.ndim == 2:
            t = t[:, 0]
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(0, half, device=t.device).float() / (half - 1)
        )
        args = t[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
        if self.dim % 2 == 1:
            emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=1)
        return emb


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)

        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)

        self.t_proj = nn.Sequential(nn.SiLU(), nn.Linear(t_dim, out_ch))
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv2d(in_ch, out_ch, 1)

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.t_proj(t_emb)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class Down(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.Conv2d(ch, ch, 4, stride=2, padding=1)

    def forward(self, x):
        return self.op(x)


class Up(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.ConvTranspose2d(ch, ch, 4, stride=2, padding=1)

    def forward(self, x):
        return self.op(x)


class VelocityUNet(nn.Module):
    def __init__(self, base_ch=64, t_dim=256, in_ch=3, out_ch=3):
        super().__init__()
        self.time_emb = nn.Sequential(
            SinusoidalTimeEmbedding(t_dim),
            nn.Linear(t_dim, t_dim),
            nn.SiLU(),
            nn.Linear(t_dim, t_dim),
        )

        self.in_conv = nn.Conv2d(in_ch, base_ch, 3, padding=1)

        self.rb1 = ResBlock(base_ch, base_ch, t_dim)
        self.down1 = Down(base_ch)

        self.rb2 = ResBlock(base_ch, base_ch * 2, t_dim)
        self.down2 = Down(base_ch * 2)

        self.mid1 = ResBlock(base_ch * 2, base_ch * 2, t_dim)
        self.mid2 = ResBlock(base_ch * 2, base_ch * 2, t_dim)

        self.up2 = Up(base_ch * 2)
        self.rb_up2 = ResBlock(base_ch * 2 + base_ch * 2, base_ch * 2, t_dim)

        self.up1 = Up(base_ch * 2)
        self.rb_up1 = ResBlock(base_ch * 2 + base_ch, base_ch, t_dim)

        self.out_norm = nn.GroupNorm(8, base_ch)
        self.out_conv = nn.Conv2d(base_ch, out_ch, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_emb(t)

        x0 = self.in_conv(x)
        h1 = self.rb1(x0, t_emb)
        d1 = self.down1(h1)

        h2 = self.rb2(d1, t_emb)
        d2 = self.down2(h2)

        m = self.mid1(d2, t_emb)
        m = self.mid2(m, t_emb)

        u2 = self.up2(m)
        u2 = torch.cat([u2, h2], dim=1)
        u2 = self.rb_up2(u2, t_emb)

        u1 = self.up1(u2)
        u1 = torch.cat([u1, h1], dim=1)
        u1 = self.rb_up1(u1, t_emb)

        out = self.out_conv(F.silu(self.out_norm(u1)))
        return out


model = VelocityUNet(base_ch=64, t_dim=256, in_ch=3, out_ch=3).to(device)
print("params:", sum(p.numel() for p in model.parameters()))

In [ ]:
def flow_matching_batch(x1):
    """
    x1: [B,3,H,W] in [-1,1]
    """
    B = x1.shape[0]
    x0 = torch.randn_like(x1)                 # base noise
    t = torch.rand(B, 1, device=x1.device)    # [B,1]
    t_img = t[:, :, None, None]
    xt = (1 - t_img) * x0 + t_img * x1
    u = x1 - x0
    return xt, u, t


@torch.no_grad()
def sample_euler(model, n=64, steps=50, img_size=64):
    model.eval()
    x = torch.randn(n, 3, img_size, img_size, device=device)
    dt = 1.0 / steps
    for k in range(steps):
        t = torch.full((n, 1), k * dt, device=device)
        v = model(x, t)
        x = x + dt * v
    return x


@torch.no_grad()
def trajectory_snapshots(model, n=8, steps=30, times=(0.0, 0.25, 0.5, 0.75, 1.0), img_size=64):
    model.eval()
    x = torch.randn(n, 3, img_size, img_size, device=device)
    dt = 1.0 / steps
    snaps = {}

    for k in range(steps + 1):
        tval = k * dt
        for st in times:
            if (st not in snaps) and abs(tval - st) < 0.5 * dt:
                snaps[st] = x.detach().clone()

        if k == steps:
            break

        t = torch.full((n, 1), tval, device=device)
        v = model(x, t)
        x = x + dt * v

    # show each time as a separate grid
    for st in times:
        show_grid_any(snaps[st], title=f"Trajectory snapshots at t={st:.2f}", nrow=n)

In [ ]:
epochs = 100         # increase (10–50) for better quality if you have a GPU
lr = 2e-4
sample_steps = 50

opt = optim.AdamW(model.parameters(), lr=lr)
use_amp = (device.type == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

loss_hist = []
start = time.time()

for epoch in range(1, epochs + 1):
    model.train()
    batch_losses = []

    for x1, _ in train_loader:
        x1 = x1.to(device)

        xt, u, t = flow_matching_batch(x1)

        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            v = model(xt, t)
            loss = F.mse_loss(v, u)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        batch_losses.append(loss.item())

    mean_loss = float(np.mean(batch_losses))
    loss_hist.append(mean_loss)
    

    # show every 10 epochs (and at the end)
    if (epoch + 1) % 10 == 0:
        print(f"epoch {epoch:02d}/{epochs}  loss={mean_loss:.5f}  elapsed={(time.time()-start)/60:.1f} min")
        xgen = sample_euler(model, n=64, steps=sample_steps)
        show_grid_any(xgen, title=f"Generated samples (epoch {epoch+1})", nrow=8)
        trajectory_snapshots(model, n=8, steps=30)
    
# Loss curve
plt.figure(figsize=(7.5,4))
plt.plot(loss_hist)
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.title("Flow Matching training loss Office Flowers 102")
plt.grid(True, alpha=0.25)
plt.show()

# Save
torch.save({"model": model.state_dict(), "img_size": img_size}, "flower_flow_matching_vtheta.pt")
print("Saved: flower_flow_matching_vtheta.pt") 

## Printing the various parameters of the trained model

In [ ]:
import torch

# 1) Print a quick summary of common hyperparameter-like attributes (if they exist)
def print_model_attrs(model):
    keys = ["in_ch", "out_ch", "base_ch", "t_dim", "img_size", "n_classes", "null_id"]
    print("=== Model attributes (if present) ===")
    for k in keys:
        if hasattr(model, k):
            print(f"{k}: {getattr(model, k)}")
    print()

# 2) Print all parameter names + shapes (and total count)
def print_param_shapes(model, max_rows=None):
    print("=== Parameter shapes ===")
    total = 0
    rows = 0
    for name, p in model.named_parameters():
        print(f"{name:50s} {tuple(p.shape)}  requires_grad={p.requires_grad}")
        total += p.numel()
        rows += 1
        if max_rows is not None and rows >= max_rows:
            print(f"... (stopped after {max_rows} rows)")
            break
    print("\nTotal parameters:", total)

# 3) Print just the layers that reveal key dims (Conv in/out and Linear in/out)
def print_key_layers(model):
    print("=== Key layers (Conv2d / Linear) ===")
    for name, m in model.named_modules():
        if isinstance(m, torch.nn.Conv2d):
            print(f"{name:50s} Conv2d(in={m.in_channels}, out={m.out_channels}, k={m.kernel_size}, s={m.stride})")
        elif isinstance(m, torch.nn.ConvTranspose2d):
            print(f"{name:50s} ConvT(in={m.in_channels}, out={m.out_channels}, k={m.kernel_size}, s={m.stride})")
        elif isinstance(m, torch.nn.Linear):
            print(f"{name:50s} Linear(in={m.in_features}, out={m.out_features})")
    print()

# Example usage:
print_model_attrs(model)
print_key_layers(model)
print_param_shapes(model)

## Latent Flow Matching (LFM)

Latent Flow Matching combines two ideas:

1. **Learn a compact latent space** where the data becomes easier to model (via an autoencoder / VAE).
2. **Learn a flow-matching velocity field in that latent space** to transport noise into latents that decode into realistic samples.

It is the flow-matching analogue of “latent diffusion,” but instead of learning a score/noise predictor you learn a **velocity field** and sample via ODE integration.

#### Why do flow matching in latent space?

##### Modeling in pixel space can be unnecessarily hard
High-dimensional observations (images, audio frames, etc.) have:
- very large dimension $D$,
- complex local correlations,
- and perceptual structure that MSE in pixel space does not capture well.

Learning a transport field $v_\theta(x,t)$ directly in observation space can be slow and unstable because the model must handle every detail at full resolution.

#### A learned latent space reduces dimensionality and improves geometry
An encoder $E_\psi$ maps data $x \in \mathbb{R}^D$ to latents $z \in \mathbb{R}^d$ with $d \ll D$:

$
z = E_\psi(x).
$

The decoder $D_\psi$ reconstructs:

$
\hat{x} = D_\psi(z).
$

In a good latent space:
- semantic factors are more linear/separable,
- nuisance pixel-level variability is compressed away,
- and the distribution $p(z)$ is smoother than $p(x)$.

#### Flow matching benefits directly
Flow matching learns a vector field:

$
v_\theta(z,t)\in\mathbb{R}^d
$

and samples by integrating:

$
\frac{dz}{dt} = v_\theta(z,t),\qquad t\in[0,1].
$

Doing this in lower-dimensional latent space typically means:
- fewer compute-heavy network layers,
- fewer ODE steps needed for good quality,
- easier learning of the global transport.

### Overall architecture (high-level)

There are two learned modules:

#### Latent representation model (autoencoder or VAE)
- Encoder: $E_\psi: \mathbb{R}^D \to \mathbb{R}^d$ 
- Decoder: $D_\psi: \mathbb{R}^d \to \mathbb{R}^D$

Two common choices:

**Deterministic AE**
$
z = E_\psi(x),\qquad \hat{x}=D_\psi(z).
$

**VAE-style (stochastic encoder)**
$
q_\psi(z\mid x)=\mathcal{N}(\mu_\psi(x),\mathrm{diag}(\sigma^2_\psi(x))),
\qquad
z=\mu_\psi(x)+\sigma_\psi(x)\odot \epsilon,\ \epsilon\sim\mathcal{N}(0,I).
$

#### Latent flow matching model (velocity field)
A neural network (often U-Net-like for spatial latents, or MLP/Transformer) that outputs a latent velocity:

$
v_\theta(z,t)\in\mathbb{R}^d.
$

## Latent flow matching Mathematics

### Define distributions in latent space

Let $x\sim p_{\text{data}}$.

- **AE latents:**
$
z_1 = E_\psi(x).
$

- **VAE latents:**
$
z_1 \sim q_\psi(z\mid x).
$

Choose a simple base distribution in latent space:
$
z_0 \sim p_0(z)=\mathcal{N}(0,I).
$

**Goal:** learn a transport that maps $z_0$ samples into the data-latent distribution.

### Choose a bridge (intermediate latent points)

A common bridge is linear interpolation:
$
z_t=(1-t)z_0 + t z_1,\qquad t\in[0,1].
$

The corresponding “true” velocity along this synthetic path is:
$
u_t=\frac{dz_t}{dt}=z_1 - z_0.
$

Interpretation: $z_t$ is a latent point “between noise and data,” and $u_t$ is the wind (velocity) that would move a particle along that straight path.

### Flow matching loss in latent space

Train $v_\theta$ to match the target velocity $u_t$ at the bridge point $z_t$:

$$
\mathcal{L}_{\text{FM}}(\theta,\psi)
=
\mathbb{E}\left[
\left\|v_\theta(z_t,t) - (z_1 - z_0)\right\|_2^2
\right],
$$

where the expectation is over:
$$
x\sim p_{\text{data}},
\quad
z_1\sim q_\psi(z\mid x)\ \text{(or }z_1=E_\psi(x)\text{)},
\quad
z_0\sim p_0,
\quad
t\sim \mathrm{Uniform}[0,1].
$$

### Training the autoencoder (latent quality)

#### Option A: AE reconstruction objective
A deterministic autoencoder is typically trained with a reconstruction loss:
$$
\mathcal{L}_{\text{AE}}(\psi)
=
\mathbb{E}_{x\sim p_{\text{data}}}
\left[
\ell\big(x, D_\psi(E_\psi(x))\big)
\right].
$$

Here $\ell(\cdot,\cdot)$ can be pixel MSE, perceptual loss, or a combination (depending on the modality).

#### Option B: VAE objective (ELBO form)
A VAE trains a stochastic encoder using the ELBO:
$$
\mathcal{L}_{\text{VAE}}(\psi)
=
\mathbb{E}_{x\sim p_{\text{data}}}\left[
\mathbb{E}_{z\sim q_\psi(z\mid x)}[-\log p_\psi(x\mid z)]
+
\beta\,\mathrm{KL}\big(q_\psi(z\mid x)\ \|\ p(z)\big)
\right].
$$

This balances reconstruction quality with latent regularization.

### Practical training strategies

#### Strategy A: Two-stage (most common)
1. Train $E_\psi,D_\psi$ until reconstructions are good.
2. Freeze $\psi$ and train $v_\theta$ on latents.

This is stable because the latent space does not drift while learning the flow.

#### Strategy B: Joint training (end-to-end)
Train both modules with a combined objective:
$$
\mathcal{L}_{\text{total}}(\theta,\psi)=\mathcal{L}_{\text{AE/VAE}}(\psi) + \lambda\,\mathcal{L}_{\text{FM}}(\theta,\psi).
$$

Joint training can work but requires careful balancing so that:
- the autoencoder preserves reconstruction,
- the flow learns a consistent transport in the latent geometry.

### Sampling stage (generation)

After training (typically with a frozen decoder):

1. Sample base latent:
$
z(0)=z_0\sim \mathcal{N}(0,I).
$

2. Integrate the latent ODE:
$
\frac{dz}{dt}=v_\theta(z,t),\qquad t:0\to 1.
$

3. Decode:
$
\hat{x}=D_\psi(z(1)).
$

If $v_\theta$ is accurate, the distribution of $z(1)$ matches the data-latent distribution, and decoding yields realistic samples.

### Why latent flow matching can be fast

Latent flow matching is often faster than operating in observation space because:

- The ODE lives in $\mathbb{R}^d$ with $d \ll D$.
- The velocity network runs on compact latents (or low-resolution spatial latents).
- The latent distribution is often smoother, making the vector field easier to learn and integrate.
- Fewer ODE solver steps can be sufficient for high-quality samples.

A compact summary:

$$
x \in \mathbb{R}^D
\ \xrightarrow{E_\psi}\
z \in \mathbb{R}^d
\ \xrightarrow{\text{ODE: }dz/dt=v_\theta(z,t)}\
z(1)
\ \xrightarrow{D_\psi}\
\hat{x}.
$$

# Latent Flowmatching Example for Oxford Flowers II 

In [ ]:
# -----------------------------
# AE for Oxford Flowers 102 (RGB, img_size x img_size, outputs in [-1,1])
# Assumes transform:
# transforms.ToTensor(); transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))  # -> [-1,1]
# -----------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvEncoderFlowers_AE(nn.Module):
    def __init__(self, zdim=128, base=64, img_size=64):
        super().__init__()
        assert img_size % 16 == 0, "Use img_size divisible by 16 (e.g., 64, 128)."
        self.net = nn.Sequential(
            nn.Conv2d(3, base,     4, 2, 1), nn.SiLU(),      # 64 -> 32
            nn.Conv2d(base, base*2, 4, 2, 1), nn.SiLU(),     # 32 -> 16
            nn.Conv2d(base*2, base*4, 4, 2, 1), nn.SiLU(),   # 16 -> 8
            nn.Conv2d(base*4, base*8, 4, 2, 1), nn.SiLU(),   # 8  -> 4
        )
        s = img_size // 16
        feat_dim = base * 8 * s * s
        self.fc_z = nn.Linear(feat_dim, zdim)

    def forward(self, x):
        h = self.net(x).flatten(1)
        z = self.fc_z(h)
        return z


class AE_Flowers(nn.Module):
    def __init__(self, zdim=128, base=64, img_size=64):
        super().__init__()
        assert img_size % 16 == 0, "Use img_size divisible by 16 (e.g., 64, 128)."
        self.img_size = img_size
        self.base = base
        self.zdim = zdim
        self.s = img_size // 16

        self.enc = ConvEncoderFlowers_AE(zdim=zdim, base=base, img_size=img_size)
        self.dec_fc = nn.Linear(zdim, base * 8 * self.s * self.s)
        self.dec_net = nn.Sequential(
            nn.ConvTranspose2d(base*8, base*4, 4, 2, 1), nn.SiLU(),  # 4 -> 8
            nn.ConvTranspose2d(base*4, base*2, 4, 2, 1), nn.SiLU(),  # 8 -> 16
            nn.ConvTranspose2d(base*2, base,   4, 2, 1), nn.SiLU(),  # 16 -> 32
            nn.ConvTranspose2d(base,   3,      4, 2, 1),             # 32 -> 64
            nn.Tanh(),  # output in [-1,1]
        )

    def decode(self, z):
        h = self.dec_fc(z).view(z.shape[0], self.base*8, self.s, self.s)
        return self.dec_net(h)

    def forward(self, x):
        z = self.enc(x)
        xhat = self.decode(z)
        return xhat, z


def ae_loss(x, xhat, recon="l1"):
    # For [-1,1] images, L1 is a solid default; MSE is smoother.
    if recon == "l1":
        rec = F.l1_loss(xhat, x, reduction="mean")
    else:
        rec = F.mse_loss(xhat, x, reduction="mean")
    return rec


# Recommended starting values for Flowers102 @ 64x64
img_size = 64
zdim = 128      # try 64/128/256
base = 64       # try 64 (GPU) or 32 (CPU)

ae = AE_Flowers(zdim=zdim, base=base, img_size=img_size).to(device)
print("AE params:", sum(p.numel() for p in ae.parameters()))

In [ ]:
# -----------------------------
# Train AE (Oxford Flowers 102) — matches AE_Flowers forward: (xhat, z) = ae(x)
# Assumes images are normalized to [-1, 1].
# -----------------------------
import time
import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

ae_epochs = 200
ae_lr = 2e-4
recon_type = "l1"   # "l1" (sharper) or "mse" (smoother)

opt_ae = optim.AdamW(ae.parameters(), lr=ae_lr)
use_amp = (device.type == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

ae_hist = {"rec": []}
t0 = time.time()

def recon_loss_fn(x, xhat, recon="l1"):
    if recon == "l1":
        return F.l1_loss(xhat, x, reduction="mean")
    else:
        return F.mse_loss(xhat, x, reduction="mean")

for ep in range(1, ae_epochs + 1):
    ae.train()
    recs = []

    for xb, _ in train_loader:
        xb = xb.to(device)

        opt_ae.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            xhat, z = ae(xb)
            loss = recon_loss_fn(xb, xhat, recon=recon_type)

        scaler.scale(loss).backward()
        scaler.step(opt_ae)
        scaler.update()

        recs.append(loss.item())

    ae_hist["rec"].append(float(np.mean(recs)))

    # print + visualize every 50 epochs
    if ep % 50 == 0:
        print(f"[AE] ep {ep}/{ae_epochs}  rec={ae_hist['rec'][-1]:.4f}  min={(time.time()-t0)/60:.1f}")

        ae.eval()
        xb, _ = next(iter(train_loader))
        xb = xb.to(device)[:64]
        with torch.no_grad():
            xhat, _ = ae(xb)

        show_grid_any(xb,   title=f"AE input (ep {ep})", nrow=8)
        show_grid_any(xhat, title=f"AE recon (ep {ep})", nrow=8)

plt.figure(figsize=(8,4))
plt.plot(ae_hist["rec"], label="recon")
plt.title("AE training curve (reconstruction loss)")
plt.xlabel("epoch")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()

In [ ]:
# -----------------------------
# Latent Flow Matching (AE version)
#   Train v_theta(z,t) in AE latent space, using:
#     z0 ~ N(0,I)
#     z1 = E(x)         (deterministic AE encoder)
#     z_t = (1-t) z0 + t z1
#     target velocity u = z1 - z0
# -----------------------------
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

# Freeze AE (we train only the flow in latent)
ae.eval()
for p in ae.parameters():
    p.requires_grad = False

class SinusoidalTimeEmbedding1D(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        # t: [B,1] in [0,1]
        if t.ndim == 2:
            t = t[:, 0]
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000.0) * torch.arange(0, half, device=t.device).float() / max(half - 1, 1)
        )
        args = t[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
        if self.dim % 2 == 1:
            emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=1)
        return emb

class LatentVelocityMLP(nn.Module):
    def __init__(self, zdim=128, tdim=128, hidden=512):
        super().__init__()
        self.tproj = nn.Sequential(
            SinusoidalTimeEmbedding1D(tdim),
            nn.Linear(tdim, tdim), nn.SiLU(),
            nn.Linear(tdim, tdim)
        )
        self.net = nn.Sequential(
            nn.Linear(zdim + tdim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, zdim)
        )
    def forward(self, z, t):
        temb = self.tproj(t)           # [B,tdim]
        inp = torch.cat([z, temb], 1)  # [B,zdim+tdim]
        return self.net(inp)

# zdim must match the AE latent size you used when creating `ae`
# (if you used zdim earlier for AE_Flowers, reuse it here)
vtheta = LatentVelocityMLP(zdim=zdim, tdim=128, hidden=512).to(device)
print("Latent flow params:", sum(p.numel() for p in vtheta.parameters()))

@torch.no_grad()
def encode_to_latent(x):
    # deterministic AE latent: z1 = E(x)
    z1 = ae.enc(x)
    return z1

latent_epochs = 500      # increase (20–50) for better quality
latent_lr = 2e-4
opt_flow = optim.AdamW(vtheta.parameters(), lr=latent_lr)

flow_loss_hist = []
t0 = time.time()

for ep in range(1, latent_epochs + 1):
    vtheta.train()
    losses = []
    for xb, _ in train_loader:
        xb = xb.to(device)

        with torch.no_grad():
            z1 = encode_to_latent(xb)         # [B,zdim]
        z0 = torch.randn_like(z1)             # base latent noise
        t = torch.rand(z1.shape[0], 1, device=device)

        zt = (1 - t) * z0 + t * z1
        u  = z1 - z0

        opt_flow.zero_grad(set_to_none=True)
        pred = vtheta(zt, t)
        loss = F.mse_loss(pred, u)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(vtheta.parameters(), 1.0)
        opt_flow.step()

        losses.append(loss.item())

    flow_loss_hist.append(float(np.mean(losses)))
    if ep % 50 == 0:
        print(f"[LatentFlow-AE] ep {ep}/{latent_epochs}  loss={flow_loss_hist[-1]:.5f}  min={(time.time()-t0)/60:.1f}")

plt.figure(figsize=(8,4))
plt.plot(flow_loss_hist)
plt.title("Latent Flow Matching loss (AE latents)")
plt.xlabel("epoch")
plt.grid(True, alpha=0.2)
plt.show()

In [ ]:
# -----------------------------
# Sampling (AE): integrate latent ODE, then decode with AE decoder
#   z(0) ~ N(0,I)
#   dz/dt = v_theta(z,t)
# -----------------------------
import torch

@torch.no_grad()
def sample_latent_euler(vtheta, n=64, steps=60, zdim=zdim, device=device):
    vtheta.eval()
    z = torch.randn(n, zdim, device=device)
    dt = 1.0 / steps
    for k in range(steps):
        t = torch.full((n, 1), k * dt, device=device)
        dz = vtheta(z, t)
        z = z + dt * dz
    return z

@torch.no_grad()
def decode_images_ae(ae, z):
    """
    Decode latent vectors z -> RGB images in [-1,1] using the Flowers AE.
    Works with AE_Flowers class (defines ae.decode()).

    z: Tensor [B, zdim]
    returns: Tensor [B, 3, img_size, img_size] in [-1,1]
    """
    ae.eval()
    x = ae.decode(z)   # [-1,1]
    return x

# Generate samples
zgen = sample_latent_euler(vtheta, n=64, steps=80)
xgen = decode_images_ae(ae, zgen)
show_grid_any(xgen, title="Latent Flow Matching samples (AE latents)", nrow=8)

# Using VAE Instead

The code below shows how to do the same with a VAE. However, it is much more difficult to train the VAE to generate appropriate images. 

In [ ]:
# -----------------------------
# VAE for Oxford Flowers 102 (RGB, img_size x img_size, outputs in [-1,1])
# Assumes your Flowers102 transform is:
# transforms.ToTensor(); transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))  # -> [-1,1]
# Also assumes you resized/cropped to img_size (e.g., 64).
# -----------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvEncoderFlowers(nn.Module):
    def __init__(self, zdim=128, base=64, img_size=64):
        super().__init__()
        assert img_size % 16 == 0, "Use img_size divisible by 16 (e.g., 64, 128)."
        # img_size -> img_size/2 -> /4 -> /8 -> /16
        self.net = nn.Sequential(
            nn.Conv2d(3, base,   4, 2, 1), nn.SiLU(),        # 64 -> 32
            nn.Conv2d(base, base*2, 4, 2, 1), nn.SiLU(),     # 32 -> 16
            nn.Conv2d(base*2, base*4, 4, 2, 1), nn.SiLU(),   # 16 -> 8
            nn.Conv2d(base*4, base*8, 4, 2, 1), nn.SiLU(),   # 8  -> 4
        )
        s = img_size // 16  # for 64 -> 4
        feat_dim = base * 8 * s * s
        self.fc_mu = nn.Linear(feat_dim, zdim)
        self.fc_logvar = nn.Linear(feat_dim, zdim)

    def forward(self, x):
        h = self.net(x).flatten(1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar


class ConvDecoderFlowers(nn.Module):
    def __init__(self, zdim=128, base=64, img_size=64):
        super().__init__()
        assert img_size % 16 == 0, "Use img_size divisible by 16 (e.g., 64, 128)."
        s = img_size // 16  # 64 -> 4
        self.fc = nn.Linear(zdim, base * 8 * s * s)
        # s -> 2s -> 4s -> 8s -> 16s (= img_size)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(base*8, base*4, 4, 2, 1), nn.SiLU(),  # 4 -> 8
            nn.ConvTranspose2d(base*4, base*2, 4, 2, 1), nn.SiLU(),  # 8 -> 16
            nn.ConvTranspose2d(base*2, base,   4, 2, 1), nn.SiLU(),  # 16 -> 32
            nn.ConvTranspose2d(base,   3,      4, 2, 1),             # 32 -> 64
            nn.Tanh(),  # output in [-1,1]
        )

    def forward(self, z):
        # reshape to [B, base*8, s, s]
        # (use -1 for channels; s is known)
        B = z.shape[0]
        # infer s from fc output size / (base*8)
        h = self.fc(z)
        # We stored s in construction via fc shape, so reshape safely:
        # channels = base*8; spatial = img_size//16
        # We'll compute spatial from h length:
        # (h.shape[1] / (base*8)) must be s*s
        # But easiest: store s as an attribute.
        raise NotImplementedError("Use the VAE_Flowers class below which stores s cleanly.")


class VAE_Flowers(nn.Module):
    def __init__(self, zdim=128, base=64, img_size=64):
        super().__init__()
        self.img_size = img_size
        self.base = base
        self.zdim = zdim
        self.s = img_size // 16

        self.enc = ConvEncoderFlowers(zdim=zdim, base=base, img_size=img_size)
        self.dec_fc = nn.Linear(zdim, base * 8 * self.s * self.s)
        self.dec_net = nn.Sequential(
            nn.ConvTranspose2d(base*8, base*4, 4, 2, 1), nn.SiLU(),
            nn.ConvTranspose2d(base*4, base*2, 4, 2, 1), nn.SiLU(),
            nn.ConvTranspose2d(base*2, base,   4, 2, 1), nn.SiLU(),
            nn.ConvTranspose2d(base,   3,      4, 2, 1),
            nn.Tanh(),
        )

    def reparam(self, mu, logvar):
        eps = torch.randn_like(mu)
        std = torch.exp(0.5 * logvar)
        return mu + std * eps

    def decode(self, z):
        h = self.dec_fc(z).view(z.shape[0], self.base*8, self.s, self.s)
        return self.dec_net(h)

    def forward(self, x):
        mu, logvar = self.enc(x)
        z = self.reparam(mu, logvar)
        xhat = self.decode(z)
        return xhat, mu, logvar, z


def vae_loss(x, xhat, mu, logvar, beta=0.1, recon="l1"):
    # Recon loss: L1 often yields sharper recon; MSE is smoother
    if recon == "l1":
        rec = F.l1_loss(xhat, x, reduction="mean")
    else:
        rec = F.mse_loss(xhat, x, reduction="mean")

    # KL(q(z|x) || N(0,I))
    kl = 0.5 * torch.mean(torch.sum(torch.exp(logvar) + mu**2 - 1.0 - logvar, dim=1))
    return rec + beta * kl, rec, kl


# Recommended starting values for Flowers102 @ 64x64
img_size = 64
zdim = 128      # try 64/128/256
base = 64       # try 64 (GPU) or 32 (CPU)

vae = VAE_Flowers(zdim=zdim, base=base, img_size=img_size).to(device)
print("VAE params:", sum(p.numel() for p in vae.parameters()))

In [ ]:
# -----------------------------
# Train VAE
# -----------------------------
vae_epochs = 1         # increase to 20+ for better recon + generation
vae_lr = 2e-4
beta = 0.05              # try 0.1..1.0

beta_max = 0.1
warmup_epochs = 10


opt_vae = optim.AdamW(vae.parameters(), lr=vae_lr)
use_amp = (device.type == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

vae_hist = {"loss": [], "rec": [], "kl": []}
t0 = time.time()

for ep in range(1, vae_epochs + 1):
    vae.train()
    losses, recs, kls = [], [], []
    for xb, _ in train_loader:
        xb = xb.to(device)

        opt_vae.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            xhat, mu, logvar, z = vae(xb)
            # warm up KL term to avoid bad local minima at start of training
            beta = beta_max * min(1.0, ep / warmup_epochs)
            loss, rec, kl = vae_loss(xb, xhat, mu, logvar, beta=beta)

        scaler.scale(loss).backward()
        scaler.step(opt_vae)
        scaler.update()

        losses.append(loss.item()); recs.append(rec.item()); kls.append(kl.item())

    vae_hist["loss"].append(float(np.mean(losses)))
    vae_hist["rec"].append(float(np.mean(recs)))
    vae_hist["kl"].append(float(np.mean(kls)))

    print(f"[VAE] ep {ep}/{vae_epochs}  loss={vae_hist['loss'][-1]:.4f}  rec={vae_hist['rec'][-1]:.4f}  kl={vae_hist['kl'][-1]:.4f}  min={(time.time()-t0)/60:.1f}")

    # recon preview
    vae.eval()
    xb, _ = next(iter(train_loader))
    xb = xb.to(device)[:64]
    with torch.no_grad():
        xhat, _, _, _ = vae(xb)
    show_grid_any(xb,   title=f"VAE input (ep {ep})", nrow=8)
    show_grid_any(xhat, title=f"VAE recon (ep {ep})", nrow=8)

plt.figure(figsize=(8,4))
plt.plot(vae_hist["loss"], label="total")
plt.plot(vae_hist["rec"], label="recon")
plt.plot(vae_hist["kl"], label="KL")
plt.title("VAE training curves")
plt.xlabel("epoch")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()

In [ ]:
# -----------------------------
# Latent Flow Matching
#    Train v_theta(z,t) in latent space, using:
#      z0 ~ N(0,I)
#      z1 ~ q_phi(z|x)  (sampled from VAE encoder)
#      z_t = (1-t) z0 + t z1
#      target velocity u = z1 - z0
# -----------------------------

# Freeze VAE (we train only the flow in latent)
vae.eval()
for p in vae.parameters():
    p.requires_grad = False

class SinusoidalTimeEmbedding1D(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        # t: [B,1] in [0,1]
        if t.ndim == 2:
            t = t[:,0]
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(0, half, device=t.device).float() / (half - 1))
        args = t[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
        if self.dim % 2 == 1:
            emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=1)
        return emb

class LatentVelocityMLP(nn.Module):
    def __init__(self, zdim=64, tdim=128, hidden=512):
        super().__init__()
        self.tproj = nn.Sequential(
            SinusoidalTimeEmbedding1D(tdim),
            nn.Linear(tdim, tdim), nn.SiLU(),
            nn.Linear(tdim, tdim)
        )
        self.net = nn.Sequential(
            nn.Linear(zdim + tdim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, zdim)
        )
    def forward(self, z, t):
        temb = self.tproj(t)          # [B,tdim]
        inp = torch.cat([z, temb], 1) # [B,zdim+tdim]
        return self.net(inp)

vtheta = LatentVelocityMLP(zdim=zdim, tdim=128, hidden=512).to(device)
print("Latent flow params:", sum(p.numel() for p in vtheta.parameters()))

def encode_to_latent(x):
    # sample z1 ~ q(z|x)
    mu, logvar = vae.enc(x)
    eps = torch.randn_like(mu)
    z = mu + torch.exp(0.5*logvar) * eps
    return z

latent_epochs = 10      # increase (20–50) for better quality
latent_lr = 2e-4
opt_flow = optim.AdamW(vtheta.parameters(), lr=latent_lr)

flow_loss_hist = []
t0 = time.time()

for ep in range(1, latent_epochs + 1):
    vtheta.train()
    losses = []
    for xb, _ in train_loader:
        xb = xb.to(device)

        with torch.no_grad():
            z1 = encode_to_latent(xb)             # [B,zdim]
        z0 = torch.randn_like(z1)                 # base latent noise
        t = torch.rand(z1.shape[0], 1, device=device)

        zt = (1 - t) * z0 + t * z1
        u  = z1 - z0

        opt_flow.zero_grad(set_to_none=True)
        pred = vtheta(zt, t)
        loss = F.mse_loss(pred, u)
        loss.backward()
        opt_flow.step()

        losses.append(loss.item())

    flow_loss_hist.append(float(np.mean(losses)))
    print(f"[LatentFlow] ep {ep}/{latent_epochs}  loss={flow_loss_hist[-1]:.5f}  min={(time.time()-t0)/60:.1f}")

plt.figure(figsize=(8,4))
plt.plot(flow_loss_hist)
plt.title("Latent Flow Matching loss")
plt.xlabel("epoch")
plt.grid(True, alpha=0.2)
plt.show()

In [ ]:
# -----------------------------
# Sampling: integrate latent ODE, then decode
#    z(0) ~ N(0,I)
#    dz/dt = v_theta(z,t)
# -----------------------------
@torch.no_grad()
def sample_latent_euler(vtheta, n=64, steps=60):
    vtheta.eval()
    z = torch.randn(n, zdim, device=device)
    dt = 1.0 / steps
    for k in range(steps):
        t = torch.full((n,1), k*dt, device=device)
        dz = vtheta(z, t)
        z = z + dt * dz
    return z

@torch.no_grad()
def decode_images(z):
    """
    Decode latent vectors z -> RGB images in [-1,1] using the Flowers VAE.
    Works with the VAE_Flowers class I provided (which defines vae.decode()).
    
    z: Tensor [B, zdim]
    returns: Tensor [B, 3, img_size, img_size] in [-1,1]
    """
    vae.eval()
    x = vae.decode(z)   # [-1,1]
    return x

# Generate samples
zgen = sample_latent_euler(vtheta, n=64, steps=80)
xgen = decode_images(zgen)
show_grid_any(xgen, title="Latent Flow Matching samples", nrow=8)

In [ ]:
# Show trajectory snapshots in image space (Flowers)
# z(t) evolves under the ODE, decode intermediate images

import math
import matplotlib.pyplot as plt
import torch

@torch.no_grad()
def show_grid_rgb(imgs, title="", nrow=8):
    """
    imgs: [N,3,H,W] in [-1,1] or [0,1]
    """
    imgs = imgs.detach().cpu()
    if imgs.min().item() < -0.1:
        imgs = (imgs + 1) / 2  # [-1,1] -> [0,1]
    imgs = imgs.clamp(0, 1)

    N = imgs.shape[0]
    ncol = nrow
    nrow_actual = int(math.ceil(N / ncol))

    plt.figure(figsize=(ncol, nrow_actual))
    for i in range(N):
        plt.subplot(nrow_actual, ncol, i + 1)
        img = imgs[i].permute(1, 2, 0).numpy()
        plt.imshow(img)
        plt.axis("off")
    plt.suptitle(title)
    plt.show()


@torch.no_grad()
def latent_trajectory_snapshots_flowers(vtheta, vae, zdim, n=8, steps=50,
                                       times=(0.0, 0.25, 0.5, 0.75, 1.0),
                                       img_size=64, device=None):
    """
    vtheta: latent velocity model v_theta(z,t)
    vae: VAE_Flowers (must have vae.decode(z))
    zdim: latent dimension
    n: number of samples to visualize
    steps: Euler steps for ODE integration
    times: snapshot times in [0,1]
    """
    if device is None:
        device = next(vtheta.parameters()).device

    vtheta.eval()
    vae.eval()

    z = torch.randn(n, zdim, device=device)
    dt = 1.0 / steps

    snaps = {}
    for k in range(steps + 1):
        tval = k * dt

        # record snapshots when close to desired times
        for st in times:
            if (st not in snaps) and abs(tval - st) < 0.5 * dt:
                snaps[st] = z.detach().clone()

        if k == steps:
            break

        t = torch.full((n, 1), tval, device=device)
        z = z + dt * vtheta(z, t)

    # decode and show each snapshot as an RGB grid
    for st in times:
        x = vae.decode(snaps[st])  # [-1,1], shape [n,3,img_size,img_size]
        show_grid_rgb(x, title=f"Flowers: decoded latent trajectory at t={st:.2f}", nrow=n)

# Example usage (assuming you already have vtheta, vae, zdim):
latent_trajectory_snapshots_flowers(vtheta, vae, zdim=zdim, n=8, steps=60, times=(0,0.25,0.5,0.75,1.0), img_size=64, device=device)

## Class-Conditional, Unguided, and Classifier-Free Guided Generation in Flow Matching

Threre are three obvious ways :
1) **Unguided (unconditional) flow matching**
2) **Class-conditional (unguided) generation**
3) **Classifier-free guidance (CFG)** for *guided* generation

We assume the standard **ODE generative model**:

$$
\frac{dx}{dt} = v_\theta(x,t),
\qquad t\in[0,1].
$$

### Unguided / Unconditional Flow Matching

#### Training
Sample:
- noise (source): $x_0 \sim p_0 = \mathcal{N}(0,I)$
- data (target): $x_1 \sim p_{\text{data}}$
- time: $t \sim \mathrm{Unif}(0,1)$

Define a straight-line path:
$x_t = (1-t)x_0 + t x_1.$

Target velocity:
$\frac{d x_t}{dt} = x_1 - x_0.$

Train by regression:
$$
\mathcal{L}(\theta) =
\mathbb{E}_{x_0,x_1,t}\left[\left\|v_\theta(x_t,t) - (x_1-x_0)\right\|^2\right].
$$

#### Sampling (unguided)
Generate by integrating the ODE from $t=0$ to $t=1$:
- sample $x(0)\sim \mathcal{N}(0,I)$
- integrate $dx/dt=v_\theta(x,t)$
- output $x(1)$

### Class-Conditional (Unguided) Generation

Let each data point have a label $y$. We want the conditional distribution $p_{\text{data}}(x\mid y)$.

#### Conditional velocity field
$\frac{dx}{dt} = v_\theta(x,t,y).$

#### Training objective (same idea)
Sample labeled data $(x_1,y)\sim p_{\text{data}}(x,y)$, plus $x_0\sim \mathcal{N}(0,I)$ and $t\sim\mathrm{Unif}(0,1)$.
Construct $x_t=(1-t)x_0+t x_1$ and target velocity $x_1-x_0$.

Train:
$$
\mathcal{L}(\theta)=
\mathbb{E}_{x_0,(x_1,y),t}\left[
\left\|v_\theta(x_t,t,y) - (x_1-x_0)\right\|^2
\right].
$$

#### Conditional sampling (unguided)
To generate class $y$:
- sample $x(0)\sim \mathcal{N}(0,I)$
- integrate $dx/dt=v_\theta(x,t,y)$
- output $x(1)$

### Why “Guidance” is needed (intuition)

Even with conditional training, sampling may trade off:
- **diversity** vs **faithfulness** to the label
- some samples drift toward “generic” images that are plausible but weakly class-specific

Guidance amplifies the label signal during sampling.

### Classifier-Free Guidance (CFG) for Flow Matching

#### Key trick: train a model that can run with and without labels
During training, randomly “drop” the label with probability $p$.

Let $\varnothing$ denote “no label”.

Then the same network learns:
- conditional velocity: $v_\theta(x,t,y)$
- unconditional velocity: $v_\theta(x,t,\varnothing)$

The regression target stays the same:
$$
\left\|v_\theta(x_t,t,c) - (x_1-x_0)\right\|^2,
\quad\text{where}\quad
c \in \{y,\varnothing\}.
$$

#### CFG sampling formula
At sampling time, compute:
- $v_{\text{cond}} = v_\theta(x,t,y)$
- $v_{\text{uncond}} = v_\theta(x,t,\varnothing)$

Define the guided velocity:
$$
v_{\text{cfg}}(x,t,y)
=
v_{\text{uncond}}(x,t)
+
s\Big(v_{\text{cond}}(x,t,y) - v_{\text{uncond}}(x,t)\Big),
$$

equivalently:
$$
v_{\text{cfg}}(x,t,y)
=
(1+s)\,v_{\text{cond}}(x,t,y)
-
s\,v_{\text{uncond}}(x,t).
$$

Here $s \ge 0$ is the **guidance scale**.

#### What the formula means (principle)
- $v_{\text{uncond}}$: “generic realism” direction (match the dataset overall)
- $v_{\text{cond}} - v_{\text{uncond}}$: “class-specific push”
- scaling by $s$ amplifies the class-specific component

So CFG is:
- **realism** + $s$ × **class-specific push**

#### Effect of the guidance scale $s$
- $s=0$: unconditional sampling
- small $s$ (e.g., 1–2): better label fidelity, mild tradeoff
- large $s$ (e.g., 5–10): strong label enforcement, possible artifacts / less diversity

### “Classifier guidance” (external classifier) vs CFG

#### Classifier guidance (conceptual)
Train a classifier $p_\phi(y\mid x)$ and add a gradient push:

$$
\frac{dx}{dt}
=
v_\theta(x,t)
+
\lambda\, \nabla_x \log p_\phi(y\mid x).
$$

This requires a separate classifier and gradients through it during sampling.

#### Why CFG is preferred
CFG needs no external classifier:
- train one model with label dropout
- do two forward passes at sampling time and combine them

### The same ideas in Latent Flow Matching (VAE)

If you use a VAE and do flow matching in latent space $z$:

$
\frac{dz}{dt} = v_\theta(z,t,y).
$

CFG becomes:
$$
v_{\text{cfg}}(z,t,y)
=
(1+s)\,v_\theta(z,t,y)
-
s\,v_\theta(z,t,\varnothing).
$$

Then decode $z(1)$ back to image space.

### Summary
- **Unconditional**: learn $v_\theta(x,t)$ and integrate.
- **Conditional (unguided)**: learn $v_\theta(x,t,y)$ and integrate with chosen $y$.
- **CFG**: train with label dropout to get both conditional and unconditional velocities; sample with:
  $
  v_{\text{cfg}} = (1+s)v_{\text{cond}} - s v_{\text{uncond}}.
  $
As $s$ increases, label fidelity increases but diversity can decrease.

### Why train a model that can run *with and without* labels (Classifier-Free Guidance)

Classifier-Free Guidance (CFG) works best if the *same* generator can produce:
- a **conditional** velocity field: $v_\theta(x,t,y)$
- an **unconditional** velocity field: $v_\theta(x,t,\varnothing)$

Below is the rationale **intuitively** and **mathematically**.

### Intuition: “realism” vs “class-specificity” are different directions

Think of $v_\theta(x,t,y)$ as a **wind field** that moves a sample from noise to data.

At any intermediate point $(x,t)$, there are typically two competing objectives:

1) **Make it look like a plausible dataset image** (realism / generic structure)  
2) **Make it look like the particular class $y$** (class-specific details)

If you only have a conditional model, it tends to blend these two objectives in a way you can’t tune at sampling time.

CFG introduces a knob $s$ (“guidance strength”) that lets you decide:
- how much to prioritize *class identity*
- while still keeping the sample *on-manifold* (realistic)

To do that, you need a reference direction that represents “what the model would do without caring about $y$”:
- that is the unconditional velocity $v_\theta(x,t,\varnothing)$

Then the difference:
- $v_\theta(x,t,y) - v_\theta(x,t,\varnothing)$  
is interpreted as “the extra push that makes the sample class-$y$”.

Without an unconditional branch, you don’t have a clean baseline to subtract.

### Why not use a separate unconditional model?

You *could* train:
- one unconditional model $v_{\theta_u}(x,t)$
- one conditional model $v_{\theta_c}(x,t,y)$

But CFG is much cleaner when both behaviors live in **one shared network**:
- shared features for general image statistics
- shared time-conditioning machinery
- only the conditioning input changes

This makes the “baseline realism” and “class-specific push” consistent and comparable.

Practically, it also reduces:
- extra training time
- mismatch between two separately trained models
- implementation complexity

### Mathematical view: CFG as a decomposition of the velocity field

Assume there exists an (ideal) decomposition:
- a label-agnostic component $v_{\text{base}}(x,t)$
- a label-specific component $\Delta v_y(x,t)$

so that:
$v(x,t,y) = v_{\text{base}}(x,t) + \Delta v_y(x,t).$

If we can approximate:
- $v_{\text{base}}(x,t) \approx v_\theta(x,t,\varnothing)$
- $v(x,t,y) \approx v_\theta(x,t,y)$

then:
$$
\Delta v_y(x,t) \approx v_\theta(x,t,y) - v_\theta(x,t,\varnothing).
$$

CFG simply amplifies the label-specific term:
$$
v_{\text{cfg}}(x,t,y)
=
v_\theta(x,t,\varnothing) + s\Big(v_\theta(x,t,y) - v_\theta(x,t,\varnothing)\Big).
$$

Equivalently:
$$
v_{\text{cfg}}(x,t,y) = (1+s)v_\theta(x,t,y) - s\,v_\theta(x,t,\varnothing).
$$

Interpretation:
- $v_\theta(x,t,\varnothing)$ keeps you on the data manifold (“realism”)
- the difference term pushes you toward class $y$
- $s$ controls the tradeoff

### Why label dropout training produces both velocities

#### Training data for flow matching (recap)
We create:
- $x_0 \sim \mathcal{N}(0,I)$
- $(x_1,y) \sim p_{\text{data}}(x,y)$
- $t \sim \mathrm{Unif}(0,1)$
- $x_t = (1-t)x_0 + t x_1$
- target velocity $u = x_1 - x_0$

#### “Classifier-free” conditioning variable
Define a conditioning variable $c$ that is either:
- the label $y$, or
- the null symbol $\varnothing$

We sample:
- $c=y$ with probability $1-p_{\text{drop}}$
- $c=\varnothing$ with probability $p_{\text{drop}}$

Then train a single model $v_\theta(x,t,c)$ with the same regression objective:
$$
\mathcal{L}(\theta)
=
\mathbb{E}\left[
\left\|v_\theta(x_t,t,c) - u\right\|^2
\right].
$$

Because $c$ sometimes equals $\varnothing$, the model is explicitly trained to solve:
$$
v_\theta(x,t,\varnothing) \approx \mathbb{E}[u \mid x_t=x, t]
$$
which is the **unconditional** flow-matching velocity field.

And when $c=y$, it is trained to solve:
$$
v_\theta(x,t,y) \approx \mathbb{E}[u \mid x_t=x, t, y]
$$
which is the **class-conditional** velocity field.

So dropout makes the model learn both conditional expectations:
- one conditioned on $(x,t)$ only
- one conditioned on $(x,t,y)$

This is the core mathematical reason CFG works without an external classifier.

### Why the unconditional branch stabilizes guidance

If you crank $s$ large, you are effectively extrapolating beyond the conditional model:
$$
(1+s)v_{\text{cond}} - s v_{\text{uncond}}.
$$

The unconditional term acts like an “anchor”:
- it preserves global dataset structure
- it prevents the sample from drifting into weird off-manifold regions
- it improves stability vs using only the conditional direction

Empirically, this often yields:
- better class fidelity at moderate $s$
- fewer artifacts than naively “over-conditioning” the model

### Practical guidance values

- Dropout probability: $p_{\text{drop}} \in [0.1, 0.2]$ is common.
- Guidance scale: start with $s \in [1, 3]$ and increase if class identity is weak.

Large $s$ can reduce diversity and cause artifacts, so treat $s$ as a controllable tradeoff knob.

### Summary
Training with label dropout is essential because it gives you:
- $v_\theta(x,t,y)$ (conditional “do what class $y$ wants”)
- $v_\theta(x,t,\varnothing)$ (unconditional “do what the dataset wants”)

Then CFG is just:
- take the class-specific difference
- amplify it by $s$
- keep the unconditional baseline as an anchor

## Conditional Flow Matching Example (Flowers 102)

In [ ]:
# CONDITIONAL FLOW MATCHING (Flowers102)
# Assumes:
#   - train_loader yields (x1, y) with x1: [B,3,H,W] in [-1,1], y: [B] in {0..101}
#   - img_size, device are defined
#   - show_grid_rgb(imgs, title, nrow) is defined (RGB display)
#   - You are using straight-line flow matching: x_t=(1-t)x0 + t x1, target u=x1-x0

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# ---- Time embedding (same as before) ----
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
    def forward(self, t: torch.Tensor) -> torch.Tensor:
        # t: [B,1]
        if t.ndim == 2:
            t = t[:, 0]
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(0, half, device=t.device).float() / (half - 1))
        args = t[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
        if self.dim % 2 == 1:
            emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=1)
        return emb

# ---- ResBlock unchanged: takes t_emb and adds it as a channel-wise bias ----
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)

        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)

        self.t_proj = nn.Sequential(nn.SiLU(), nn.Linear(t_dim, out_ch))
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv2d(in_ch, out_ch, 1)

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.t_proj(t_emb)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)

class Down(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.Conv2d(ch, ch, 4, stride=2, padding=1)
    def forward(self, x):
        return self.op(x)

class Up(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.ConvTranspose2d(ch, ch, 4, stride=2, padding=1)
    def forward(self, x):
        return self.op(x)

# ---- Conditional VelocityUNet: label embedding is added to the time embedding ----
class ConditionalVelocityUNet(nn.Module):
    def __init__(self, n_classes=102, base_ch=64, t_dim=256, in_ch=3, out_ch=3):
        super().__init__()
        self.n_classes = n_classes
        self.t_dim = t_dim

        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(t_dim),
            nn.Linear(t_dim, t_dim), nn.SiLU(),
            nn.Linear(t_dim, t_dim),
        )
        self.label_emb = nn.Embedding(n_classes, t_dim)

        self.in_conv = nn.Conv2d(in_ch, base_ch, 3, padding=1)

        self.rb1 = ResBlock(base_ch, base_ch, t_dim)
        self.down1 = Down(base_ch)

        self.rb2 = ResBlock(base_ch, base_ch*2, t_dim)
        self.down2 = Down(base_ch*2)

        self.mid1 = ResBlock(base_ch*2, base_ch*2, t_dim)
        self.mid2 = ResBlock(base_ch*2, base_ch*2, t_dim)

        self.up2 = Up(base_ch*2)
        self.rb_up2 = ResBlock(base_ch*2 + base_ch*2, base_ch*2, t_dim)

        self.up1 = Up(base_ch*2)
        self.rb_up1 = ResBlock(base_ch*2 + base_ch, base_ch, t_dim)

        self.out_norm = nn.GroupNorm(8, base_ch)
        self.out_conv = nn.Conv2d(base_ch, out_ch, 3, padding=1)

    def forward(self, x, t, y):
        # t: [B,1], y: [B]
        t_emb = self.time_mlp(t) + self.label_emb(y)  # combine time + label

        x0 = self.in_conv(x)
        h1 = self.rb1(x0, t_emb)
        d1 = self.down1(h1)

        h2 = self.rb2(d1, t_emb)
        d2 = self.down2(h2)

        m = self.mid1(d2, t_emb)
        m = self.mid2(m, t_emb)

        u2 = self.up2(m)
        u2 = torch.cat([u2, h2], dim=1)
        u2 = self.rb_up2(u2, t_emb)

        u1 = self.up1(u2)
        u1 = torch.cat([u1, h1], dim=1)
        u1 = self.rb_up1(u1, t_emb)

        out = self.out_conv(F.silu(self.out_norm(u1)))
        return out

def flow_matching_batch_cond(x1, y):
    B = x1.shape[0]
    x0 = torch.randn_like(x1)
    t = torch.rand(B, 1, device=x1.device)
    t_img = t[:, :, None, None]
    xt = (1 - t_img) * x0 + t_img * x1
    u = x1 - x0
    return xt, u, t, y

@torch.no_grad()
def sample_conditional(model, y, n=64, steps=80):
    # y: int or Tensor [n]
    model.eval()
    x = torch.randn(n, 3, img_size, img_size, device=device)
    dt = 1.0 / steps
    if isinstance(y, int):
        y = torch.full((n,), y, device=device, dtype=torch.long)
    for k in range(steps):
        t = torch.full((n,1), k*dt, device=device)
        v = model(x, t, y)
        x = x + dt * v
    return x

# ---- Train loop (conditional) ----
model = ConditionalVelocityUNet(n_classes=102, base_ch=64, t_dim=256).to(device)
opt = optim.AdamW(model.parameters(), lr=2e-4)

epochs = 500  # increase for better results

for epoch in range(1, epochs + 1):  # increase for better results
    model.train()
    losses = []
    for x1, y in train_loader:
        x1 = x1.to(device)
        y = y.to(device)

        xt, u, t, y = flow_matching_batch_cond(x1, y)
        pred = model(xt, t, y)
        loss = F.mse_loss(pred, u)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        losses.append(loss.item())
    if epoch % 50 == 0:
        print(f"[COND] epoch {epoch} loss={sum(losses)/len(losses):.5f}")
        # sample 1 class for sanity
        xgen = sample_conditional(model, y=0, n=64, steps=80)
        show_grid_any(xgen, title=f"Conditional samples (class=0) epoch {epoch}", nrow=8)

In [ ]:
# Load the previous stored checkpoint and model for inference or further training

import torch

ckpt_path = "flower_flow_matching_vtheta.pt"
ckpt = torch.load(ckpt_path, map_location=device)

img_size = ckpt.get("img_size", 64)  # fallback if not saved
print("Loaded img_size:", img_size)

# Recreate the SAME model class/args you used for training
# Example for an unconditional VelocityUNet:
flow_model = VelocityUNet(base_ch=64, t_dim=256, in_ch=3, out_ch=3).to(device)

# Load weights
flow_model.load_state_dict(ckpt["model"])
flow_model.eval()
print("Loaded flow model from:", ckpt_path)

## Classifier Guided Flow Matching Example (Flowers 102)

### Training a small custom classifier

In [ ]:
# CLASSIFIER-GUIDED FLOW MATCHING on Oxford Flowers 102

#  Train a CLASSIFIER p_phi(y|x)

# -----------------------------
# Classifier p_phi(y|x)
# -----------------------------
class SmallFlowersClassifier(nn.Module):
    """
    Simple CNN classifier for 64x64 RGB.
    Output: logits [B, 102]
    """
    def __init__(self, n_classes=102, base=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, base, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  # 64->32

            nn.Conv2d(base, base*2, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  # 32->16

            nn.Conv2d(base*2, base*4, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  # 16->8

            nn.Conv2d(base*4, base*4, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1)),  # -> [B, base*4, 1, 1]
        )
        self.fc = nn.Linear(base*4, n_classes)

    def forward(self, x):
        h = self.features(x).flatten(1)
        return self.fc(h)

# -----------------------------
# Train classifier
# -----------------------------
clf = SmallFlowersClassifier(n_classes=102, base=64).to(device)
opt_clf = optim.AdamW(clf.parameters(), lr=2e-4)
clf_epochs = 100  # increase (10–20) for better guidance quality

@torch.no_grad()
def eval_acc(model, loader):
    model.eval()
    correct = 0
    total = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = model(xb)
        pred = logits.argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += yb.numel()
    return correct / max(1, total)

print("\nTraining classifier...")
t0 = time.time()
for ep in range(1, clf_epochs + 1):
    clf.train()
    losses = []
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = clf(xb)
        loss = F.cross_entropy(logits, yb)

        opt_clf.zero_grad(set_to_none=True)
        loss.backward()
        opt_clf.step()

        losses.append(loss.item())

    acc = eval_acc(clf, val_loader)
    if ep % 10 == 0:
        print(f"[CLF] ep {ep}/{clf_epochs} loss={np.mean(losses):.4f}  val_acc={acc*100:.2f}%  min={(time.time()-t0)/60:.1f}")


### Training a more powerful classifier using transfer learning

The small classifier did not work well. We can use transfer learning to train a more competent classifier. In doing so, one of the primary issues is the **input size** as most pre-trained **heads** are trained assuming a specific size (e.g., **256x256**) that forces us to **upsample** because our input size is **64x64**. In this case we use **ResNet-18** with a slight change to make it behave better for our **64x64** input size. 

In [ ]:
# ResNet-18 classifier for Flowers102 at 64x64x3 (NO up/downsampling beyond 64)
# Notebook-friendly, self-contained: downloads Flowers102, builds loaders, trains, validates, saves.
#
# Key idea: small-image ResNet-18 tweak:
#   - conv1: 7x7 stride2 -> 3x3 stride1
#   - maxpool removed
# This makes ResNet behave better on 64x64.

import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from torchvision.datasets import Flowers102
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

# ----------------------------
# Setup
# ----------------------------
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

root = "./data_flowers102"
img_size = 64
batch_size = 64
num_workers = 0  # notebook-safe
epochs = 50      # increase to 30+ for better accuracy
lr = 3e-4
weight_decay = 1e-4
num_classes = 102

# ImageNet normalization (good default for ResNet pretrained weights)
mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.Resize(img_size),
    transforms.CenterCrop(img_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

eval_tf = transforms.Compose([
    transforms.Resize(img_size),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# ----------------------------
# Dataset + Loaders
# ----------------------------
train_ds = Flowers102(root=root, split="train", download=True, transform=train_tf)
val_ds   = Flowers102(root=root, split="val",   download=True, transform=eval_tf)
test_ds  = Flowers102(root=root, split="test",  download=True, transform=eval_tf)

train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True,
    num_workers=num_workers, pin_memory=torch.cuda.is_available(), drop_last=True
)
val_loader = DataLoader(
    val_ds, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=torch.cuda.is_available(), drop_last=False
)
test_loader = DataLoader(
    test_ds, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=torch.cuda.is_available(), drop_last=False
)

xb, yb = next(iter(train_loader))
print("batch:", xb.shape, yb.shape)

# ----------------------------
# Model: ResNet-18 small-image tweak
# ----------------------------
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Small-image tweak
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()

# New classification head
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

print("params:", sum(p.numel() for p in model.parameters()))

# Optional: freeze most of the backbone for very fast training, then unfreeze later
freeze_backbone = False
if freeze_backbone:
    for name, p in model.named_parameters():
        if not name.startswith("fc."):
            p.requires_grad = False

# ----------------------------
# Train/Eval helpers
# ----------------------------
def top1_acc(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    losses, accs = [], []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        loss = nn.functional.cross_entropy(logits, y)
        losses.append(loss.item())
        accs.append(top1_acc(logits, y))
    return float(np.mean(losses)), float(np.mean(accs))

# ----------------------------
# Optimizer + AMP
# ----------------------------
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=lr,
    weight_decay=weight_decay
)

use_amp = (device.type == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# ----------------------------
# Training loop
# ----------------------------
best_val_acc = 0.0
best_state = None
t0 = time.time()

for epoch in range(1, epochs + 1):
    model.train()
    tr_losses, tr_accs = [], []

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(x)
            loss = nn.functional.cross_entropy(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        tr_losses.append(loss.item())
        tr_accs.append(top1_acc(logits.detach(), y))

    tr_loss = float(np.mean(tr_losses))
    tr_acc  = float(np.mean(tr_accs))

    va_loss, va_acc = evaluate(model, val_loader)

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if epoch % 10 == 0 or epoch == epochs:
        print(f"epoch {epoch:02d}/{epochs} | "
            f"train loss {tr_loss:.4f} acc {tr_acc*100:.2f}% | "
            f"val loss {va_loss:.4f} acc {va_acc*100:.2f}% | "
            f"best {best_val_acc*100:.2f}% | "
            f"min={(time.time()-t0)/60:.1f}")

# Restore best checkpoint
if best_state is not None:
    model.load_state_dict(best_state)

te_loss, te_acc = evaluate(model, test_loader)
print(f"TEST: loss {te_loss:.4f} acc {te_acc*100:.2f}%")

# ----------------------------
# Save
# ----------------------------
ckpt = {
    "model": model.state_dict(),
    "arch": "resnet18_small64",
    "img_size": img_size,
    "num_classes": num_classes,
    "mean": mean,
    "std": std,
}
torch.save(ckpt, "flowers102_resnet18_64x64.pt")
print("Saved: flowers102_resnet18_64x64.pt")

In [ ]:
# Rename the model to use in Classifier Guided Flow Matching section below

clf = model  # rename for clarity in the next steps

## Sampling with Classifier Guidance

In [ ]:
#  Sample with CLASSIFIER GUIDANCE:
#        dx/dt = v_theta(x,t) + lam * ∇_x log p_phi(y|x)
#
# Notes:
#  - This is "classifier guidance" (external classifier), not classifier-free guidance.
#  - Keep img_size modest (64) to make it runnable.
#  - For better results: increase epochs, model capacity, and sampling steps, or use Heun/RK2.

# -----------------------------
# Samplers
# -----------------------------
@torch.no_grad()
def sample_unconditional(flow_model, n=64, steps=80):
    flow_model.eval()
    x = torch.randn(n, 3, img_size, img_size, device=device)
    dt = 1.0 / steps
    for k in range(steps):
        t = torch.full((n,1), k*dt, device=device)
        v = flow_model(x, t)
        x = x + dt * v
    return x

def sample_classifier_guided(flow_model, clf, y_target, n=64, steps=80, lam=2.0):
    """
    Classifier-guided ODE integration:
      x_{k+1} = x_k + dt * ( v_theta(x_k,t_k) + lam * grad_x log p_phi(y|x_k) )
    """
    flow_model.eval()
    clf.eval()

    x = torch.randn(n, 3, img_size, img_size, device=device)
    dt = 1.0 / steps

    if isinstance(y_target, int):
        y_target = torch.full((n,), y_target, device=device, dtype=torch.long)

    for k in range(steps):
        t = torch.full((n,1), k*dt, device=device)

        # flow term (no grad needed for this part)
        with torch.no_grad():
            v = flow_model(x, t)

        # classifier gradient term wrt x
        x_in = x.detach().requires_grad_(True)
        logits = clf(x_in)
        logp = F.log_softmax(logits, dim=1)
        obj = logp[torch.arange(n, device=device), y_target].sum()   # sum log p(y|x)
        grad = torch.autograd.grad(obj, x_in, retain_graph=False, create_graph=False)[0]

        # update
        x = x + dt * (v + lam * grad)

        # optional: keep bounded to avoid blow-ups
        x = x.clamp(-3, 3)

    return x

# -----------------------------
# Classifier-guided
# -----------------------------
# Pick a target class ID in [0..101]. (Flowers102 labels are 0..101 in torchvision.)
target_class = 100
steps = 100
lam = 2.0

x_guided = sample_classifier_guided(flow_model, clf, y_target=target_class, n=64, steps=steps, lam=lam)
show_grid_any(x_guided, title=f"Classifier-guided samples (class={target_class}, lam={lam})", nrow=8)

## Classifier-Free Guidance Example (Flowers 102)

In [ ]:
# CLASSIFIER-FREE GUIDANCE (CFG) FLOW MATCHING (Flowers102)
# This trains ONE conditional model that can also run "unconditionally" by dropping labels.
#
# Training:
#  - with prob p_drop: replace y with a special null token (y_null = n_classes)
#  - model learns both v_theta(x,t,y) and v_theta(x,t,null)
#
# Sampling (CFG):
#   v_cfg = (1+s) * v_cond - s * v_uncond
# where:
#   v_cond   = v_theta(x,t,y)
#   v_uncond = v_theta(x,t,null)

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class CFGVelocityUNet(nn.Module):
    """
    Same as ConditionalVelocityUNet, but label embedding includes an extra token for 'null'.
    """
    def __init__(self, n_classes=102, base_ch=64, t_dim=256, in_ch=3, out_ch=3):
        super().__init__()
        self.n_classes = n_classes
        self.null_id = n_classes  # extra id
        self.t_dim = t_dim

        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(t_dim),
            nn.Linear(t_dim, t_dim), nn.SiLU(),
            nn.Linear(t_dim, t_dim),
        )
        self.label_emb = nn.Embedding(n_classes + 1, t_dim)  # +1 for null

        self.in_conv = nn.Conv2d(in_ch, base_ch, 3, padding=1)

        self.rb1 = ResBlock(base_ch, base_ch, t_dim)
        self.down1 = Down(base_ch)

        self.rb2 = ResBlock(base_ch, base_ch*2, t_dim)
        self.down2 = Down(base_ch*2)

        self.mid1 = ResBlock(base_ch*2, base_ch*2, t_dim)
        self.mid2 = ResBlock(base_ch*2, base_ch*2, t_dim)

        self.up2 = Up(base_ch*2)
        self.rb_up2 = ResBlock(base_ch*2 + base_ch*2, base_ch*2, t_dim)

        self.up1 = Up(base_ch*2)
        self.rb_up1 = ResBlock(base_ch*2 + base_ch, base_ch, t_dim)

        self.out_norm = nn.GroupNorm(8, base_ch)
        self.out_conv = nn.Conv2d(base_ch, out_ch, 3, padding=1)

    def forward(self, x, t, y):
        t_emb = self.time_mlp(t) + self.label_emb(y)
        x0 = self.in_conv(x)

        h1 = self.rb1(x0, t_emb)
        d1 = self.down1(h1)

        h2 = self.rb2(d1, t_emb)
        d2 = self.down2(h2)

        m = self.mid1(d2, t_emb)
        m = self.mid2(m, t_emb)

        u2 = self.up2(m)
        u2 = torch.cat([u2, h2], dim=1)
        u2 = self.rb_up2(u2, t_emb)

        u1 = self.up1(u2)
        u1 = torch.cat([u1, h1], dim=1)
        u1 = self.rb_up1(u1, t_emb)

        out = self.out_conv(F.silu(self.out_norm(u1)))
        return out

def flow_matching_batch_cfg(x1, y, null_id, p_drop=0.1):
    B = x1.shape[0]
    x0 = torch.randn_like(x1)
    t = torch.rand(B, 1, device=x1.device)
    t_img = t[:, :, None, None]
    xt = (1 - t_img) * x0 + t_img * x1
    u = x1 - x0

    # label dropout
    drop = (torch.rand(B, device=x1.device) < p_drop)
    y2 = y.clone()
    y2[drop] = null_id
    return xt, u, t, y2

@torch.no_grad()
def sample_cfg(model, y, n=64, steps=80, s=3.0):
    """
    model: CFGVelocityUNet
    y: int or Tensor [n] (label for conditional branch)
    s: guidance scale
    """
    model.eval()
    x = torch.randn(n, 3, img_size, img_size, device=device)
    dt = 1.0 / steps

    if isinstance(y, int):
        y = torch.full((n,), y, device=device, dtype=torch.long)
    y_null = torch.full((n,), model.null_id, device=device, dtype=torch.long)

    for k in range(steps):
        t = torch.full((n,1), k*dt, device=device)

        v_cond = model(x, t, y)
        v_uncond = model(x, t, y_null)

        v_cfg = (1.0 + s) * v_cond - s * v_uncond
        x = x + dt * v_cfg

    return x

# ---- Train loop (CFG) ----
cfg_model = CFGVelocityUNet(n_classes=102, base_ch=64, t_dim=256).to(device)
opt = optim.AdamW(cfg_model.parameters(), lr=2e-4)

p_drop = 0.1  # 0.1–0.2 is common

epochs = 500  # increase for better results
for epoch in range(1,epochs ):  # increase for better results
    cfg_model.train()
    losses = []
    for x1, y in train_loader:
        x1 = x1.to(device)
        y = y.to(device)

        xt, u, t, y2 = flow_matching_batch_cfg(x1, y, null_id=cfg_model.null_id, p_drop=p_drop)
        pred = cfg_model(xt, t, y2)
        loss = F.mse_loss(pred, u)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        losses.append(loss.item())
    if epoch % 50 == 0: 
        print(f"[CFG] epoch {epoch} loss={sum(losses)/len(losses):.5f}")
        # sample with CFG
        xgen = sample_cfg(cfg_model, y=0, n=64, steps=80, s=3.0)
        show_grid_any(xgen, title=f"CFG samples (class=0, s=3) epoch {epoch}", nrow=8)

## Optimal-Transport (OT) Couplings for Flow Matching

You already know **rectilinear (straight-line) flow matching**: we pick endpoints $x_0\sim p_0$ (noise) and $x_1\sim p_{\text{data}}$ (data), define a straight path $x_t=(1-t)x_0+t x_1$, and train a velocity field $v_\theta(x,t)$ to match the path derivative.

OT-based flow matching keeps the *same* straight-line path **but changes the pairing** between noise samples and data samples. That pairing choice is the whole point.

### Why pairings matter

In the standard rectilinear baseline, $x_0$ and $x_1$ are often paired **independently** (random pairing). This makes many segments unnecessarily long and can create “messy” velocity fields that must explain transport paths that cut through low-density regions.

OT says: instead of random pairings, find a coupling (matching) between *batches* of noise and data that moves mass “efficiently.”

### OT coupling summarized

Optimal transport finds a coupling $\pi(x_0,x_1)$ that moves probability mass from $p_0$ to $p_1$ while minimizing an expected cost, typically squared distance:
$$
\pi^\star \in \arg\min_{\pi\in\Pi(p_0,p_1)} \; \mathbb{E}_{(x_0,x_1)\sim \pi}\left[\|x_0-x_1\|^2\right].
$$

Here $\Pi(p_0,p_1)$ is the set of all joint distributions (couplings) whose marginals are $p_0$ and $p_1$.

### OT displacement interpolation gives “nice” straight trajectories

A key idea in flow matching is that you can choose different **probability paths** (how distributions evolve with $t$). Lipman et al. highlight **OT displacement interpolation** as a particularly interesting choice of path for flow matching. 

Intuition:
- OT pairing tends to align $x_0$ with a “nearby” $x_1$ (globally, not just locally),
- so straight lines from $x_0$ to $x_1$ are shorter and more coherent,
- which makes the learned velocity field easier to approximate and often easier to integrate at sampling time.

### How OT flow matching looks mathematically

#### Choose the coupling
Instead of sampling independent pairs, sample $(x_0,x_1)$ from an OT coupling $\pi^\star$.

In practice (mini-batch setting), you approximate OT using:
- a discrete assignment / matching for a batch, or
- an entropic OT plan (Sinkhorn) for soft couplings.

#### Keep the same rectilinear path
Define the straight-line interpolation:
$$
x_t = (1-t)x_0 + t x_1.
$$

The velocity along that straight segment is still:
$$
\frac{dx_t}{dt} = x_1 - x_0.
$$

#### Train as usual
You still do flow matching regression, but now the expectation is under the OT coupling:
$$
\mathcal{L}(\theta)
=
\mathbb{E}_{(x_0,x_1)\sim \pi^\star,\; t\sim \mathrm{Unif}(0,1)}
\left[
\left\|v_\theta(x_t,t) - (x_1-x_0)\right\|^2
\right].
$$

So the *code structure* can look very similar to rectilinear FM; the difference is that OT changes which $x_1$ is paired with each $x_0$.

### Why it often works “really well”

#### Shorter, straighter transport implies easier learning
OT reduces the average transport distance:
$$
\mathbb{E}_{\pi^\star}\|x_1-x_0\|^2
\;\;\le\;\;
\mathbb{E}_{\pi}\|x_1-x_0\|^2
\quad\text{for other couplings }\pi.
$$
So the network sees velocity targets with smaller magnitude and more consistent direction fields.

#### More coherent vector fields implies fewer solver steps
A velocity field that corresponds to “straight” transport typically integrates with fewer function evaluations (practically, fewer Euler/Heun steps) to get good samples.

A line of work (e.g., “Optimal Flow Matching”) explicitly targets learning *straight OT-like trajectories* because straightness helps fast integration.

#### Minibatch OT is a pragmatic compromise
Exact OT at full dataset scale is expensive. Many practical OT-FM approaches compute OT within minibatches or use Sinkhorn-style approximations; recent work focuses on scaling these couplings and showing they improve flow matching quality.

### Summary

- **Flow matching** is a framework: learn $v_\theta(x,t)$ by regressing to a target velocity induced by a chosen probability path.
- **Rectilinear FM** chooses straight paths but uses easy (often random) couplings.
- **OT Flow Matching** still uses straight paths, but chooses *better pairings* $(x_0,x_1)$ via optimal transport, which often yields straighter, shorter, more learnable trajectories and improves sample quality / efficiency. :contentReference[oaicite:5]{index=5}

### References
- Y. Lipman et al., *Flow Matching for Generative Modeling* (arXiv:2210.02747)
- N. Kornilov et al., *Optimal Flow Matching: Learning Straight Trajectories in Generative Modeling* (NeurIPS 2024).
- A. Mousavi-Hosseini et al., *On Fitting Flow Models with Large Sinkhorn Couplings* (OpenReview/NeurIPS workshop, 2025).
- F. Santambrogio, *Optimal Transport for Applied Mathematicians* (OT background).

## Hungarian Algorithm 

### Problem it solves
You have a **cost matrix** $C \in \mathbb{R}^{n\times n}$ where $C_{ij}$ is the cost of pairing item $i$ (e.g., noise sample $x_0^i$) with item $j$ (e.g., data sample $x_1^j$).

You want a **one-to-one matching** (a permutation $\pi$) that minimizes total cost:

$$
\min_{\pi \in S_n} \sum_{i=1}^{n} C_{i,\pi(i)}.
$$

This is the classic **linear assignment problem**.

### Basic idea
The Hungarian algorithm finds the **globally optimal** matching efficiently by:
- transforming the cost matrix with **row/column reductions** (subtracting minima),
- finding a set of **zero-cost edges** that can cover all rows/columns,
- and iteratively adjusting the matrix until a perfect zero-based matching exists.

The important conceptual point:

> It does not “pick the cheapest available edge one by one.”  
> It manipulates the matrix so the optimal assignment can be read off as a set of zeros.

### Why it’s useful for Optimal Transport in minibatches
In minibatch OT for flow matching, you often build
$$
C_{ij} = \|x_0^i - x_1^j\|^2
$$
and want the matching that minimizes total squared distance.

Hungarian gives the **exact optimal** one-to-one assignment for that batch.

### Complexity and practical notes
- Time complexity is typically $O(n^3)$.
- For batch sizes like 128–512, it can become a bottleneck on CPU.
- That’s why people sometimes use approximate OT (e.g., Sinkhorn) or greedy approximations.

### Summary
- **Output:** a permutation $\pi$ that gives the globally minimum total matching cost.
- **Guarantee:** optimal for the batch.
- **Tradeoff:** more expensive than greedy for large $n$.

## Example: Hungarian Algorithm (minibatch OT for flow matching)

Assume we sampled a minibatch:
- **2 noise points** (source) $x_0^1, x_0^2 \sim \mathcal{N}(0,1)$
- **2 data points** (target) $x_1^1, x_1^2 \sim p_{\text{data}}$

We build a cost matrix using squared distance:
$$
C_{ij} = \|x_0^i - x_1^j\|^2.
$$

In minibatch OT assignment, we choose the **one-to-one matching** (a permutation) minimizing:
$$
\min_{\pi \in S_2} \sum_{i=1}^{2} C_{i,\pi(i)}.
$$

Let:
- Noise: $x_0^1 = -0.2$, $x_0^2 = 1.1$
- Data:  $x_1^1 = 0.0$,  $x_1^2 = 1.5$

Compute costs:

Row 1 ($x_0^1=-0.2$):
- to $x_1^1=0.0$: $(-0.2-0.0)^2 = 0.04$
- to $x_1^2=1.5$: $(-0.2-1.5)^2 = (-1.7)^2 = 2.89$

Row 2 ($x_0^2=1.1$):
- to $x_1^1=0.0$: $(1.1-0.0)^2 = 1.21$
- to $x_1^2=1.5$: $(1.1-1.5)^2 = (-0.4)^2 = 0.16$

So:
$$
C=
\begin{bmatrix}
0.04 & 2.89\\
1.21 & 0.16
\end{bmatrix}.
$$

### Solving the OT assignment

There are only two one-to-one matchings:

1) Identity matching: $(1\to1,\ 2\to2)$  
Total cost:
$$
C_{11} + C_{22} = 0.04 + 0.16 = 0.20
$$

2) Swapped matching: $(1\to2,\ 2\to1)$  
Total cost:
$$
C_{12} + C_{21} = 2.89 + 1.21 = 4.10
$$

The minimum is the identity matching:
$$
x_0^1 \rightarrow x_1^1,\qquad x_0^2 \rightarrow x_1^2.
$$

### Integrating with flow matching

After OT pairing, for each matched pair $(x_0, x_1)$ we define:
$$
x_t = (1-t)x_0 + t x_1,
\qquad
u = \frac{dx_t}{dt} = x_1 - x_0.
$$

So the target velocities are:
- Pair 1: $u^1 = x_1^1 - x_0^1 = 0.0 - (-0.2) = 0.2$
- Pair 2: $u^2 = x_1^2 - x_0^2 = 1.5 - 1.1 = 0.4$

Training then regresses:
$$
v_\theta(x_t,t) \approx u
$$
but with **OT-matched** endpoints rather than random pairings.

In [ ]:
# ============================================================
# Optimal Flow Matching (OFM) on MNIST — Standalone Implementation
# ============================================================
# What makes this "optimal":
#   - Standard (rectilinear) flow matching pairs x0 ~ N(0,I) with x1 ~ data *randomly*
#   - OFM pairs the *batch* of x0 with the *batch* of x1 using an (approx) Optimal Transport
#     assignment that minimizes total cost sum_i ||x0_i - x1_{pi(i)}||^2
#
# Then we still use the same straight path:
#   x_t = (1-t) x0 + t x1_matched
# and target velocity:
#   u = x1_matched - x0
# and train v_theta(x_t,t) ≈ u
#
# Sampling:
#   x(0) ~ N(0,I)
#   dx/dt = v_theta(x,t)
#   integrate t:0->1 (Euler)
#
# Notebook-safe: num_workers=0, no main, no external deps required.
# If SciPy is available, we use Hungarian algorithm for the optimal assignment.
# Otherwise we fall back to a greedy approximate assignment.

import math, time, os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# -------------------------
# Setup
# -------------------------
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# -------------------------
# Data (MNIST -> [-1,1])
# -------------------------
transform = transforms.Compose([
    transforms.ToTensor(),                 # [0,1]
    transforms.Normalize((0.5,), (0.5,))   # [-1,1]
])

# Download MNIST
train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=0, drop_last=True)

# -------------------------
# Display helper
# -------------------------
@torch.no_grad()
def show_grid_gray(imgs, title="", nrow=8):
    imgs = imgs.detach().cpu()
    if imgs.min().item() < -0.1:
        imgs = (imgs + 1) / 2
    imgs = imgs.clamp(0, 1)
    N = imgs.shape[0]
    ncol = nrow
    nrow_actual = int(math.ceil(N / ncol))
    plt.figure(figsize=(ncol, nrow_actual))
    for i in range(N):
        plt.subplot(nrow_actual, ncol, i + 1)
        plt.imshow(imgs[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
        plt.axis("off")
    plt.suptitle(title)
    plt.show()

# -------------------------
# Time embedding
# -------------------------
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        # t: [B,1] in [0,1] -> [B,dim]
        if t.ndim == 2:
            t = t[:, 0]
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(0, half, device=t.device).float() / (half - 1)
        )
        args = t[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
        if self.dim % 2 == 1:
            emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=1)
        return emb

# -------------------------
# Velocity U-Net (small) for MNIST
# -------------------------
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)

        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)

        # time embedding -> per-channel bias
        self.t_proj = nn.Sequential(nn.SiLU(), nn.Linear(t_dim, out_ch))
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv2d(in_ch, out_ch, 1)

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        # Add time embedding as a bias after the first conv
        # : : t_emb is [B, t_dim], we project to [B, out_ch], then add as bias
        # None, None to broadcast over spatial dims
        h = h + self.t_proj(t_emb)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        # adding the skip connection
        return h + self.skip(x)

class Down(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.Conv2d(ch, ch, 4, stride=2, padding=1)  # 28->14->7
    def forward(self, x):
        return self.op(x)

class Up(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.ConvTranspose2d(ch, ch, 4, stride=2, padding=1)  # 7->14->28
    def forward(self, x):
        return self.op(x)

class VelocityUNetMNIST(nn.Module):
    def __init__(self, base=64, t_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(t_dim),
            nn.Linear(t_dim, t_dim), nn.SiLU(),
            nn.Linear(t_dim, t_dim),
        )

        self.in_conv = nn.Conv2d(1, base, 3, padding=1)

        self.rb1 = ResBlock(base, base, t_dim)
        self.down1 = Down(base)           # 28->14

        self.rb2 = ResBlock(base, base*2, t_dim)
        self.down2 = Down(base*2)         # 14->7

        self.mid1 = ResBlock(base*2, base*2, t_dim)
        self.mid2 = ResBlock(base*2, base*2, t_dim)

        self.up2 = Up(base*2)             # 7->14
        self.rb_up2 = ResBlock(base*2 + base*2, base*2, t_dim)

        self.up1 = Up(base*2)             # 14->28
        self.rb_up1 = ResBlock(base*2 + base, base, t_dim)

        self.out_norm = nn.GroupNorm(8, base)
        self.out_conv = nn.Conv2d(base, 1, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_mlp(t)

        x0 = self.in_conv(x)
        h1 = self.rb1(x0, t_emb)
        d1 = self.down1(h1)

        h2 = self.rb2(d1, t_emb)
        d2 = self.down2(h2)

        m = self.mid1(d2, t_emb)
        m = self.mid2(m, t_emb)

        u2 = self.up2(m)
        u2 = torch.cat([u2, h2], dim=1)
        u2 = self.rb_up2(u2, t_emb)

        u1 = self.up1(u2)
        u1 = torch.cat([u1, h1], dim=1)
        u1 = self.rb_up1(u1, t_emb)

        out = self.out_conv(F.silu(self.out_norm(u1)))
        return out

# -------------------------
# OT matching: Hungarian if available, else greedy
# -------------------------
def try_hungarian(cost_np: np.ndarray):
    """
    Solve min_{perm} sum_i cost[i, perm[i]] using Hungarian algorithm.
    Requires SciPy. If not available, returns None.
    """
    try:
        from scipy.optimize import linear_sum_assignment
        row_ind, col_ind = linear_sum_assignment(cost_np)
        # row_ind should be [0..B-1]
        return col_ind
    except Exception:
        return None

def greedy_assignment(cost: torch.Tensor):
    """
    Greedy approximate assignment.
    cost: [B,B] torch tensor on CPU
    Returns perm: [B] where perm[i] is assigned column for row i.
    """
    B = cost.shape[0]
    cost_np = cost.numpy()
    used = np.zeros(B, dtype=bool)
    perm = np.zeros(B, dtype=np.int64)

    # Assign rows in increasing order of their best cost (helps a bit vs naive row-order)
    row_order = np.argsort(cost_np.min(axis=1))

    for i in row_order:
        j = np.argmin(np.where(used, np.inf, cost_np[i]))
        perm[i] = j
        used[j] = True
    return perm

@torch.no_grad()
def ot_pair_batch(x0: torch.Tensor, x1: torch.Tensor):
    """
    Given two batches:
      x0: [B,1,28,28] noise samples
      x1: [B,1,28,28] data samples
    compute an (approx) OT assignment pi minimizing sum ||x0_i - x1_{pi(i)}||^2
    and return x1_matched: [B,1,28,28]
    """
    B = x0.shape[0]
    x0f = x0.view(B, -1)
    x1f = x1.view(B, -1)

    # cost matrix: squared L2
    # torch.cdist gives L2; square it for squared cost
    cost = torch.cdist(x0f, x1f, p=2).pow(2)  # [B,B]

    # Hungarian on CPU (if available)
    cost_cpu = cost.detach().cpu()
    perm = try_hungarian(cost_cpu.numpy())
    if perm is None:
        perm = greedy_assignment(cost_cpu)

    perm = torch.from_numpy(perm).to(x1.device)
    x1_matched = x1[perm]
    return x1_matched

# -------------------------
# OFM batch construction
# -------------------------
def ofm_training_batch(x1: torch.Tensor):
    """
    x1: [B,1,28,28] data
    1) sample x0 ~ N(0,I)
    2) OT-match x0 batch to x1 batch -> x1_matched
    3) sample t ~ U(0,1)
    4) x_t = (1-t)x0 + t x1_matched
    5) target u = x1_matched - x0
    """
    B = x1.shape[0]
    x0 = torch.randn_like(x1)
    x1m = ot_pair_batch(x0, x1)      # OT coupling within batch

    t = torch.rand(B, 1, device=x1.device)
    t_img = t[:, :, None, None]
    xt = (1 - t_img) * x0 + t_img * x1m
    u = x1m - x0
    return xt, u, t

# -------------------------
# Sampling (Euler)
# -------------------------
@torch.no_grad()
def sample_ofm(model, n=64, steps=60):
    model.eval()
    x = torch.randn(n, 1, 28, 28, device=device)
    dt = 1.0 / steps
    for k in range(steps):
        t = torch.full((n, 1), k * dt, device=device)
        v = model(x, t)
        x = x + dt * v
    return x

# -------------------------
# Train OFM on MNIST
# -------------------------
model = VelocityUNetMNIST(base=64, t_dim=128).to(device)
opt = optim.AdamW(model.parameters(), lr=2e-4)

epochs = 100    # for a nicer demo, try 10–20 on GPU
steps = 60

t0 = time.time()
for ep in range(1, epochs + 1):
    model.train()
    losses = []
    for x1, _ in train_loader:
        x1 = x1.to(device)

        # OFM: same FM regression, but OT-matched endpoints
        xt, u, t = ofm_training_batch(x1)
        pred = model(xt, t)
        loss = F.mse_loss(pred, u)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        losses.append(loss.item())
    if ep % 10 == 0:
        print(f"[OFM] epoch {ep}/{epochs} loss={float(np.mean(losses)):.5f}  min={(time.time()-t0)/60:.1f}")
        xgen = sample_ofm(model, n=64, steps=steps)
        show_grid_gray(xgen, title=f"OFM samples (epoch {ep})", nrow=8)

print("Done.")
# Optional save
torch.save({"model": model.state_dict(), "t_dim": 128, "base": 64}, "mnist_ofm_vtheta.pt")
print("Saved: mnist_ofm_vtheta.pt")

## Consistency Methods for Generation

Consistency methods are generative models that learn a mapping which is **self-consistent across noise levels (or “times”)**, enabling **very fast sampling**—often **1–2 steps**.

They can be viewed as a bridge between:
- **diffusion / score models** (many-step denoising),
- **rectified flows / ODE transport** (continuous-time deterministic sampling),
- and **distillation** (compressing many steps into a few).

### The Basic Idea: “same object at different noise levels should map to the same clean sample”

In diffusion-like processes you have a continuum of noisy versions of the same underlying clean sample:
- $x_t$ is a noisy state at time $t$ (high noise for large $t$, low noise for small $t$).

A *consistency model* learns a function $f_\theta$ such that:

> If you take the same underlying example and observe it at two different noise levels, both should map to the same clean output.

In other words, the model is trained so that “denoise-to-clean” is **consistent** across time.

This is the key that allows **jumping directly** from a noisy point to a clean sample without iterating many times.

### Core consistency condition (the defining equation)

Let $f_\theta(x,t)$ be a “canonicalization” function that maps a state at time $t$ to a reference time (often $t=0$, i.e., a clean sample).

A consistency condition is:

$$
f_\theta(x_{t_1}, t_1) \approx f_\theta(x_{t_2}, t_2),
\qquad \text{for } t_1 > t_2,
$$

where $(x_{t_1}, x_{t_2})$ are two states from the **same forward noising process** (so they are coupled).

Interpretation:
- $f_\theta$ should produce the **same “denoised identity”** regardless of which noise level you start from.

Often the “reference” is the clean endpoint:
$$
f_\theta(x_t,t) \approx x_0.
$$

### How to get paired states $(x_{t_1}, x_{t_2})$

To enforce consistency you need pairs that correspond to the same underlying sample.

Common options:
1) **Use the known forward noising process** (diffusion-style) to sample $x_{t_1}$ and then obtain $x_{t_2}$ from the same chain.
2) **Use a teacher model / solver** (score model, rectified flow, diffusion sampler) to map between noise levels and create consistent pairs.

The key is: $(x_{t_1}, x_{t_2})$ should be meaningfully linked.

### A typical training objective (pairwise consistency loss)

One canonical form:

$$
\mathcal{L}(\theta)
=
\mathbb{E}\left[
\left\| f_\theta(x_{t_1}, t_1) - f_\theta(x_{t_2}, t_2) \right\|^2
\right].
$$

Some variants include:
- weighted losses (emphasize certain noise levels),
- additional “boundary” losses to encourage $f_\theta(x_0,0)\approx x_0$,
- perceptual losses (for images),
- or a combination with diffusion/flow-style regression.

### Sampling (why it’s fast)

Once $f_\theta$ is trained, you can sample with:
- **one step**: draw $x_T \sim p_T$ (pure noise) and output
  $$
  \hat{x}_0 = f_\theta(x_T, T).
  $$

Or **few steps** (improved quality):
- define a small sequence of times $T=t_K > t_{K-1} > \dots > t_0=0$
- repeatedly “project” to the canonical form and optionally re-noise to the next level
- in practice: 2–4 steps can be enough for strong results

This is the practical advantage over diffusion sampling, which may require tens to hundreds of steps.

### Relation to flow matching/Rectified flow

You already know flow matching learns a velocity field and samples by integration:

$$
\frac{dx}{dt}=v_\theta(x,t).
$$

Consistency methods can be seen as learning a **direct map** that “solves the flow” in big jumps:
- Instead of integrating small steps, learn a function that moves you close to the final clean point immediately.

In this sense, consistency models are:
- an **amortized solver** for the generative dynamics
- a **distilled** version of multi-step generation

So:
- **Flow/rectified flow**: learn dynamics, then integrate
- **Consistency**: learn a map that is invariant across time, enabling large jumps

### When to use consistency methods

They are ideal when:
- you need **ultra-fast sampling** (real-time or edge constraints)
- you can afford a more involved training pipeline (often involves teacher/paired states)
- you want diffusion-like quality with far fewer steps

Tradeoffs:
- training can be more delicate
- requires careful time schedules / pair sampling
- the best results often use teacher distillation or hybrid losses

## Summary
Consistency methods enforce:
- **time-invariant denoising / canonicalization**
- by training $f_\theta(x,t)$ so that its output is consistent across noise levels:
  $$
  f_\theta(x_{t_1}, t_1) \approx f_\theta(x_{t_2}, t_2).
  $$

This enables **one-step or few-step generation**, making them a powerful acceleration path beyond standard flow matching or rectified flows.

## Discrete vs. Continuous diffusion formulations 

Diffusion models can be written in two closely related ways:

- **Discrete-time diffusion (DDPM-style):** a finite Markov chain with timesteps $t=1,\dots,T$.
- **Continuous-time diffusion (SDE/ODE-style):** a stochastic differential equation evolving over $t\in[0,T]$ (or $t\in[0,1]$.

Both describe the same idea: gradually transform data $x_0\sim p_{\text{data}}$ into noise.

### Discrete-time diffusion (DDPM)

### Stepwise forward process
Define a noise schedule \(\beta_t\in(0,1)\) and
$$
\alpha_t = 1-\beta_t.
$$

Forward transition:
$$
q(x_t\mid x_{t-1})
=
\mathcal{N}\!\left(\sqrt{\alpha_t}\,x_{t-1},\ (1-\alpha_t)I\right).
$$

Equivalent sampling:
$$
x_t = \sqrt{\alpha_t}\,x_{t-1} + \sqrt{1-\alpha_t}\,\varepsilon_t,
\qquad \varepsilon_t\sim\mathcal{N}(0,I).
$$

### Closed-form marginal from $x_0$
Define:
$$
\bar{\alpha}_t=\prod_{i=1}^t \alpha_i.
$$

Then:
$$
x_t = \sqrt{\bar{\alpha}_t}\,x_0 + \sqrt{1-\bar{\alpha}_t}\,\varepsilon,
\qquad \varepsilon\sim\mathcal{N}(0,I).
$$

**What’s “discrete” here?**  
Time is an integer index $t\in\{0,1,\dots,T\}$, and the forward process is a finite chain.

### Continuous-time diffusion (SDE view)

Instead of a chain, define a stochastic differential equation (SDE):
$$
dx = f(x,t)\,dt + g(t)\,dW_t,
\qquad t\in[0,T],
$$
where $W_t$ is Brownian motion.

A common form is **VP diffusion** (variance-preserving):
$$
dx = -\frac{1}{2}\beta(t)\,x\,dt + \sqrt{\beta(t)}\,dW_t.
$$

Here $\beta(t)$ is a *continuous-time* noise schedule.

**What’s “continuous” here?**  
Time is real-valued. Noise is injected continuously through $dW_t$.

#### VP-style parameterization (closed-form marginal)

For VP diffusion, the marginal distribution is Gaussian and can be written in the familiar “mix data + noise” form:
$$
x_t = \alpha(t)\,x_0 + \sigma(t)\,\varepsilon,
\qquad \varepsilon\sim\mathcal{N}(0,I),
$$
with the **variance-preserving constraint**:
$$
\alpha(t)^2 + \sigma(t)^2 = 1.
$$

A canonical choice derived from $\beta(t)$ is:
$$
\alpha(t)=\exp\!\left(-\frac{1}{2}\int_0^t \beta(s)\,ds\right),
\qquad
\sigma(t)=\sqrt{1-\alpha(t)^2}.
$$

This is the continuous-time analogue of the DDPM marginal
$$
x_t = \sqrt{\bar{\alpha}_t}\,x_0 + \sqrt{1-\bar{\alpha}_t}\,\varepsilon.
$$

**Mapping intuition:**
- Discrete: $\sqrt{\bar{\alpha}_t}$ plays the role of $\alpha(t)$
- Discrete: $\sqrt{1-\bar{\alpha}_t}$ plays the role of $\sigma(t)$

### Key differences (practical)

#### Time representation
- **DDPM:** $t$ is an integer index; often embedded as $\tfrac{t}{T}$ for networks.
- **Continuous/SDE:** $t$ is real-valued; naturally supports ODE/SDE solvers.

### Forward sampling
- **DDPM:** either stepwise $x_{t-1}\to x_t$, or closed-form via $\bar{\alpha}_t$.
- **Continuous:** typically uses the closed-form $\alpha(t),\sigma(t)$ (if available), or integrates the SDE.

### Reverse generation
- **DDPM:** reverse Markov chain $p_\theta(x_{t-1}\mid x_t)$ (with samplers like DDPM/DDIM).
- **Continuous:** reverse-time SDE or its associated **probability flow ODE** (samplers via numerical integration).

### Summary

**Discrete diffusion** is a finite noise-adding Markov chain parameterized by $\beta_t$, while **continuous diffusion** is an SDE over real time; **VP-style** is a continuous formulation whose marginals can be written as a clean mixture $x_t=\alpha(t)x_0+\sigma(t)\varepsilon$, directly analogous to the DDPM $\sqrt{\bar{\alpha}_t}$ form.

### Problem Formulation

Let $x_0 \in \mathbb{R}^d$ be a clean data sample drawn from the data distribution:

$
x_0 \sim p_{\text{data}}.
$

We define a **forward noising process** (diffusion-style corruption) so that for each continuous time $t \in [0,T]$ we can sample a noisy version $x_t$ of $x_0$ in **closed form**:

$$
x_t = \alpha(t)\,x_0 + \sigma(t)\,\varepsilon,
\qquad
\varepsilon \sim \mathcal{N}(0, I).
$$

A common requirement on the schedule is:

$$
\alpha(t)^2 + \sigma(t)^2 = 1,
\qquad
\alpha(0)=1,\ \sigma(0)=0,
\qquad
\alpha(T)\approx 0,\ \sigma(T)\approx 1.
$$

**Intuition**

- At $t=0$, $x_t = x_0$ (no noise).
- At $t=T$, $x_T \approx \varepsilon$ (almost pure noise).

This forward process defines a family of Gaussian marginals $q(x_t \mid x_0)$ that are tractable.

### What is a Consistency Model?

A **consistency model** is a function

$$
f_\theta : \mathbb{R}^d \times [0,T] \to \mathbb{R}^d
\quad\text{with}\quad
f_\theta(x,t) \approx \text{a canonical representation of the underlying clean sample}.
$$

Most commonly, we want the canonical representation to be the clean sample itself:

$
f_\theta(x_t,t) \approx x_0.
$

The key idea is **time-invariance along a forward trajectory** generated from the same underlying $x_0$ and the same noise $\varepsilon$.

### The consistency condition (core constraint)

Take any two times $0 \le s < t \le T$. If we generate both noisy states using the same pair $(x_0,\varepsilon)$:

$$
x_s = \alpha(s)\,x_0 + \sigma(s)\,\varepsilon,
\qquad
x_t = \alpha(t)\,x_0 + \sigma(t)\,\varepsilon,
$$

then both $x_s$ and $x_t$ lie on the same forward-noise path.

A perfect consistency function would satisfy:

$
f_\theta(x_s,s) = f_\theta(x_t,t).
$

If additionally we want the canonical output to equal the clean sample, then along the same path we want:

$
f_\theta(x_s,s) = f_\theta(x_t,t) = x_0.
$


### Consistency Training (teacher-free): objective

In practice, we enforce the condition with a squared loss over randomly sampled pairs (often neighboring times on a grid):

$$
\mathcal{L}_{\text{cons}}(\theta)
=
\mathbb{E}_{x_0,\varepsilon,s,t}
\left[
\left\|
f_\theta(x_s,s) - f_\theta(x_t,t)
\right\|_2^2
\right].
$$

The condition $f_\theta(x_s,s)=f_\theta(x_t,t)$ alone does not uniquely define what the shared value should be; for example, a constant function could satisfy it.

So we add a **boundary (anchor) condition** at $t=0$:

$
f_\theta(x_0,0) \approx x_0,
$

implemented with:

$$
\mathcal{L}_{\text{bdry}}(\theta)
=
\mathbb{E}_{x_0 \sim p_{\text{data}}}
\left[
\left\|
f_\theta(x_0,0) - x_0
\right\|_2^2
\right].
$$

The full teacher-free consistency training loss is:

$$
\mathcal{L}(\theta)
=
\mathcal{L}_{\text{cons}}(\theta)
+
\lambda\,\mathcal{L}_{\text{bdry}}(\theta).
$$

### Discrete-time version

Choose a time grid:

$
0 = t_0 < t_1 < \dots < t_K = T.
$

A common choice is to enforce consistency between neighbors $t_k$ and $t_{k+1}$.

For each training step:

1. Sample $x_0 \sim p_{\text{data}}$ and $\varepsilon \sim \mathcal{N}(0,I)$.
2. Pick $k \in \{0,\dots,K-1\}$ and set:

$$
s = t_k,\qquad t=t_{k+1}.
$$

3. Forward-sample:

$$
x_s = \alpha(s)\,x_0 + \sigma(s)\,\varepsilon,
\qquad
x_t = \alpha(t)\,x_0 + \sigma(t)\,\varepsilon.
$$

4. Compute loss:

$$
\mathcal{L}(\theta)
=
\left\| f_\theta(x_s,s) - f_\theta(x_t,t) \right\|_2^2
+
\lambda \left\| f_\theta(x_0,0) - x_0 \right\|_2^2.
$$

5. Update by gradient descent:

$$
\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}(\theta).
$$

### Consistency Distillation (student from a diffusion teacher)

Consistency distillation uses an existing **teacher diffusion model** (e.g., an $\varepsilon_\phi(x_t,t)$ predictor or a score model) to define a high-quality mapping from a larger time $t$ to a smaller time $s<t$.

We denote a single teacher step as:

$
\tilde{x}_s = \mathrm{TeacherStep}(x_t, t \to s).
$

Then we train $f_\theta$ to be consistent between:

- the original noisy point $x_t$ at time $t$, and
- the teacher-improved point $\tilde{x}_s$ at time $s$.

The distillation loss is:

$$
\mathcal{L}_{\text{distill}}(\theta)
=
\mathbb{E}
\left[
\left\|
f_\theta(x_t,t) - f_\theta(\tilde{x}_s,s)
\right\|_2^2
\right]
+
\lambda\,\mathcal{L}_{\text{bdry}}(\theta).
$$

This tends to produce a consistency model that matches the teacher’s behavior but can be sampled in very few steps.

### Why sampling can be 1-step (or few-step)

After training, we can sample starting from noise at $t=T$:

$
x_T \sim \mathcal{N}(0,I).
$

If training succeeded such that:

$
f_\theta(x_t,t) \approx x_0,
$

then in particular:

$
\hat{x}_0 = f_\theta(x_T,T)
$

is a **one-step generator**: it maps pure noise to a clean sample.

## Summary

Forward/noising:

$
x_t = \alpha(t)\,x_0 + \sigma(t)\,\varepsilon,\qquad \varepsilon\sim\mathcal{N}(0,I).
$

Consistency:

$$
f_\theta(x_s,s) \approx f_\theta(x_t,t)\quad \text{for } x_s,x_t \text{ generated from same } (x_0,\varepsilon).
$$

Anchor:

$
f_\theta(x_0,0) \approx x_0.
$

Total objective (CT):

$$
\mathcal{L}(\theta)
=
\mathbb{E}\left[\| f_\theta(x_s,s)-f_\theta(x_t,t)\|_2^2\right]
+
\lambda\,\mathbb{E}\left[\| f_\theta(x_0,0)-x_0\|_2^2\right].
$$

One-step sampling:

$$
x_T \sim \mathcal{N}(0,I),
\qquad
\hat{x}_0 = f_\theta(x_T,T).
$$

## Example: 1D Consistency Model on a Bimodal Distribution

We model a 1D bimodal distribution:
$$
p_{\text{data}}(x) = \tfrac{1}{2}\mathcal{N}(-\mu,\sigma_{\text{data}}^2) + \tfrac{1}{2}\mathcal{N}(+\mu,\sigma_{\text{data}}^2).
$$

Forward noising (closed-form marginals):
$$
x_t = \alpha(t)\,x_0 + \sigma(t)\,\varepsilon,\qquad \varepsilon\sim \mathcal{N}(0,1),
$$
with a schedule satisfying:
$$
\alpha(t)^2 + \sigma(t)^2 = 1.
$$

Define the signal and noise coefficients:
$$
\alpha(t) = e^{-t},
\qquad
\sigma(t) = \sqrt{1 - e^{-2t}}.
$$

Then the variance-preserving property holds:
$$
\alpha(t)^2 + \sigma(t)^2
=
e^{-2t} + \left(1 - e^{-2t}\right)
=
1.
$$

We train a consistency model $$f_\theta(x,t)\approx x_0$$ by enforcing:
$$
f_\theta(x_s,s)\approx f_\theta(x_t,t)\quad \text{for } s<t \text{ on the same forward path},
$$
and anchoring:
$$
f_\theta(x_0,0)\approx x_0.
$$

Loss (neighbor times):
$$
\mathcal{L}(\theta)=\mathbb{E}\left[\|f_\theta(x_s,s)-f_\theta(x_t,t)\|_2^2\right]
+ \lambda\,\mathbb{E}\left[\|f_\theta(x_0,0)-x_0\|_2^2\right].
$$

Sampling (1-step):
$$
x_T \sim \mathcal{N}(0,1),\qquad \hat{x}_0 = f_\theta(x_T,T).
$$

## Simple Example Following the Equations

In [ ]:
# Imports + device
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

# Data distribution (bimodal GMM) + schedule

@torch.no_grad()
def sample_gmm(n, mu=2.0, sigma_data=0.5, p=0.5, device=device):
    # Mixture: N(-mu, sigma_data^2) and N(+mu, sigma_data^2)
    u = torch.rand(n, 1, device=device)
    choose_left = (u < p).float()
    means = choose_left * (-mu) + (1.0 - choose_left) * (+mu)
    x0 = means + sigma_data * torch.randn(n, 1, device=device)
    return x0

# Schedule: alpha(t)=exp(-t), sigma(t)=sqrt(1-exp(-2t)) so alpha^2+sigma^2=1
def alpha(t):
    return torch.exp(-t)

def sigma(t):
    return torch.sqrt(1.0 - torch.exp(-2.0*t))

@torch.no_grad()
def forward_sample(x0, t):
    # x0: (B,1), t: (B,1)
    eps = torch.randn_like(x0)
    xt = alpha(t) * x0 + sigma(t) * eps
    return xt, eps

# Time embedding + model f_theta(x,t) -> x0_hat

def sinusoidal_time_embedding(t, dim=32):
    """
    t: (B,1) in [0,T]
    returns: (B,dim)
    """
    half = dim // 2
    freqs = torch.exp(
        -math.log(10000.0) * torch.arange(0, half, device=t.device).float() / max(half - 1, 1)
    )
    args = t * freqs.view(1, -1)  # (B, half)
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
    if dim % 2 == 1:
        emb = F.pad(emb, (0,1))
    return emb

# This class implements a simple MLP that takes (x,t) as input and predicts f_theta(x,t).

class ConsistencyMLP(nn.Module):
    def __init__(self, time_dim=32, hidden=128):
        super().__init__()
        self.time_dim = time_dim
        self.net = nn.Sequential(
            nn.Linear(1 + time_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, x, t):
        # x: (B,1), t: (B,1)
        te = sinusoidal_time_embedding(t, self.time_dim)
        inp = torch.cat([x, te], dim=1)
        return self.net(inp)

model = ConsistencyMLP(time_dim=32, hidden=128).to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
# Training loop (Consistency Training: neighbor pairs + boundary anchor)

def train_consistency(
    model,
    steps=1000,
    batch_size=1024,
    mu=2.0,
    sigma_data=0.5,
    T=3.0,
    K=32,
    lr=2e-4,
    lam_bdry=1.0,
    print_every=1000,
):
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    # discrete time grid not caring about exact values, just need neighbors, so we can precompute and index into it
    ts = torch.linspace(0.0, T, K+1, device=device).view(-1)  # (K+1,)
    losses = []

    for it in range(1, steps+1):
        # sample x0, eps once
        x0 = sample_gmm(batch_size, mu=mu, sigma_data=sigma_data, device=device)  # (B,1)
        eps = torch.randn_like(x0)

        # pick neighbor times: s=t_k, t=t_{k+1}

        # pick a random k in [0..K-1] and use the pair (t_k, t_{k+1}) as neighbors for this batch
        k = torch.randint(0, K, (1,), device=device).item()
        s = ts[k].view(1,1).expand(batch_size,1)
        t = ts[k+1].view(1,1).expand(batch_size,1)

        # forward samples with SAME eps
        xs = alpha(s)*x0 + sigma(s)*eps
        xt = alpha(t)*x0 + sigma(t)*eps

        # model outputs (canonical ~ x0)
        # the model is trained to predict the velocity f_theta(x,t) = dx/dt, which for the straight path is (x1-x0) = (xt-xs)
        ys = model(xs, s)
        yt = model(xt, t)

        # boundary anchor at t=0
        t0 = torch.zeros_like(s)
        # at t=0, we want the model to predict zero velocity (since x0 is already a perfect sample from the data distribution)
        y0 = model(x0, t0)

        # losses
        L_cons = F.mse_loss(ys, yt)
        L_bdry = F.mse_loss(y0, x0)
        L = L_cons + lam_bdry * L_bdry

        opt.zero_grad(set_to_none=True)
        L.backward()
        opt.step()

        losses.append(L.item())

        if it % print_every == 0:
            print(f"step {it:6d} | L={L.item():.6f}  (cons={L_cons.item():.6f}, bdry={L_bdry.item():.6f})")

    return ts, losses

ts, losses = train_consistency(
    model,
    steps=20000,
    batch_size=1024,
    mu=2.0,
    sigma_data=0.5,
    T=1.0,
    K=64,
    lr=2e-4,
    lam_bdry=1.0,
    print_every=1000,
)

In [ ]:
# Plot training loss
plt.figure()
plt.plot(losses)
plt.title("Training loss: L = L_cons + λ L_bdry")
plt.xlabel("step")
plt.ylabel("loss")
plt.show()

In [ ]:
# Evaluate how well it models the bimodal distribution

@torch.no_grad()
def one_step_sample(model, n=20000, T=1.0):
    # Start from noise at time T: x_T ~ N(0,1)
    xT = torch.randn(n, 1, device=device)
    tT = torch.full((n,1), T, device=device)
    xhat0 = model(xT, tT)
    return xhat0

@torch.no_grad()
def true_sample(n=20000, mu=2.0, sigma_data=0.5):
    return sample_gmm(n, mu=mu, sigma_data=sigma_data, device=device)

# Generate
N = 30000
x_true = true_sample(N, mu=2.0, sigma_data=0.5).cpu().numpy().flatten()
x_gen  = one_step_sample(model, n=N, T=float(ts[-1].item())).cpu().numpy().flatten()

# Histograms (overlay)
plt.figure()
plt.hist(x_true, bins=120, density=True, alpha=0.6, label="true data")
plt.hist(x_gen,  bins=120, density=True, alpha=0.6, label="one-step f(x_T,T)")
plt.title("Bimodal fit: true vs generated (one-step)")
plt.xlabel("x")
plt.ylabel("density")
plt.legend()
plt.show()

## Why consistency-based models are tricky to train

Consistency models try to learn a map that is **invariant across noise levels**—often something like:

$$
f_\theta(x_t, t)\approx x_0
$$

and enforce (for the same underlying data/noise trajectory):

$$
f_\theta(x_s, s)\approx f_\theta(x_t, t)\qquad (s<t).
$$

This idea is powerful (few-step or one-step sampling), but training is fragile because the objective can become **ill-posed** or **degenerate** unless you add stabilizers.

### Core difficulty: the problem becomes ambiguous at high noise

Forward corruption (VP-style or DDPM marginal) has the generic form:

$$
x_t = \alpha(t)\,x_0 + \sigma(t)\,\varepsilon.
$$

As $t$ increases, the signal-to-noise ratio typically collapses:

$$
\mathrm{SNR}(t)=\frac{\alpha(t)^2}{\sigma(t)^2}.
$$

When $\mathrm{SNR}(t)\ll 1$, the observation $x_t$ contains almost no information about which $x_0$ generated it.

If you train with an L2/MSE regression loss, the optimal solution in ambiguous regions is the **conditional mean**:

$$
f^*(x_t,t)=\mathbb{E}[x_0\mid x_t].
$$

For multimodal data (e.g., a symmetric bimodal mixture), this mean can sit **between modes**, causing classic **mode averaging / collapse**.

**Symptom:** samples cluster around the mean (e.g., around 0 for a symmetric bimodal).

#### Pitfall: pure consistency loss can admit trivial solutions

If your training objective is mainly:

$$
\mathcal{L}_{\text{cons}}
=
\mathbb{E}\left[\left\|f_\theta(x_s,s)-f_\theta(x_t,t)\right\|^2\right],
$$

then a constant function is consistent:

$$
f_\theta(x,t)=c
\quad\Rightarrow\quad
f_\theta(x_s,s)=f_\theta(x_t,t)=c.
$$

So consistency **alone** does not force correctness.

**Symptom:** quick collapse to a constant output or a narrow distribution

#### Pitfall: “both sides drift together” (coupled collapse)

If you backprop through both sides of the consistency term, the network can minimize the loss by moving *both* predictions toward an easy fixed point.

**Fix:** use one-way / stop-gradient consistency:

$$
\mathcal{L}_{\text{cons}}
=
\left\|f_\theta(x_s,s)-\mathrm{sg}\!\left(f_\theta(x_t,t)\right)\right\|^2,
$$

where $\mathrm{sg}(\cdot)$ stops gradients.

This prevents the “target” side from moving during the update.

#### Pitfall: off-policy multi-step sampling (error accumulation)

Many setups train the student as a **one-step distillation** model:

$$
f_\theta(x_t,t)\approx x_0^{\text{teacher}}.
$$

But if you then sample with multiple steps, intermediate states come from the student's own errors:

- those states may not appear in training,
- errors accumulate,
- the chain drifts toward an incorrect attractor.

**Symptom:** increasing number of steps does not improve quality; sometimes it gets worse.

**Fixes:**

- train explicitly for K-step behavior (multi-step distillation),
- use a teacher/trajectory method designed for multi-step,
- or sample in the model’s native regime (one-step if trained for one-step).

#### Pitfall: trajectory mismatch (resampling noise breaks “same path”)

Consistency requires that $\left(x_s,x_t\right)$ are on the **same forward trajectory**.

If you do:

- sample $x_t$ with one noise draw $\varepsilon$,
- sample $x_s$ with a different fresh draw $\varepsilon$,

then $\left(x_s,x_t\right)$ are **not** paired states from one trajectory, and the loss becomes noisy or biased.

**Fix:** sample once and reuse:

- sample $x_0$,
- sample $\varepsilon$,
- compute both $x_s$ and $x_t$ using the same $\varepsilon$ (VP marginal), or use a teacher step that preserves the same inferred $\hat{\varepsilon}$.

#### Pitfall: scale / base distribution mismatch

A common sampling pattern is:

$$
x_T \sim \mathcal{N}(0,I),
\qquad
\hat{x}_0=f_\theta(x_T,T).
$$

But this only makes sense if the forward process actually yields $x_T$ close to $\mathcal{N}(0,I)$ (or you trained under that assumption).

If your data is not unit-scale, or your schedule makes $x_T$ have a different variance, the model sees a mismatch between training and sampling.

**Fixes:**

- normalize data to roughly unit variance, or
- sample $x_T$ from a moment-matched base:

$$
x_T\sim \mathcal{N}\!\left(\alpha(T)\mu_0,\ \alpha(T)^2\sigma_0^2+\sigma(T)^2\right),
$$

where $\mu_0$ and $\sigma_0^2$ are estimated from data.

#### Pitfall: too much training mass at extremely noisy times

If $t\sim\mathcal{U}(0,T)$ and your schedule makes $\alpha(t)$ tiny for much of the range, training is dominated by ambiguous inputs.

**Fixes:**

- cap the maximum time used in training, e.g. $t\le t_{\max}$,
- warp time sampling toward lower noise

$$
t = T\cdot u^\gamma,
\qquad
u\sim\mathcal{U}(0,1),
\qquad
\gamma>1,
$$

- sample uniformly in log-SNR.

## Practical “recipe” of stabilizers (what usually works)

A robust consistency-student loss often combines:

#### (A) Distillation or reconstruction anchor

Teacher-guided:

$$
\mathcal{L}_{\text{distill}}
=
\left\|f_\theta(x_t,t)-\hat{x}_0^{(\text{teacher})}(x_t,t)\right\|^2.
$$

No-teacher (synthetic setting where $$x_0$$ is available):

$$
\mathcal{L}_{\text{recon}}
=
\left\|f_\theta(x_t,t)-x_0\right\|^2.
$$

#### (B) One-way consistency (stop-gradient)

$$
\mathcal{L}_{\text{cons}}
=
\left\|f_\theta(x_s,s)-\mathrm{sg}\!\left(f_\theta(x_t,t)\right)\right\|^2.
$$

#### (C) Boundary / anchor term

Force correctness at low noise:

$$
\mathcal{L}_{\text{bdry}}
=
\left\|f_\theta(x_0,0)-x_0\right\|^2.
$$

#### (D) Time sampling fix

Avoid over-emphasizing near-pure-noise:

- cap $t$, or warp sampling, or sample in log-SNR.

$$
\mathrm{SNR}(t)=\frac{\alpha(t)^2}{\sigma(t)^2},
\qquad
\lambda(t)=\log \mathrm{SNR}(t).
$$

**Sampling in log-SNR** means:

1. Sample $\lambda \sim \mathcal{U}(\lambda_{\min},\lambda_{\max})$  
2. Convert to $t$ using the inverse mapping $t=\lambda^{-1}$. For example

$$
t=\lambda^{-1}(\lambda)=\frac{1}{2}\log\left(1+e^{-\lambda}\right).
$$

This avoids oversampling extremely noisy times where $\mathrm{SNR}(t)\ll 1$ and the learning problem becomes ambiguous.



#### (E) Preconditioning / parameterization tricks

Often improves stability at high noise:

- predict $\varepsilon$ or $v$ instead of $x_0$, then convert,
- EDM-style preconditioning (scale inputs/outputs by noise level).

#### Quick checklist (debugging symptoms)

- **Mode collapse / mean-seeking:** too much high-noise training, naive MSE on ambiguous states, lack of distill/recon anchor.
- **Constant output:** pure consistency objective without reconstruction/boundary anchors.
- **Multi-step gets worse:** student trained for one-step; sampling off-policy.
- **Training unstable:** not pairing $\left(x_s,x_t\right)$ from same trajectory; missing stop-gradient; time/shape mismatches.

## Summary

Consistency models are difficult because they try to learn a noise-invariant mapping from highly ambiguous noisy inputs, and the naive objective admits degenerate solutions; stable training typically requires **trajectory-correct pairing**, **time sampling control**, **anchors (boundary/recon)**, often **teacher guidance**, and **stop-gradient** to prevent coupled drift.

## Modifying code to include 

1. **sample** $x_T$ from a moment-matched base implemented in **sample_xT_from_running_stats**
2. use **warped time grid** (more mass near t=0)


In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
#  "Black-box" data sampler (replace with your own source)
#   This is the only place where "bimodal" appears.
#   If you truly want zero param knowledge, treat this as a data stream.
# ============================================================
def sample_data(B, device=device):
    # Example bimodal generator (you can replace with real dataset sampling)
    mu = 2.0
    sigma_data = 0.3
    mix = torch.randint(0, 2, (B,1), device=device).float()
    means = (2*mix - 1.0) * mu
    return means + sigma_data * torch.randn(B,1, device=device)

# ============================================================
# VP schedule 
# ============================================================
def alpha(t):
    return torch.exp(-t)

def sigma(t):
    return torch.sqrt(torch.clamp(1.0 - torch.exp(-2.0*t), min=0.0))

def vp_forward(x0, eps, t):
    # x0, eps: (B,1), t: (B,1)
    return alpha(t)*x0 + sigma(t)*eps

# ============================================================
# Running mean/var from observed x0 (no target distribution knowledge)
# ============================================================
class RunningMoments:
    def __init__(self, device):
        self.device = device
        self.n = torch.tensor(0.0, device=device)
        self.mean = torch.tensor(0.0, device=device)
        self.M2 = torch.tensor(0.0, device=device)

    @torch.no_grad()
    def update(self, x):
        v = x.detach().reshape(-1).to(self.device)
        if v.numel() == 0:
            return
        batch_n = torch.tensor(float(v.numel()), device=self.device)
        batch_mean = v.mean()
        batch_M2 = ((v - batch_mean)**2).sum()

        if self.n.item() == 0:
            self.n = batch_n
            self.mean = batch_mean
            self.M2 = batch_M2
        else:
            delta = batch_mean - self.mean
            new_n = self.n + batch_n
            self.mean = self.mean + delta * (batch_n / new_n)
            self.M2 = self.M2 + batch_M2 + delta**2 * (self.n * batch_n / new_n)
            self.n = new_n

    @torch.no_grad()
    def var(self):
        return (self.M2 / torch.clamp(self.n, min=1.0)).clamp(min=1e-12)

@torch.no_grad()
def sample_xT_from_running_stats(stats: RunningMoments, n: int, T: float, device=device):
    """
    Moment-matched VP base:
      mean_T = alpha(T) * mean0
      var_T  = alpha(T)^2 * var0 + sigma(T)^2
    """
    aT = alpha(torch.tensor(T, device=device)).item()
    sT = sigma(torch.tensor(T, device=device)).item()

    # sample $x_T$ from a moment-matched base
    mean0 = stats.mean.item()
    var0  = stats.var().item()

    meanT = aT * mean0
    varT  = (aT**2) * var0 + (sT**2)

    xT = torch.randn(n, 1, device=device) * math.sqrt(varT) + meanT
    return xT, meanT, varT

# ============================================================
# Student model f(x,t) -> x0_hat  (time via sin/cos)
# ============================================================
class SinTime(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.dim = dim
    def forward(self, t):  # t: (B,1)
        t = t[:,0]
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(0, half, device=t.device).float() / (half - 1))
        args = t[:,None] * freqs[None,:]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0,1))
        return emb

class Student1D(nn.Module):
    def __init__(self, tdim=64, hidden=256):
        super().__init__()
        self.temb = nn.Sequential(SinTime(tdim), nn.Linear(tdim, tdim), nn.SiLU(), nn.Linear(tdim, tdim))
        self.net  = nn.Sequential(
            nn.Linear(1 + tdim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 1)
        )
    def forward(self, x, t):  # x: (B,1), t: (B,1)
        e = self.temb(t)
        return self.net(torch.cat([x, e], dim=1))

# ============================================================
#    - no normalization
#    - uses warped time grid (more mass near t=0)
#    - base sampling uses running stats (mean/var)
# ============================================================
def train_student_no_teacher_stats(
    steps=20000,
    batch=4096,
    T=1.0,         # keep modest for one-step stability with alpha(t)=exp(-t)
    K=32,
    gamma_grid=2.0,
    lr=2e-4,
    lam_recon=1.0,   # uses true x0 (available from data stream)
    lam_cons=0.5,
    lam_bdry=1.0,
    print_every=1000,
):
    student = Student1D().to(device)
    opt = torch.optim.AdamW(student.parameters(), lr=lr, weight_decay=1e-4)

    stats = RunningMoments(device=device)

    u = torch.linspace(0.0, 1.0, K+1, device=device)
    ts = (u**gamma_grid) * T  # denser near 0

    hist = []
    t0 = time.time()
    for it in range(1, steps+1):
        x0 = sample_data(batch, device=device)     # (B,1)
        stats.update(x0)                           # <-- key: learn mean/var from observed data
        eps = torch.randn_like(x0)

        # adjacent grid times per sample: s < t
        k = torch.randint(1, K+1, (batch,), device=device)
        
        # The function of view is to reshape the tensor to have a specific shape without changing its data. 
        # In this case, ts[k] is a 1D tensor of shape (B,), and we want to reshape it to (B,1) 
        # so that it can be broadcasted correctly when passed to the student model.
        t = ts[k].view(batch,1)
        s = ts[k-1].view(batch,1)

        xt = vp_forward(x0, eps, t)
        xs = vp_forward(x0, eps, s)               # same eps => same forward path

        x0_t = student(xt, t)
        x0_s = student(xs, s)

        loss_recon = F.mse_loss(x0_t, x0)                # stabilizes modes without teacher
        # we detach because we only want gradients to update student via x0_s, not x0_t (one-way consistency)
        loss_cons  = F.mse_loss(x0_s, x0_t.detach())     # one-way consistency
        loss_bdry  = F.mse_loss(student(x0, torch.zeros_like(t)), x0)

        loss = lam_recon*loss_recon + lam_cons*loss_cons + lam_bdry*loss_bdry

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        opt.step()

        hist.append(float(loss.item()))
        if it % print_every == 0:
            dt = time.time() - t0
            print(f"[train] {it:6d}/{steps}  loss={np.mean(hist[-print_every:]):.4f}  "
                  f"mean0={stats.mean.item():.4f}  var0={stats.var().item():.4f}  {dt:.1f}s")

    return student, stats, np.array(hist, dtype=np.float32)

@torch.no_grad()
def sample_student_one_step_from_stats(student, stats, n=200000, T=1.0):
    student.eval()
    xT, meanT, varT = sample_xT_from_running_stats(stats, n=n, T=T, device=device)
    tT = torch.full((n,1), T, device=device)
    x0_hat = student(xT, tT)
    return x0_hat.squeeze(1).cpu().numpy(), (meanT, varT)

# ============================================================
#    Evaluation without knowing analytic target:
#    KL between histograms of *real samples* vs *generated samples*.
#    (only needs sampling access to data stream)
# ============================================================
def kl_hist_samples(x_real, x_gen, bins=400):
    lo = float(min(x_real.min(), x_gen.min()))
    hi = float(max(x_real.max(), x_gen.max()))
    pad = 0.05*(hi-lo + 1e-6)
    lo -= pad; hi += pad

    edges = np.linspace(lo, hi, bins+1)
    dx = edges[1]-edges[0]

    p, _ = np.histogram(x_real, bins=edges, density=True)
    q, _ = np.histogram(x_gen,  bins=edges, density=True)

    eps = 1e-12
    p = np.clip(p, eps, None)
    q = np.clip(q, eps, None)

    return float(np.sum(p * (np.log(p) - np.log(q)) * dx))

# ============================================================
# Run
# ============================================================
T = 1.0
student, stats, loss_hist = train_student_no_teacher_stats(
    steps=20000, batch=4096, T=T, K=32, gamma_grid=2.0,
    lam_recon=1.0, lam_cons=0.5, lam_bdry=1.0, print_every=1000
)

x_gen, (meanT, varT) = sample_student_one_step_from_stats(student, stats, n=200000, T=T)
x_real = sample_data(200000, device=device).squeeze(1).cpu().numpy()

kl = kl_hist_samples(x_real, x_gen)
print(f"Base moments at T={T}: mean_T={meanT:.4f}, var_T={varT:.4f}")
print("Approx KL( real || gen ) via hist =", kl)

plt.figure(figsize=(8,3))
plt.hist(x_real, bins=200, density=True, alpha=0.4, label="real (samples)")
plt.hist(x_gen,  bins=200, density=True, alpha=0.4, label="student (1-step)")
plt.title(f"1D: no-teacher consistency + VP base from running stats | KL≈{kl:.4f}")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()

plt.figure(figsize=(8,3))
plt.plot(loss_hist)
plt.title("Training loss")
plt.grid(True, alpha=0.2)
plt.show()

## Teacher–Student Consistency Models

### Why do we need a teacher in the first place?

A *pure* consistency objective tries to make a model output invariant across time:
$$
f_\theta(x_s,s) \approx f_\theta(x_t,t)\qquad (s<t)
$$
for noisy states $x_s, x_t$ that come from the same underlying clean sample $x_0$.

The problem is that this objective is **not identifiable**: the trivial solution
$
f_\theta(x,t) \equiv c
$
for any constant $c$ makes the loss small, because both sides are always equal.  
In multimodal data, minimizing a squared-error consistency loss often drives the network toward an “average” output, causing **mode collapse**.

A **teacher** is introduced to provide a *non-trivial target* that preserves the data structure (e.g., multimodality) at noisy times. In short:

- **Consistency alone** says: “outputs should match across time.”
- **Teacher guidance** says: “and the shared output should be meaningful (close to the data manifold).”

The teacher makes the learning problem **well-posed** by anchoring what the correct canonical prediction should be when the input is noisy.

### Setup: forward corruption (diffusion/noising)

Let $x_0 \sim p_{\text{data}}$ be a clean sample.

A forward process defines noisy versions $x_t$:
- **Continuous (Gaussian) form** (common in diffusion):
$$
x_t = \alpha(t)\,x_0 + \sigma(t)\,\varepsilon,
\qquad \varepsilon \sim \mathcal{N}(0,I),
\qquad \alpha(t)^2 + \sigma(t)^2 = 1.
$$

### Teacher model (diffusion teacher)

A **teacher** is a strong generative model (often a diffusion model) trained to approximate a denoising posterior.

Typical teacher outputs:

- **Teacher posterior over clean data**:
$
p_\phi(x_0 \mid x_t, t)
$

or (equivalently) a parameterization such as predicting noise:
$
\varepsilon_\phi(x_t,t) \approx \varepsilon
$
from which a posterior / denoising estimate can be constructed.

Intuition: the teacher knows how to “move toward the data manifold” from a noisy point.

### Student consistency model

The **student** is a faster model (often fewer steps / smaller network) trained so that for any noisy input $x_t$ it outputs a canonical prediction of the underlying clean sample:

- In continuous form:
$
f_\theta(x_t,t) \approx x_0
$

The student is trained using **two ideas**:

1) **Distill** teacher knowledge at noisy times (anchor).
2) Enforce **consistency** across different times.

### Core idea: a teacher-defined "step" from time $t$ to $s<t$

To enforce time-consistency, we want a pair $x_t$ (noisier) and $x_s$ (less noisy) that correspond to the same underlying clean sample $x_0$.

Assume a **continuous diffusion** (e.g., DDPM) where:

- $x_0 \in \mathbb{R}^{d}$ is the clean data
- $x_t \in \mathbb{R}^{d}$ is the noisy sample at time $t$
- $0 \le s < t \le 1$ (or any monotone time parameterization)

#### Forward (noising) process: Gaussian transitions

A common continuous-time / variance-preserving parameterization yields:

$
q(x_t \mid x_0, t) = \mathcal{N}\!\left(x_t;\; \alpha(t)\,x_0,\; \sigma^2(t)\,I\right),
$

where $\alpha(t)$ decreases with $t$ and $\sigma^2(t)$ increases with $t$.

Similarly at time $s$:

$
q(x_s \mid x_0, s) = \mathcal{N}\!\left(x_s;\; \alpha(s)\,x_0,\; \sigma^2(s)\,I\right).
$

#### Teacher posterior over the clean sample

A teacher model (often a trained diffusion model) provides an approximation to the posterior:

$
p_\phi(x_0 \mid x_t, t).
$

In practice, this teacher posterior is typically represented implicitly (e.g., via a predicted noise, score, or predicted $x_0$), but conceptually it is a distribution over possible clean samples consistent with $x_t$.

#### Teacher “denoise step” from $t$ to $s$ by marginalizing over $x_0$

We define a principled distribution for a less-noisy state $x_s$ conditioned on $x_t$ by integrating out $x_0$:

$$
p(x_s \mid x_t, t, s)
=
\int q(x_s \mid x_0, s)\, p_\phi(x_0 \mid x_t, t)\, dx_0.
$$

This is the continuous analogue of the discrete marginalization:
- draw a plausible clean sample $x_0$ from the teacher posterior
- re-noise it to the earlier time $s$ using the known forward kernel $q(x_s\mid x_0,s)$

#### A practical sampling view (how you actually construct $x_s$)

The teacher defines a principled step $t \rightarrow s$ in the continuous case by **sampling or integrating over** $x_0$:

$
p(x_s \mid x_t,t,s) = \int q(x_s \mid x_0,s)\,p_\phi(x_0 \mid x_t,t)\,dx_0.
$

Here:
- $q(x_s\mid x_0,s)$ is known and Gaussian (the forward noising kernel)
- $p_\phi(x_0\mid x_t,t)$ is provided by the teacher model

#### We can use Monte Carlo (sampling) approximation in most cases to integrate

Use the fact that an integral of the form
$$
\int f(x_0)\,p(x_0)\,dx_0
$$
is just an expectation:
$$
\mathbb{E}_{x_0\sim p(x_0)}[f(x_0)].
$$

So:
$
p(x_s \mid x_t,t,s) = \mathbb{E}_{x_0\sim p_\phi(\cdot\mid x_t,t)}\big[q(x_s\mid x_0,s)\big].
$

The following can be used to sample $x_s$:
1. Sample a clean candidate from the teacher:
   $
   x_0^{(m)} \sim p_\phi(x_0\mid x_t,t).
   $
2. Sample $x_s$ from the known forward kernel:
   $
   x_s^{(m)} \sim q(x_s\mid x_0^{(m)},s).
   $

Doing this once gives one draw from the mixture distribution. Doing it $M$ times gives a Monte Carlo approximation.

#### Monte Carlo approximation with $M$ samples
The mixture density can be approximated as:
$
p(x_s \mid x_t,t,s) \approx \frac{1}{M}\sum_{m=1}^M q(x_s \mid x_0^{(m)}, s).
$

**Interpretation:** you are representing $p(x_s\mid x_t)$ as a mixture of Gaussians, one Gaussian per sampled $x_0^{(m)}$.

> In training (consistency / distillation), people usually take \(M=1\) because it’s cheap and works well.

### Losses used to train the student

#### Distillation loss (anchor to teacher)

Make the student match teacher denoising belief at the same noisy point:

If teacher provides a clean estimate $\hat{x}_0^\phi(x_t,t)$,
$$
\mathcal{L}_{\text{distill}}
=
\mathbb{E}\left[
\big\| f_\theta(x_t,t) - \hat{x}_0^\phi(x_t,t) \big\|_2^2
\right].
$$

This term prevents collapse because it forces outputs to track the teacher’s *non-trivial* multimodal structure.

#### Consistency loss (invariance across time)

Enforce that the student’s canonical prediction is stable when moving from $t$ to $s<t$:

$$
\mathcal{L}_{\text{cons}}
=
\mathbb{E}\left[
\big\| f_\theta(x_t,t) - f_\theta(\tilde{x}_s,s) \big\|_2^2
\right].
$$

#### Optional boundary anchor at $t=0$

To ensure the student is correct on clean data:
$$
\mathcal{L}_{\text{bdry}}
=
\mathbb{E}\left[
\| f_\theta(x_0,0) - x_0 \|_2^2
\right]
\quad\text{or}\quad
\mathcal{L}_{\text{bdry}}=\mathbb{E}\left[-\log p_\theta(x_0\mid x_0,0)\right].
$$

### Full objective

A typical combined objective is:
$$
\mathcal{L}(\theta)
=
\mathcal{L}_{\text{distill}}
+
\lambda_{\text{cons}}\,\mathcal{L}_{\text{cons}}
+
\lambda_{\text{bdry}}\,\mathcal{L}_{\text{bdry}}.
$$

### Step-by-step training algorithm (teacher–student consistency)

Assume a time range $t \in [0,T]$ and choose $s<t$.

**Repeat (SGD steps):**

1. Sample clean data:
$
x_0 \sim p_{\text{data}}.
$

2. Sample time(s):
$
t \sim \mathrm{Uniform}(0,T),
\qquad
s \sim \mathrm{Uniform}(0,t).
$

3. Forward-noise to get $x_t$:
$
x_t=\alpha(t)x_0+\sigma(t)\varepsilon.
$

4. Teacher computes its denoising belief:
$
p_\phi(x_0\mid x_t,t)\quad \text{or}\quad \hat{x}_0^\phi(x_t,t).
$

5. Teacher constructs a less-noisy state at $s$:
$
\tilde{x}_s = \mathrm{TeacherStep}(x_t,t\to s).
$

6. Student predictions:
$f_\theta(x_t,t), f_\theta(\tilde{x}_s,s)$.

7. Compute losses:
$
\mathcal{L}_{\text{distill}} \;+\; \lambda_{\text{cons}}\mathcal{L}_{\text{cons}} \;+\; \lambda_{\text{bdry}}\mathcal{L}_{\text{bdry}}.
$

8. Update student parameters:
$
\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}(\theta).
$

## Sampling (fast generation)

After training, sampling is fast because the student is trained to map noisy states to clean predictions directly.

- **One-step**:
$$
x_T \sim p_{\text{noise}},
\qquad
\hat{x}_0 \sim p_\theta(x_0\mid x_T,T)
\quad\text{or}\quad
\hat{x}_0=f_\theta(x_T,T).
$$

- **Few-step** (optional): evaluate the student at intermediate times before $T$ for better quality, then distill down to 1-step if desired.

## Summary

- The teacher is required because **consistency constraints alone admit trivial constant solutions** (collapse).
- The teacher provides a **non-trivial denoising target** at noisy times.
- The student learns both:
  - to **match the teacher** (distillation), and
  - to be **time-consistent** (invariance across times), enabling **very fast sampling**.

## Teacher-guided Consistency (1D bimodal)

This note explains the math behind the following pipeline:

- **Data**: 1D bimodal Gaussian mixture (unknown to the models; they just see samples)
- **Teacher**: DDPM-trained **noise predictor** $\varepsilon_\phi(x_t,t)$
- **Teacher sampling**: deterministic **DDIM** transport from $x_T$ to $x_0$ (many steps)
- **Student**: predictor $f_\theta(x_t,t)\approx x_0$
- **Student training**: distill teacher’s final output (and optionally add consistency)
- **Student sampling**: one step $x_T\mapsto \hat{x}_0$

The key conceptual point: the **teacher is a multi-step transport sampler**, not a posterior-mean estimator. The student learns to approximate the teacher’s *final* sample given states along that teacher trajectory.

### Data distribution (bimodal mixture)

The dataset is a 1D mixture:

$$
p_{\text{data}}(x_0)
=
p\,\mathcal{N}(-\mu,\sigma_{\text{data}}^2)
+
(1-p)\,\mathcal{N}(+\mu,\sigma_{\text{data}}^2).
$$

In code, you sample $x_0\sim p_{\text{data}}$ using `sample_gmm(...)`.

### Discrete DDPM forward process (the diffusion model)

We define a discrete time grid with $T$ steps ( code uses `T_steps`).

A standard DDPM schedule defines:

$$
\beta_t \in (0,1),
\qquad
\alpha_t = 1-\beta_t,
\qquad
\bar{\alpha}_t = \prod_{i=0}^{t}\alpha_i.
$$

The closed-form forward marginal is:

$$
q(x_t\mid x_0)
=
\mathcal{N}\!\left(
\sqrt{\bar{\alpha}_t}\,x_0,\ (1-\bar{\alpha}_t)\mathbf{I}
\right).
$$

Equivalently, we can sample:

$$
x_t = \sqrt{\bar{\alpha}_t}\,x_0 + \sqrt{1-\bar{\alpha}_t}\,\varepsilon,
\qquad
\varepsilon\sim\mathcal{N}(0,\mathbf{I}).
$$

In the 1D code (scalar case), this becomes:

$$
x_t = \sqrt{\bar{\alpha}_t}\,x_0 + \sqrt{1-\bar{\alpha}_t}\,\varepsilon,
\qquad
\varepsilon\sim\mathcal{N}(0,1).
$$

This is exactly what `q_sample(x0, t_idx)` implements.

### TEACHER MODEL

#### Teacher parameterization: noise prediction

The teacher is trained as a noise predictor:

$
\varepsilon_\phi(x_t,t)\approx \varepsilon.
$

#### Teacher training objective (DDPM-style)

Given $$x_0\sim p_{\text{data}}$$, sample $$t\sim \{0,\dots,T-1\}$$ and form:

$$
x_t=\sqrt{\bar{\alpha}_t}\,x_0 + \sqrt{1-\bar{\alpha}_t}\,\varepsilon,
\qquad
\varepsilon\sim\mathcal{N}(0,1).
$$

Then minimize:

$$
\mathcal{L}_{\text{teacher}}(\phi)
=
\mathbb{E}_{x_0,t,\varepsilon}
\left[
\left\|
\varepsilon_\phi(x_t,t)-\varepsilon
\right\|_2^2
\right].
$$

This corresponds directly to:

- `pred_eps = teacher(xt, t01)`
- `loss = mse(pred_eps, eps)`

where $t_{01}=t/(T-1)\in[0,1]$ is a normalized time input.

#### Teacher reconstruction: $\hat{x}_0$ from $x_t$

From the forward equation:

$$
x_t = \sqrt{\bar{\alpha}_t}\,x_0 + \sqrt{1-\bar{\alpha}_t}\,\varepsilon,
$$

solve for $x_0$:

$$
x_0
=
\frac{x_t-\sqrt{1-\bar{\alpha}_t}\,\varepsilon}{\sqrt{\bar{\alpha}_t}}.
$$

The teacher estimates $$\varepsilon$$ with $$\varepsilon_\phi(x_t,t)$$, giving:

$$
\hat{x}_0(x_t,t)
=
\frac{x_t-\sqrt{1-\bar{\alpha}_t}\,\varepsilon_\phi(x_t,t)}{\sqrt{\bar{\alpha}_t}}.
$$

In code:
- $\sqrt{\bar{\alpha}_t} \leftrightarrow \texttt{sqrt\_ab[t]}$
- $\sqrt{1-\bar{\alpha}_t} \leftrightarrow \texttt{sqrt\_1mab[t]}$

and:

$$
\hat{x}_0
=
\frac{x_t - \texttt{sqrt\_1mab[t]}\cdot \varepsilon_\phi(x_t,t)}
{\texttt{sqrt\_ab[t]}}.
$$

The implementation includes a clamp for numerical safety when $\sqrt{\bar{\alpha}_t}$ is small.

#### Teacher sampling: deterministic DDIM trajectory (transport)

The teacher is used as a multi-step sampler to map base noise $x_T$ to a final sample near the data distribution.

The code uses a deterministic DDIM-style update:

1. predict noise:
$
\hat{\varepsilon}_t = \varepsilon_\phi(x_t,t)
$

2. reconstruct:
$$
\hat{x}_0
=
\frac{x_t-\sqrt{1-\bar{\alpha}_t}\,\hat{\varepsilon}_t}{\sqrt{\bar{\alpha}_t}}
$$

3. deterministically map to the previous time:
$$
x_{t-1}
=
\sqrt{\bar{\alpha}_{t-1}}\,\hat{x}_0
+
\sqrt{1-\bar{\alpha}_{t-1}}\,\hat{\varepsilon}_t.
$$

This differs from DDPM sampling because it does **not** inject fresh Gaussian noise at each step; it reuses $\hat{\varepsilon}_t$ in a deterministic way (DDIM transport).

The function `teacher_ddim_trajectory(xT)` produces:
- the whole trajectory $\{x_t\}_{t=0}^{T-1}$ (stored as `x_hist`)
- the final sample $x_0^{(\text{teacher})}$ (returned as `x0_teacher`)

We can view the teacher sampler as a transport map:

$$
x_0^{(\text{teacher})} = g_\phi(x_T),
\qquad x_T\sim\mathcal{N}(0,1).
$$

### STUDENT MODEL

#### Student parameterization: predict $x_0$ directly

The student is trained to output an estimate of the final clean sample:

$
f_\theta(x_t,t)\approx x_0.
$

But in this teacher-guided setup, the target is **not** the true data $$x_0$$; instead, the target is the teacher’s final sample from the same initial noise:

$
x_0^{(\text{teacher})}=g_\phi(x_T).
$

#### Distillation: student learns the teacher’s final output from intermediate states

In each student training iteration:

1. Sample base noise:

$
x_T \sim \mathcal{N}(0,1).
$

2. Run the teacher sampler to get a trajectory and final sample:

$
\{x_t\}_{t=0}^{T-1},\quad x_0^{(\text{teacher})}.
$

3. Pick a random time index $t$ and take $x_t$ from the trajectory.

4. Train the student so that:

$
f_\theta(x_t,t)\approx x_0^{(\text{teacher})}.
$

The distillation loss is:

$$
\mathcal{L}_{\text{distill}}(\theta)
=
\mathbb{E}_{x_T,t}
\left[
\left\|
f_\theta(x_t,t)-x_0^{(\text{teacher})}
\right\|_2^2
\right].
$$

This is exactly:

- `pred_t = student(xt, t01)`
- `L_distill = mse(pred_t, x0_teacher)`

### One-way consistency regularization (stop-gradient)

To encourage invariance across times along the same teacher trajectory, the code also samples an earlier time $s<t$ and enforces:

$
f_\theta(x_s,s)\approx f_\theta(x_t,t).
$

To prevent both sides from drifting together, the code uses stop-gradient on the $t$ side:

$$
\mathcal{L}_{\text{cons}}(\theta)
=
\mathbb{E}_{x_T,s<t}
\left[
\left\|
f_\theta(x_s,s)-\mathrm{sg}\!\left(f_\theta(x_t,t)\right)
\right\|_2^2
\right].
$$

Total student objective:

$$
\mathcal{L}_{\text{student}}(\theta)
=
\mathcal{L}_{\text{distill}}(\theta)
+
\lambda_{\text{cons}}\,\mathcal{L}_{\text{cons}}(\theta).
$$

In code:
- `L_cons = mse(pred_s, pred_t.detach())`
- `loss = L_distill + lam_cons * L_cons`

### SAMPLING WITH THE STUDENT

#### One-step student sampling

After training, the student is used as a one-step generator:

1. Draw noise:

$
x_T \sim \mathcal{N}(0,1).
$

2. Call the student once at the maximum time:

$
\hat{x}_0 = f_\theta(x_T, t=1).
$

In the code:
- `xT = torch.randn(n,1)`
- `t01 = 1.0`
- `x0_hat = student(xT, t01)`

This works because the student was trained to map intermediate/noisy teacher states to the teacher’s final sample, so applying it at the noisiest endpoint yields a direct generator.

#### Why this can work (the transport view)

- The teacher implements a **multi-step transport map** $$g_\phi:\ x_T\mapsto x_0^{(\text{teacher})}$$.
- The student learns a family of maps $$f_\theta(\cdot,t)$$ that predict the same endpoint for any state along the teacher’s path.

Informally, for teacher-generated trajectories:

$
f_\theta(x_t,t)\approx g_\phi(x_T)\quad \text{for many }t.
$

So the student approximates the teacher’s transport, but collapses it into **one evaluation**.

#### Common pitfalls

1. **Teacher quality matters**: if $\varepsilon_\phi$ is poorly trained, $g_\phi(x_T)$ is a bad transport and the student learns a bad target.

2. **Multi-step vs one-step mismatch**: the student here is trained for *one-step distillation*. If you try to run the student as a multi-step chain, you go off-policy and errors can accumulate.

3. **High-noise instability**: when $\sqrt{\bar{\alpha}_t}$ is tiny, the reconstruction
$$
\hat{x}_0 = \frac{x_t-\sqrt{1-\bar{\alpha}_t}\,\hat{\varepsilon}}{\sqrt{\bar{\alpha}_t}}
$$
can blow up; clamp or avoid extreme $t$ early in training.

4. **Consistency needs stop-gradient**: without $\mathrm{sg}(\cdot)$, both sides can drift toward a trivial fixed point.

In [ ]:
# Teacher-guided Consistency : diffusion teacher DDIM sampler -> student distilled to 1-step
# ---------------------------------------------------------------------------------------
# Why this works: teacher is a multi-step sampler (transport), not a posterior-mean estimator.
# Student learns to approximate teacher's final sample given the same initial noise.
#
# Data: 1D bimodal Gaussian mixture
# Teacher: epsilon-predictor diffusion model (DDPM training)
# Teacher sampling: deterministic DDIM sampler (many steps)
# Student: f_theta(x_t, t) -> x0 (distilled from teacher trajectories)
# Student sampling: 1-step from x_T
# ---------------------------------------------------------------------------------------

import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0); np.random.seed(0)

# -----------------------------
# Data: bimodal GMM in 1D
# -----------------------------
@torch.no_grad()
def sample_gmm(n, mu=2.0, sigma_data=0.5, p=0.5, device=device):
    u = torch.rand(n, 1, device=device)
    left = (u < p).float()
    means = left * (-mu) + (1.0 - left) * (+mu)
    x0 = means + sigma_data * torch.randn(n, 1, device=device)
    return x0

# -----------------------------
# Discrete diffusion schedule (DDPM-style)
# -----------------------------
def make_beta_schedule(T=32, beta_start=1e-4, beta_end=2e-2, device=device):
    betas = torch.linspace(beta_start, beta_end, T, device=device)  # (T,)
    alphas = 1.0 - betas
    alpha_bar = torch.cumprod(alphas, dim=0)                        # (T,)
    return betas, alphas, alpha_bar

T_steps = 64
betas, alphas, alpha_bar = make_beta_schedule(T=T_steps, beta_start=1e-4, beta_end=2e-2, device=device)

# convenience tensors
sqrt_ab   = torch.sqrt(alpha_bar)                 # sqrt(alpha_bar_t)
sqrt_1mab = torch.sqrt(1.0 - alpha_bar)           # sqrt(1-alpha_bar_t)

@torch.no_grad()
def q_sample(x0, t_idx):
    """
    x_t = sqrt(ab_t) x0 + sqrt(1-ab_t) eps
    t_idx: (B,) integer in [0..T-1]
    """
    B = x0.shape[0]
    eps = torch.randn_like(x0)
    a = sqrt_ab[t_idx].view(B,1)
    b = sqrt_1mab[t_idx].view(B,1)
    xt = a * x0 + b * eps
    return xt, eps

# -----------------------------
# Time embedding + small MLP
# -----------------------------
def sinusoidal_time_embedding(t, dim=64):
    # t: (B,1) in [0,1]
    half = dim // 2
    freqs = torch.exp(-math.log(10000.0) * torch.arange(0, half, device=t.device).float() / max(half-1, 1))
    args = t * freqs.view(1,-1)
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
    if dim % 2 == 1:
        emb = F.pad(emb, (0,1))
    return emb

class MLP(nn.Module):
    def __init__(self, time_dim=64, hidden=256, out_dim=1):
        super().__init__()
        self.time_dim = time_dim
        self.net = nn.Sequential(
            nn.Linear(1 + time_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x, t01):
        # x: (B,1), t01: (B,1) in [0,1]
        te = sinusoidal_time_embedding(t01, self.time_dim)
        inp = torch.cat([x, te], dim=1)
        return self.net(inp)

# Teacher predicts epsilon; Student predicts x0
teacher = MLP(time_dim=64, hidden=256, out_dim=1).to(device)
student = MLP(time_dim=64, hidden=256, out_dim=1).to(device)

# -----------------------------
# Train TEACHER: eps_theta(x_t,t) ~ eps  (standard diffusion training)
# -----------------------------
def train_teacher(steps=20000, batch=2048, lr=2e-4, print_every=2000):
    opt = torch.optim.AdamW(teacher.parameters(), lr=lr)
    teacher.train()
    hist = []
    t0 = time.time()
    for it in range(1, steps+1):
        x0 = sample_gmm(batch, device=device)
        t_idx = torch.randint(0, T_steps, (batch,), device=device)
        xt, eps = q_sample(x0, t_idx)
        t01 = (t_idx.float() / (T_steps-1)).view(batch,1)

        pred_eps = teacher(xt, t01)
        loss = F.mse_loss(pred_eps, eps)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(teacher.parameters(), 1.0)
        opt.step()

        hist.append(loss.item())
        if it % print_every == 0:
            print(f"[teacher] step {it:5d}/{steps}  mse={np.mean(hist[-print_every:]):.6f}  min={(time.time()-t0)/60:.1f}")
    return np.array(hist, dtype=np.float32)

# -----------------------------
# Teacher DDIM sampler (deterministic, multi-step)
#    x_{t-1} = sqrt(ab_{t-1}) x0_hat + sqrt(1-ab_{t-1}) eps_hat
# -----------------------------
@torch.no_grad()
def teacher_ddim_trajectory(xT):
    """
    Deterministic DDIM trajectory from t=T-1 down to 0.
    Returns:
      x_hist: Tensor [T_steps, B, 1]  (x_hist[t] = x_t)
      x0_final: Tensor [B,1]
    """
    teacher.eval()
    B = xT.shape[0]
    x = xT

    x_hist = torch.empty(T_steps, B, 1, device=xT.device)
    x_hist[T_steps - 1] = x

    for t in reversed(range(1, T_steps)):
        # model time in [0,1]
        t01 = torch.full((B, 1), t / (T_steps - 1), device=xT.device)

        eps_hat = teacher(x, t01)  # (B,1)

        # schedule scalars expanded to (B,1)
        a  = sqrt_ab[t].reshape(1, 1).expand(B, 1)
        b  = sqrt_1mab[t].reshape(1, 1).expand(B, 1)
        a0 = torch.clamp(a, min=1e-6)

        # x0_hat from current x_t
        x0_hat = (x - b * eps_hat) / a0

        # step to t-1 (DDIM deterministic)
        a_prev = sqrt_ab[t - 1].reshape(1, 1).expand(B, 1)
        b_prev = sqrt_1mab[t - 1].reshape(1, 1).expand(B, 1)
        x = a_prev * x0_hat + b_prev * eps_hat

        x_hist[t - 1] = x

    return x_hist, x  # x is x0_final

# -----------------------------
# Train STUDENT by distilling teacher trajectories
#    Student learns f(x_t,t) -> x0_teacher_final, plus optional consistency
# -----------------------------
def train_student(steps=30000, batch=2048, lr=2e-4, lam_cons=0.5, print_every=2000):
    opt = torch.optim.AdamW(student.parameters(), lr=lr)
    student.train()
    hist = []
    t0 = time.time()

    for it in range(1, steps + 1):
        xT = torch.randn(batch, 1, device=device)

        with torch.no_grad():
            x_hist, x0_teacher = teacher_ddim_trajectory(xT)  # x_hist: [T,B,1], x0_teacher: [B,1]

        B = batch
        idx = torch.arange(B, device=device)

        # pick random times per sample
        t_idx = torch.randint(0, T_steps, (B,), device=device)
        xt = x_hist[t_idx, idx]                  # (B,1)
        t01 = (t_idx.float() / (T_steps - 1)).view(B, 1)

        # distill target: teacher's final sample from same xT
        pred_t = student(xt, t01)
        L_distill = F.mse_loss(pred_t, x0_teacher)

        # optional consistency: pick an earlier time s < t (clamped)
        step_back = torch.randint(1, 8, (B,), device=device)
        s_idx = torch.clamp(t_idx - step_back, 0, T_steps - 1)
        xs = x_hist[s_idx, idx]                  # (B,1)
        s01 = (s_idx.float() / (T_steps - 1)).view(B, 1)

        pred_s = student(xs, s01)
        L_cons = F.mse_loss(pred_s, pred_t.detach())  # stopgrad on t-side

        loss = L_distill + lam_cons * L_cons

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        opt.step()

        hist.append(loss.item())
        if it % print_every == 0:
            print(
                f"[student] step {it:5d}/{steps}  L={np.mean(hist[-print_every:]):.6f}  "
                f"distill={L_distill.item():.6f}  cons={L_cons.item():.6f}  min={(time.time()-t0)/60:.1f}"
            )

    return np.array(hist, dtype=np.float32)


# -----------------------------
# Train + plot
# -----------------------------
teacher_hist = train_teacher(steps=20000, batch=2048, lr=2e-4, print_every=2000)
student_hist = train_student(steps=30000, batch=2048, lr=2e-4, lam_cons=0.5, print_every=2000)

plt.figure(figsize=(8,3))
plt.plot(teacher_hist); plt.title("Teacher loss (epsilon MSE)"); plt.grid(True, alpha=0.2); plt.show()

plt.figure(figsize=(8,3))
plt.plot(student_hist); plt.title("Student loss (distill + consistency)"); plt.grid(True, alpha=0.2); plt.show()

# -----------------------------
# 8) Student 1-step sampling: x_T -> x0
# -----------------------------
@torch.no_grad()
def sample_student_one_step(n=60000):
    student.eval()
    xT = torch.randn(n, 1, device=device)
    t01 = torch.full((n,1), 1.0, device=device)  # t = T_steps-1 normalized to 1
    x0_hat = student(xT, t01)
    return x0_hat

# Evaluate: histogram true vs student
@torch.no_grad()
def eval_hist(mu=2.0, sigma_data=0.5, n=60000):
    x_true = sample_gmm(n, mu=mu, sigma_data=sigma_data, device=device).cpu().numpy().flatten()
    x_gen = sample_student_one_step(n=n).cpu().numpy().flatten()

    plt.figure(figsize=(8,4))
    plt.hist(x_true, bins=120, density=True, alpha=0.6, label="true data")
    plt.hist(x_gen,  bins=120, density=True, alpha=0.6, label="student 1-step")
    plt.title("Bimodal distribution: true vs student-generated (teacher-guided distillation)")
    plt.xlabel("x"); plt.ylabel("density"); plt.legend()
    plt.show()

eval_hist(mu=2.0, sigma_data=0.5, n=60000)

## Trying multistep sampling on the student

In [ ]:
# ------------------------------------------------------------
# K-step sampling from a distilled student + KL vs steps plot
# ------------------------------------------------------------
import numpy as np
import torch
import matplotlib.pyplot as plt

# -----------------------------
# Helpers: histogram KL
# -----------------------------
def hist_prob(x_np, bins):
    """Return histogram-based probability mass (sums to 1)."""
    h, _ = np.histogram(x_np, bins=bins, density=False)
    h = h.astype(np.float64)
    h = h / (h.sum() + 1e-12)
    return h

def kl_divergence(p, q, eps=1e-12):
    """KL(p||q) for discrete distributions (arrays summing to 1)."""
    p = np.clip(p, eps, 1.0)
    q = np.clip(q, eps, 1.0)
    return float(np.sum(p * (np.log(p) - np.log(q))))

# -----------------------------
# Student-based deterministic step (DDIM-style but using student x0_hat)
# -----------------------------
@torch.no_grad()
def student_step_xt_to_xs(xt, t_idx, s_idx):
    """
    xt:   (B,1) at discrete time index t_idx (int)
    s_idx: target time index < t_idx (int)
    returns xs: (B,1)
    """
    B = xt.shape[0]

    # normalized time in [0,1] for the student network
    t01 = torch.full((B,1), t_idx / (T_steps - 1), device=xt.device)

    # student predicts x0_hat at time t
    x0_hat = student(xt, t01)

    # infer eps_hat from (xt, x0_hat)
    a_t = sqrt_ab[t_idx].reshape(1,1).expand(B,1)
    b_t = sqrt_1mab[t_idx].reshape(1,1).expand(B,1)
    eps_hat = (xt - a_t * x0_hat) / torch.clamp(b_t, min=1e-6)

    # build xs at time s using same eps_hat (deterministic)
    a_s = sqrt_ab[s_idx].reshape(1,1).expand(B,1)
    b_s = sqrt_1mab[s_idx].reshape(1,1).expand(B,1)
    xs = a_s * x0_hat + b_s * eps_hat
    return xs

# -----------------------------
# K-step sampler (K transitions from t=T-1 down to 0)
# -----------------------------
@torch.no_grad()
def sample_student_k_steps(n=60000, K=1):
    """
    Deterministic K-step sampling using the student.
    K=1 means one-step: x0_hat = student(x_T, T).
    K>1 means march down K steps with evenly spaced indices.
    """
    student.eval()
    B = n

    # Start at x_T ~ N(0,1) (note: in this discrete DDPM schedule, x_{T-1} ~ approx N)
    xt = torch.randn(B, 1, device=device)

    # Use max index as "T-1"
    t_max = T_steps - 1

    if K <= 1:
        t01 = torch.ones(B, 1, device=device)  # corresponds to t_idx = T_steps-1
        x0_hat = student(xt, t01)
        return x0_hat

    # Limit K so indices are strictly decreasing
    K = min(K, t_max)

    # Choose K+1 indices from t_max down to 0
    idxs = torch.linspace(t_max, 0, K+1).round().long().cpu().numpy().tolist()
    # Ensure strictly decreasing and end at 0
    idxs[0] = t_max
    idxs[-1] = 0
    # remove duplicates while preserving order
    idxs_strict = []
    for a in idxs:
        if len(idxs_strict) == 0 or a != idxs_strict[-1]:
            idxs_strict.append(a)
    # enforce monotone decreasing
    idxs_strict = [idxs_strict[0]] + [b for b in idxs_strict[1:] if b < idxs_strict[idxs_strict.index(b)-1]]
    if idxs_strict[-1] != 0:
        idxs_strict.append(0)

    # March from each t_idx to next s_idx
    for t_idx, s_idx in zip(idxs_strict[:-1], idxs_strict[1:]):
        xt = student_step_xt_to_xs(xt, int(t_idx), int(s_idx))

    # At the end, xt is x_0 (or close)
    return xt

# -----------------------------
# Evaluate KL for K = 1..T
# -----------------------------
@torch.no_grad()
def eval_kl_curve(n_true=200000, n_gen=60000, bins=200, K_max=None, mu=2.0, sigma_data=0.5):
    """
    Returns:
      Ks: list
      KLs: list of KL(p_true || p_gen)
      bins_edges: numpy array
      p_true: histogram prob
      p_gens: dict K -> histogram prob (for plotting)
    """
    # True samples
    x_true = sample_gmm(n_true, mu=mu, sigma_data=sigma_data, device=device).cpu().numpy().flatten()

    # bin edges fixed from true range (+ small padding)
    lo, hi = np.percentile(x_true, 0.1), np.percentile(x_true, 99.9)
    pad = 0.2 * (hi - lo)
    bins_edges = np.linspace(lo - pad, hi + pad, bins + 1)

    p_true = hist_prob(x_true, bins_edges)

    if K_max is None:
        K_max = T_steps - 1  # "T" in the diffusion sense (number of transitions)

    Ks = list(range(1, K_max + 1))
    KLs = []
    p_gens = {}

    for K in Ks:
        x_gen = sample_student_k_steps(n=n_gen, K=K).cpu().numpy().flatten()
        p_gen = hist_prob(x_gen, bins_edges)
        p_gens[K] = p_gen
        KLs.append(kl_divergence(p_true, p_gen))

    return Ks, KLs, bins_edges, p_true, p_gens

Ks, KLs, bins_edges, p_true, p_gens = eval_kl_curve(
    n_true=200000,
    n_gen=60000,
    bins=220,
    K_max=T_steps - 1,   # 1..(T_steps-1)
    mu=2.0,
    sigma_data=0.5
)

# -----------------------------
# Plot KL vs steps
# -----------------------------
plt.figure(figsize=(7,4))
plt.plot(Ks, KLs)
plt.title("KL divergence vs number of sampling steps K")
plt.xlabel("K (number of steps)")
plt.ylabel("KL(p_true || p_gen)")
plt.grid(True, alpha=0.2)
plt.show()

print(f"Best (min) KL = {min(KLs):.6f} at K = {Ks[int(np.argmin(KLs))]}")

# -----------------------------
# Show histogram fit improving for selected K's
# -----------------------------
def plot_selected_fits(selected_Ks):
    centers = 0.5 * (bins_edges[:-1] + bins_edges[1:])
    plt.figure(figsize=(8,4))
    plt.plot(centers, p_true, label="true", linewidth=2)
    for K in selected_Ks:
        plt.plot(centers, p_gens[K], label=f"gen K={K} (KL={KLs[K-1]:.3f})", alpha=0.9)
    plt.title("Histogram fit improves as K increases")
    plt.xlabel("x")
    plt.ylabel("probability mass (hist)")
    plt.grid(True, alpha=0.2)
    plt.legend()
    plt.show()

# pick a few Ks to visualize (adjust as you like)
K_max = T_steps - 1
selected = sorted(set([1, 2, 4, 8, 16, 32, K_max]))
selected = [k for k in selected if k <= K_max]
plot_selected_fits(selected)

The above graph seems **counterary to logic** in that the *KL* gets worse as we increase the number of steps. Howevever, becuase the student was trained mainly to satisfy
$
f_\theta(x_t,t)\approx x_0^{\text{teacher}},
$
then **one-step sampling** is exactly its native use case.

When we do **multi-step sampling**, we are using the student **off-policy**:

- After step 1, the intermediate $x$ values come from the student’s **own errors**.
- The student may **never have seen** those exact intermediate states during training.
- Errors **accumulate**, and the iterative chain can drift toward a **stable but wrong fixed point**.

That’s why increasing $K$ (more steps) doesn’t necessarily “improve toward truth.”

## Teacher → Student: Diffusion → Consistency Distillation for MNIST

We show a **minimal, trainable** example of **teacher–student distillation** from a continuous-time **diffusion teacher** to a **consistency student**.

### What we build
1. A **diffusion teacher** that learns to predict noise $\epsilon$ from noisy inputs $x_t$ (continuous-time).
2. A **consistency student** $f_\theta(x_t,t)$ trained so that its outputs are **consistent across time**:
   $$
   f_\theta(x_t,t) \approx f_\theta(x_s,s), \quad 0 \le s < t \le 1.
   $$
3. A **one-step sampler** using the student: sample noise at $t=1$, then map to an image at $t=0$.

### Key continuous-time forward process (simple VP-style parameterization)

We use:
- $\alpha(t) = \cos\left(\frac{\pi}{2}t\right)$
- $\sigma(t) = \sin\left(\frac{\pi}{2}t\right)$
- so $\alpha(t)^2 + \sigma(t)^2 = 1$

Noising:
$
x_t = \alpha(t)\,x_0 + \sigma(t)\,\epsilon,\quad \epsilon \sim \mathcal{N}(0,I).
$

Teacher predicts $\hat{\epsilon}_\phi(x_t,t)$ and we form:
$
\hat{x}_0 = \frac{x_t - \sigma(t)\,\hat{\epsilon}_\phi(x_t,t)}{\alpha(t)}.
$

Teacher-defined step:
$
p(x_s \mid x_t,t,s) = \int q(x_s\mid x_0,s)\,p_\phi(x_0\mid x_t,t)\,dx_0
$

We approximate by sampling:
$
x_s = \alpha(s)\,\hat{x}_0 + \sigma(s)\,z,\quad z\sim\mathcal{N}(0,I).
$

Student losses:
$
\mathcal{L}_{\text{cons}} = \mathbb{E}\left[\|f_\theta(x_t,t) - f_\theta(x_s,s)\|_2^2\right],
\quad
\mathcal{L}_{0} = \mathbb{E}\left[\|f_\theta(x_0,0) - x_0\|_2^2\right].
$

### Constraints
- Designed to run in **reasonable time** using a small CNN and a subset of data.
- Uses **MNIST via torchvision** if available; otherwise falls back to **synthetic blobs**.

In [ ]:
!pip install matplotlib


In [ ]:
import math
import random
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import torchvision
    import torchvision.transforms as T
    HAVE_TORCHVISION = True
except Exception:
    HAVE_TORCHVISION = False

import matplotlib.pyplot as plt

seed = 1337
random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)
print("torchvision available:", HAVE_TORCHVISION)

## Data: MNIST 

We normalize images to $[-1,1]$ and downsample to $16\times 16$ for speed.

In [ ]:

from torch.utils.data import DataLoader, Dataset, Subset

IMG_SIZE = 28          # 16 for speed; change to 28 for full MNIST
BATCH_SIZE = 128
TRAIN_SUBSET = 60000   # reduce for CPU (e.g., 5000)

class SyntheticBlobs(Dataset):
    def __init__(self, n=10000, size=16):
        self.n = n
        self.size = size
        self.rng = torch.Generator().manual_seed(1234)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        H = W = self.size
        img = torch.zeros(H, W)
        k = int(torch.randint(2, 5, (1,), generator=self.rng).item())
        ys = torch.arange(H).float().unsqueeze(1)
        xs = torch.arange(W).float().unsqueeze(0)
        for _ in range(k):
            cx = int(torch.randint(0, W, (1,), generator=self.rng).item())
            cy = int(torch.randint(0, H, (1,), generator=self.rng).item())
            sig = float(torch.randint(1, 4, (1,), generator=self.rng).item())
            blob = torch.exp(-((xs-cx)**2 + (ys-cy)**2) / (2*sig*sig))
            img += blob
        img = img.clamp(0, 1)
        img = img * 2 - 1  # [-1,1]
        return img.unsqueeze(0), 0

def get_dataset():
    if HAVE_TORCHVISION:
        try:
            tfm = T.Compose([
                T.Resize(IMG_SIZE),
                T.ToTensor(),
                T.Lambda(lambda x: x*2 - 1),
            ])
            ds = torchvision.datasets.MNIST(root="./data_mnist", train=True, download=True, transform=tfm)
            idx = torch.randperm(len(ds))[:TRAIN_SUBSET].tolist()
            ds = Subset(ds, idx)
            return ds, "MNIST"
        except Exception as e:
            print("MNIST download/load failed, using synthetic blobs. Error:", e)

    return SyntheticBlobs(n=TRAIN_SUBSET, size=IMG_SIZE), "SyntheticBlobs"

train_ds, ds_name = get_dataset()
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
print("dataset:", ds_name, "| batches:", len(train_loader))

xb, _ = next(iter(train_loader))
plt.figure(figsize=(6,2))
for i in range(6):
    plt.subplot(1,6,i+1)
    plt.imshow(((xb[i,0].cpu()+1)/2).numpy(), cmap="gray")
    plt.axis("off")
plt.suptitle(f"Examples from {ds_name}")
plt.show()

### Continuous-time schedule: $\alpha(t), \sigma(t)$

$$
\alpha(t) = \cos\left(\frac{\pi}{2}t\right),\quad
\sigma(t) = \sin\left(\frac{\pi}{2}t\right).
$$

Noising:
$
x_t = \alpha(t)\,x_0 + \sigma(t)\,\epsilon,\quad \epsilon \sim \mathcal{N}(0,I).
$

In [ ]:

def alpha_sigma(t: torch.Tensor):
    a = torch.cos(0.5 * math.pi * t)
    s = torch.sin(0.5 * math.pi * t)
    return a, s

def sample_xt(x0: torch.Tensor, t: torch.Tensor):
    eps = torch.randn_like(x0)
    a, s = alpha_sigma(t)
    a = a.view(-1,1,1,1)
    s = s.view(-1,1,1,1)
    xt = a * x0 + s * eps
    return xt, eps

### Time embedding (sinusoidal)

In [ ]:

def sinusoidal_time_embedding(t: torch.Tensor, dim: int):
    half = dim // 2
    freqs = torch.exp(
        -math.log(10000) * torch.arange(0, half, device=t.device).float() / max(half - 1, 1)
    )
    args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
    if dim % 2 == 1:
        emb = F.pad(emb, (0, 1))
    return emb

### Tiny CNN backbone

- Teacher: predicts $\hat{\epsilon}_\phi(x_t,t)$
- Student: predicts $f_\theta(x_t,t) \approx x_0$

In [ ]:

class TinyTimeCNN(nn.Module):
    def __init__(self, in_ch=1, base=64, time_dim=128, out_ch=1):
        super().__init__()
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, base),
            nn.SiLU(),
            nn.Linear(base, base),
        )
        self.conv1 = nn.Conv2d(in_ch, base, 3, padding=1)
        self.conv2 = nn.Conv2d(base, base, 3, padding=1)
        self.conv3 = nn.Conv2d(base, base, 3, padding=1)
        self.conv4 = nn.Conv2d(base, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, base)
        self.norm2 = nn.GroupNorm(8, base)
        self.norm3 = nn.GroupNorm(8, base)

    def forward(self, x, t):
        temb = sinusoidal_time_embedding(t, 128)
        temb = self.time_mlp(temb).view(-1, 64, 1, 1)

        h = self.conv1(x)
        h = self.norm1(h + temb)
        h = F.silu(h)

        h = self.conv2(h)
        h = self.norm2(h + temb)
        h = F.silu(h)

        h = self.conv3(h)
        h = self.norm3(h + temb)
        h = F.silu(h)

        out = self.conv4(h)
        return out

teacher = TinyTimeCNN(out_ch=1).to(device)
student = TinyTimeCNN(out_ch=1).to(device)

print("Teacher params (M):", sum(p.numel() for p in teacher.parameters())/1e6)
print("Student params (M):", sum(p.numel() for p in student.parameters())/1e6)

### Train the diffusion teacher (noise prediction)

$
\mathcal{L}_{\text{teacher}} = \mathbb{E}\left[\|\hat{\epsilon}_\phi(x_t,t) - \epsilon\|_2^2\right].
$

In [ ]:

def train_teacher(steps=40000, lr=2e-4, print_every=5000):
    teacher.train()
    opt = torch.optim.AdamW(teacher.parameters(), lr=lr, weight_decay=1e-4)

    it = iter(train_loader)
    t0 = time.time()
    for step in range(1, steps+1):
        try:
            x0, _ = next(it)
        except StopIteration:
            it = iter(train_loader)
            x0, _ = next(it)

        x0 = x0.to(device)
        t = torch.rand(x0.size(0), device=device) * 0.999 + 0.001
        xt, eps = sample_xt(x0, t)

        eps_pred = teacher(xt, t)
        loss = F.mse_loss(eps_pred, eps)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(teacher.parameters(), 1.0)
        opt.step()

        if step % print_every == 0:
            dt = time.time() - t0
            print(f"[teacher] step {step:5d}/{steps} | loss {loss.item():.4f} | elapsed {dt:.1f}s")

train_teacher(steps=100000, lr=2e-4, print_every=5000)

### Teacher reconstruction: compute $\hat{x}_0$ from $x_t$

$
\hat{x}_0 = \frac{x_t - \sigma(t)\hat{\epsilon}_\phi(x_t,t)}{\alpha(t)}.
$

In [ ]:

@torch.no_grad()
def teacher_x0_hat(xt, t):
    eps_hat = teacher(xt, t)
    a, s = alpha_sigma(t)
    a = a.view(-1,1,1,1)
    s = s.view(-1,1,1,1)
    x0_hat = (xt - s * eps_hat) / a
    return x0_hat.clamp(-1, 1)

teacher.eval()
x0, _ = next(iter(train_loader))
x0 = x0.to(device)[:8]

t = torch.ones(x0.size(0), device=device) * 0.8
xt, _ = sample_xt(x0, t)
x0_hat = teacher_x0_hat(xt, t)

plt.figure(figsize=(8,3))
for i in range(8):
    plt.subplot(2,8,i+1)
    plt.imshow(((x0[i,0].cpu()+1)/2).numpy(), cmap="gray")
    plt.axis("off")
    plt.subplot(2,8,8+i+1)
    plt.imshow(((x0_hat[i,0].cpu()+1)/2).numpy(), cmap="gray")
    plt.axis("off")
plt.suptitle("Top: x0 | Bottom: teacher x0_hat from xt at t=0.8")
plt.show()

### Consistency distillation (student)

Teacher-defined step:
$
x_s = \alpha(s)\hat{x}_0 + \sigma(s)z,\quad z\sim\mathcal{N}(0,I).
$

Student output:
$
f_\theta(x_t,t)\approx x_0.
$

Losses:
$
\mathcal{L}_{\text{cons}}=\|f_\theta(x_t,t)-f_\theta(x_s,s)\|_2^2,\quad
\mathcal{L}_0=\|f_\theta(x_0,0)-x_0\|_2^2.
$

In [ ]:
# train_student will distill teacher's x0_hat, so we want to ensure it's reasonable before starting.

for p in teacher.parameters():
    p.requires_grad_(False)
teacher.eval()

@torch.no_grad()
def teacher_x0_hat_safe(xt, t, eps=1e-3):
    """Teacher x0_hat, but avoid division blow-ups when alpha(t) is tiny."""
    eps_hat = teacher(xt, t)
    a, s = alpha_sigma(t)
    a = a.clamp(min=eps).view(-1,1,1,1)
    s = s.view(-1,1,1,1)
    x0_hat = (xt - s * eps_hat) / a
    return x0_hat.clamp(-1, 1)

@torch.no_grad()
def teacher_step_xt_to_xs(xt, t, s):
    x0_hat = teacher_x0_hat_safe(xt, t)
    z = torch.randn_like(xt)
    a_s, s_s = alpha_sigma(s)
    a_s = a_s.view(-1,1,1,1)
    s_s = s_s.view(-1,1,1,1)
    xs = a_s * x0_hat + s_s * z
    return xs, x0_hat

def train_student(
    steps=10000,
    lr=2e-4,
    lam_cons=1.0,
    lam_bdry=1.0,
    lam_distill=1.0,
    print_every=250,
):
    student.train()
    opt = torch.optim.AdamW(student.parameters(), lr=lr, weight_decay=1e-4)

    it = iter(train_loader)
    t0 = time.time()

    for step in range(1, steps+1):
        try:
            x0, _ = next(it)
        except StopIteration:
            it = iter(train_loader)
            x0, _ = next(it)

        x0 = x0.to(device)

        # IMPORTANT: avoid t extremely close to 1 early on
        t = torch.rand(x0.size(0), device=device) * 0.95 + 0.02   # t in [0.02, 0.97]
        s = torch.rand_like(t) * (t - 0.01)                      # s < t

        xt, _ = sample_xt(x0, t)

        with torch.no_grad():
            xs, x0_hat_T = teacher_step_xt_to_xs(xt, t, s)

        # student predictions
        x0_t = student(xt, t)
        x0_s = student(xs, s)

        # distill to teacher x0_hat (prevents collapse)
        loss_distill = F.mse_loss(x0_t, x0_hat_T)

        # consistency (one-way) to avoid both sides drifting together
        loss_cons = F.mse_loss(x0_s, x0_t.detach())

        # boundary/anchor at t=0
        tzero = torch.zeros(x0.size(0), device=device)
        loss_bdry = F.mse_loss(student(x0, tzero), x0)

        loss = lam_distill*loss_distill + lam_cons*loss_cons + lam_bdry*loss_bdry

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        opt.step()

        if step % print_every == 0:
            dt = time.time() - t0
            print(f"[student] {step:5d}/{steps} | dist {loss_distill.item():.4f} | "
                  f"cons {loss_cons.item():.4f} | bdry {loss_bdry.item():.4f} | "
                  f"total {loss.item():.4f} | {dt:.1f}s")


# call it
train_student(steps=100000, lr=2e-4, lam_cons=0.5, lam_bdry=1.0, lam_distill=1.0, print_every=5000)

### One-step sampling with the student

Sample:
1. $x_1 \sim \mathcal{N}(0,I)$
2. $\hat{x}_0 = f_\theta(x_1,1)$

In [ ]:

@torch.no_grad()
def sample_student(n=16):
    student.eval()
    x1 = torch.randn(n, 1, IMG_SIZE, IMG_SIZE, device=device)
    t1 = torch.ones(n, device=device)
    x0_hat = student(x1, t1).clamp(-1, 1)
    return x0_hat

samples = sample_student(16)

plt.figure(figsize=(6,6))
for i in range(16):
    plt.subplot(4,4,i+1)
    plt.imshow(((samples[i,0].cpu()+1)/2).numpy(), cmap="gray")
    plt.axis("off")
plt.suptitle("Student one-step samples (t=1 -> x0)")
plt.show()

### Trying different levels of sampling (multistep)

Here we try different values of T for each intitial guess. 

In [ ]:
import math
import torch
import matplotlib.pyplot as plt

@torch.no_grad()
def sample_student_multistep(n=16, n_steps=8, eta=0.0):
    """
    Multi-step sampling from t=1 -> t=0 using the student consistency model.

    - Start with x_{t=1} ~ N(0, I)
    - For a decreasing schedule t_i -> t_{i+1}, do:
        x0_hat = student(x_{t_i}, t_i)
        eps_hat = (x_{t_i} - alpha(t_i)*x0_hat) / sigma(t_i)
        x_{t_{i+1}} = alpha(t_{i+1})*x0_hat + sigma(t_{i+1})*eps_hat   (deterministic)
      Optionally add stochasticity via eta > 0.

    Returns:
      x0_hats: list of denoised estimates x0_hat at each step (including final)
      ts: tensor of times used (length n_steps+1)
    """
    student.eval()

    # Time schedule from 1 down to 0 (inclusive)
    ts = torch.linspace(1.0, 0.0, n_steps + 1, device=device)

    # Start from pure noise at t=1
    x = torch.randn(n, 1, IMG_SIZE, IMG_SIZE, device=device)

    x0_hats = []

    for i in range(n_steps):
        t = ts[i].expand(n)        # current time
        s = ts[i + 1].expand(n)    # next (smaller) time

        # Student predicts a clean image estimate at time t
        x0_hat = student(x, t).clamp(-1, 1)
        x0_hats.append(x0_hat)

        # Convert that to an implied noise estimate eps_hat
        a_t, sig_t = alpha_sigma(t)
        a_t = a_t.view(-1, 1, 1, 1)
        sig_t = sig_t.view(-1, 1, 1, 1)
        eps_hat = (x - a_t * x0_hat) / (sig_t + 1e-8)

        # Step to time s using a DDIM-like deterministic update
        a_s, sig_s = alpha_sigma(s)
        a_s = a_s.view(-1, 1, 1, 1)
        sig_s = sig_s.view(-1, 1, 1, 1)

        if eta > 0.0:
            # Add controlled stochasticity
            z = torch.randn_like(x)
            eps_mix = math.sqrt(1 - eta**2) * eps_hat + eta * z
            x = a_s * x0_hat + sig_s * eps_mix
        else:
            x = a_s * x0_hat + sig_s * eps_hat

    # At the end, x is at t=0; include the final clean estimate
    # (when s=0, sigma(0)=0, so x becomes x0_hat anyway)
    x0_hats.append(x.clamp(-1, 1))

    return x0_hats, ts


def show_progression(x0_hats, ts, n_show=8):
    """
    Visualize how samples improve across steps.
    Rows: time steps (from t=1 down to t=0)
    Cols: different samples (first n_show)
    """
    n_steps = len(x0_hats) - 1  # last is final
    n_show = min(n_show, x0_hats[0].size(0))

    plt.figure(figsize=(1.2*n_show, 1.2*(n_steps+1)))
    for r in range(n_steps + 1):
        # pick the denoised estimate at this step
        imgs = x0_hats[r]
        t_val = float(ts[min(r, len(ts)-1)].item())  # best-effort label
        for c in range(n_show):
            plt.subplot(n_steps + 1, n_show, r*n_show + c + 1)
            plt.imshow(((imgs[c, 0].cpu() + 1) / 2).numpy(), cmap="gray")
            plt.axis("off")
            if c == 0:
                plt.ylabel(f"t={t_val:.2f}", rotation=0, labelpad=30, va="center")
    plt.suptitle("Student multi-step sampling: denoised estimates improve as t ↓", y=1.02)
    plt.tight_layout()
    plt.show()


# ---- Run and visualize ----
x0_hats, ts = sample_student_multistep(n=16, n_steps=24, eta=0.0)  # eta=0 deterministic
show_progression(x0_hats, ts, n_show=8)  # show first 8 samples across all steps

## Summary Comparison Table

> Notation (generic):
> - Data: $x\in\mathbb{R}^D$, latent: $z\in\mathbb{R}^d$
> - Time: $t\in[0,1]$ (or $\{1,\dots,T\}$)
> - Score: $s_t(x)=\nabla_x \log p_t(x)$
> - Velocity (wind): $v_t(x)$
> - Condition (prompt/label/etc.): $c$

| Family / Variant | What the NN estimates (primary object) | Sampling mechanism | “Hard parameter” / what’s difficult | Typical tradeoffs (pros / cons) |
|---|---|---|---|---|
| **VAE (standard)** | Encoder posterior $q_\phi(z\mid x)$ (often $\mu_\phi(x),\log\sigma^2_\phi(x)$) and decoder likelihood $p_\theta(x\mid z)$ | Sample $z\sim p(z)$ then decode $x\sim p_\theta(x\mid z)$ | Approximating posterior $q_\phi(z\mid x)$ and choosing a good likelihood model $p_\theta(x\mid z)$; ELBO gap; posterior collapse | **+** Very fast sampling (1 pass), latent manipulation; **−** likelihood mismatch can blur, posterior collapse, tuning ELBO/KL is tricky |
| **VAE (latent for other models)** | Same as above, but used to define a latent space $z=E(x)$ / $x=D(z)$ | Other model runs in $z$, then decode | Learning a latent that is both reconstructive and “nice” for downstream generative modeling | **+** Enables large speedups for diffusion/flow/consistency; **−** reconstruction bottleneck can cap fidelity |
| **Diffusion (unguided)** | Usually predicts noise $\varepsilon_\theta(x_t,t)$ (or score-like, or $x_0$, or $v$) | Reverse SDE/ODE steps from $x_T\sim \mathcal{N}(0,I)$ | Implicitly estimating a **score-like** object $\nabla_x \log p_t(x)$ (via denoising). Needs many steps for high quality | **+** High quality, stable training; **−** sampling cost (many steps), careful schedules/solvers |
| **Diffusion (guided; classifier guidance)** | Same base model + separate classifier gradient $\nabla_x \log p(c\mid x_t)$ | Reverse with extra guidance term added to score | Need a classifier that works on noisy $x_t$; computing/using gradients can be unstable | **+** Strong controllability; **−** extra model, gradient costs, can reduce diversity / introduce artifacts |
| **Diffusion (guidance-free; classifier-free guidance, CFG)** | Two predictions: conditional $\varepsilon_\theta(x_t,t,c)$ and unconditional $\varepsilon_\theta(x_t,t,\varnothing)$ combined as $\varepsilon_{\text{CFG}}=\varepsilon_{\text{uncond}} + w(\varepsilon_{\text{cond}}-\varepsilon_{\text{uncond}})$ | Same reverse steps but with CFG-combined prediction | Still relies on score-like estimation; plus tuning guidance scale $w$ and training “dropout conditioning” | **+** Strong control without external classifier; **−** extra compute (two forward passes), guidance tuning, can oversharpen / reduce diversity |
| **Flow Matching (unguided)** | Velocity field $v_\theta(x,t)\in\mathbb{R}^D$ (or in latent $\mathbb{R}^d$) | Integrate ODE $dx/dt=v_\theta(x,t)$ from $x(0)\sim p_0$ | Must learn a globally consistent vector field; quality depends on coupling/bridge choice; ODE solver error | **+** Deterministic sampling, often fewer steps; avoids explicit score estimation; **−** bridge/coupling design matters, ODE solver cost/accuracy |
| **Flow Matching (guided; external guidance)** | Base $v_\theta(x,t,c)$ plus extra gradient term from a reward/classifier (implementation-dependent) | ODE integration with an added guidance component | Need guidance gradients or auxiliary models; stability vs diversity similar to diffusion guidance | **+** Control; **−** extra models/gradients, tuning guidance strength |
| **Flow Matching (guidance-free; CFG-style)** | Conditional and unconditional velocities: $v_\theta(x,t,c)$ and $v_\theta(x,t,\varnothing)$ combined as $v_{\text{CFG}}=v_{\text{uncond}} + w(v_{\text{cond}}-v_{\text{uncond}})$ | Integrate ODE using $v_{\text{CFG}}$ | Guidance tuning $w$; two model evaluations per step if done literally | **+** Control without extra classifier; **−** higher compute and guidance tuning; possible diversity loss |
| **Consistency (no teacher; CT)** | Canonical mapper $f_\theta(x_t,t)\approx x_0$ (or a canonical representation) | Few-step or 1-step mapping from $x_T$ to $x_0$ | **Identifiability / collapse**: consistency loss admits constant solutions; needs strong anchors and careful training | **+** Extremely fast sampling (1–few steps); **−** collapse risk, harder to train reliably from scratch |
| **Consistency (with teacher; distillation)** | Student $f_\theta$ trained to match teacher denoising behavior (teacher is diffusion/flow/strong model) | 1–few step student sampling | Dependence on teacher quality + distillation design; but avoids the worst collapse modes | **+** Fast sampling with high quality; more stable than teacher-free CT; **−** requires training/obtaining a teacher model |
| **Latent Diffusion (any diffusion variant in latent)** | Same as diffusion but in latent: $\varepsilon_\theta(z_t,t,(\,c\,))$ or $v_\theta$ etc. | Reverse steps in $z$ then decode $x=D(z)$ | Need a good autoencoder; still score-like estimation (but in lower dimension) | **+** Big speed/memory gains; **−** fidelity bounded by AE + still multi-step |
| **Latent Flow Matching (any flow variant in latent)** | Velocity field in latent $v_\theta(z,t,(\,c\,))$ | Integrate $dz/dt=v_\theta(z,t)$ then decode | Bridge/coupling choice in latent; latent quality matters | **+** Often very fast with good quality; avoids explicit score; **−** depends on latent geometry + solver |
| **Latent Consistency (any consistency variant in latent)** | Canonical mapper in latent $f_\theta(z_t,t)\approx z_1$ (or decode after) | 1–few step in latent then decode | Collapse risk without teacher; AE bottleneck | **+** Fastest pipelines; **−** training stability + AE limits |

## What each method is “trying to estimate”

- **VAE:** learns an *approximate posterior* $q_\phi(z\mid x)$ and a *decoder likelihood* $p_\theta(x\mid z)$.  
- **Diffusion:** learns a *denoiser* (often $\varepsilon_\theta$) that is equivalent to a *score-like* object $\nabla_x \log p_t(x)$.  
- **Flow Matching:** learns a *velocity / wind* field $v_\theta(x,t)$ directly (no explicit score).  
- **Consistency:** learns a *canonical map* $f_\theta(x_t,t)$ that should be invariant across time; **teacher-free** risks collapse, **teacher-based** anchors the target.

## Where “difficulty” typically lives

- **VAE:** posterior approximation + likelihood mismatch (ELBO gap), potential posterior collapse.  
- **Diffusion:** score-like estimation across many noise levels + expensive sampling steps; guidance needs careful tuning.  
- **Flow Matching:** picking a good bridge/coupling and learning a globally consistent vector field; ODE solver accuracy.  
- **Consistency:** identifiability/collapse without a teacher; with teacher the main difficulty shifts to distillation and teacher quality.

## Further Readings

| Category | What the neural network estimates | Key paper|
|---|---|---|
| **VAE** | Encoder posterior parameters $q_\phi(z\mid x)$(e.g., $\mu_\phi(x),\log\sigma_\phi^2(x)$) and decoder likelihood $p_\theta(x\mid z)$ | **Kingma & Welling (2013)** — *Auto-Encoding Variational Bayes* (https://arxiv.org/abs/1312.6114)|
| **Diffusion (Unguided)** | Typically noise predictor $\varepsilon_\theta(x_t,t)$ (equivalent to learning a score-like object)| **Ho, Jain & Abbeel (2020)** — *Denoising Diffusion Probabilistic Models* (https://arxiv.org/abs/2006.11239)|
| **Diffusion (Guided: classifier guidance)** | Base diffusion model + external classifier gradient $\nabla_x\log p(c\mid x_t)$ added at sampling time| **Dhariwal & Nichol (2021)** — *Diffusion Models Beat GANs on Image Synthesis* (https://dl.acm.org/doi/10.5555/3540261.3540933)|
| **Diffusion (Guidance-Free: classifier-free guidance)** | Two predictions $\varepsilon_\theta(x_t,t,c)$ and $\varepsilon_\theta(x_t,t,\varnothing)$ combined with a guidance scale | **Ho & Salimans (2022)** — *Classifier-Free Diffusion Guidance* (https://arxiv.org/abs/2207.12598)|
| **Flow Matching (Unguided)** | Velocity (wind) field $v_\theta(x,t)\in\mathbb{R}^D$ regressed from path-defined targets| **Lipman et al. (2022)** — *Flow Matching for Generative Modeling* (https://arxiv.org/abs/2210.02747)|
| **Flow Matching (Guided: general guidance framework)** | Adds/derives guidance terms for FM (training-free and/or training-based guidance on top of $v_\theta$) | **Feng et al. (2025)** — *On the Guidance of Flow Matching* (https://arxiv.org/abs/2502.02150)|
| **Flow Matching (Guidance-Free: CFG for FM)** | Conditional/unconditional velocities $v_\theta(x,t,c)$ and $v_\theta(x,t,\varnothing)$ combined in a CFG-like way| **Zheng et al. (2023)** — *Guided Flows for Generative Modeling and Decision Making* (https://arxiv.org/abs/2311.13443)|
| **Consistency (with Teacher / Distillation)** | Student consistency map $f_\theta(x_t,t)\approx x_0$ distilled from a strong teacher (often diffusion) :contentReference[oaicite:14]{index=14} | **Song et al. (2023)** — *Consistency Models* (https://arxiv.org/abs/2303.01469)|
| **Consistency (without Teacher / Standalone CT)** | Directly trained $f_\theta(x_t,t)$ with improved objectives/schedules to avoid relying on distillation| **Song & Dhariwal (2023)** — *Improved Techniques for Training Consistency Models* (https://arxiv.org/abs/2310.14189)|
| **Latent Diffusion** | Diffusion model in latent space (e.g., $\varepsilon_\theta(z_t,t,c)$) + decoder back to $x$ | **Rombach et al. (2021/2022)** — *High-Resolution Image Synthesis with Latent Diffusion Models* (https://arxiv.org/abs/2112.10752)|
| **Latent Flow Matching** | Velocity field in latent space $v_\theta(z,t)$; decode $z(1)$ to $x$| **Dao et al. (2023)** — *Flow Matching in Latent Space* (https://arxiv.org/abs/2307.08698)|
| **Latent Consistency** | Consistency map in latent space (few-step / fast inference for LDMs), typically distilled| **Luo et al. (2023)** — *Latent Consistency Models: Synthesizing High-Resolution Images with Few-Step Inference* (https://arxiv.org/abs/2310.04378) |

## Flow Matching Variants

A useful way to organize most flow-matching / rectified-flow / stochastic-interpolant style methods is by **which lever they change**.

### Lever 0: the basic formulation

Start distribution:
$
x_0 \sim p_0
$

Target distribution:
$
x_1 \sim p_1
$

Pick time:
$
t \sim \mathrm{Uniform}[0,1]
$

**Baseline bridge (linear interpolant):**
$$
x_t = (1-t)x_0 + t x_1
$$

**Baseline target velocity:**
$$
u_t = \frac{d x_t}{dt} = x_1 - x_0
$$

**Flow matching loss:**
$$
\mathcal{L}(\theta)=\mathbb{E}\left[\|v_\theta(x_t,t)-u_t\|_2^2\right].
$$

### Lever 1: change how you pick / pair $x_0$ and $x_1$ (the *coupling*)

#### What stays the same
Often the bridge remains:
$$
x_t = (1-t)x_0 + t x_1,
\qquad
u_t = x_1 - x_0.
$$

#### What changes
Instead of sampling $x_0$ and $x_1$ independently, you sample from a **coupling**:
$$
(x_0,x_1)\sim \pi,
\qquad \text{where marginals of }\pi \text{ are } p_0 \text{ and } p_1.
$$

- **Naive coupling:** $\pi = p_0 \otimes p_1$ (independent pairing)
- **Smarter coupling:** choose $\pi$ to reduce crossing paths / make transport easier (e.g., OT-style couplings)

#### Why it matters
Changing $\pi$ changes which pairs occur, hence changes the training distribution over:
$$
(x_t,t,u_t).
$$

So the model sees different “wind supervision” and learns a different (often easier, more consistent) vector field.

### Lever 2: change the bridge $x_t=\dots$ (the *interpolant*)

#### What stays the same
You still learn a velocity field:
$$
v_\theta(x,t)
$$
by supervised regression to a target velocity:
$$
u_t=\frac{d x_t}{dt}.
$$

#### What changes
You replace the linear bridge with a more general (often nonlinear) interpolant:
$$
x_t = I(x_0,x_1,t).
$$

Then the target velocity becomes:
$$
u_t = \frac{\partial}{\partial t} I(x_0,x_1,t).
$$

Examples of “bridge changes”:
- time-dependent scaling of endpoints,
- nonlinear curves in space,
- stochastic interpolants (where $x_t$ may include noise)

#### Why it matters
Changing the bridge changes both:
- the locations $x_t$ at which the model is queried,
- and the supervision target $u_t$.

This can make trajectories smoother, reduce solver steps, or improve quality.

### Lever 3: “other” common modifications (parameterization / regularization / solver / distillation)

These changes don’t always fit perfectly into “coupling” or “bridge,” but they are widespread.

#### Change what the network outputs (parameterization)
Instead of predicting raw velocity, you may predict a transformed velocity:
$$
\tilde{v}_\theta(x,t)=A(t)\,v_\theta(x,t)
$$
or model a flow map directly (a global transformation rather than instantaneous velocity).

#### Add trajectory-shaping regularization
Encourage easier-to-integrate dynamics, e.g. “straighter” trajectories or lower curvature:
$$
\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{FM}} + \lambda\,\mathcal{R}(v_\theta),
$$
where $$\mathcal{R}$$ might penalize curvature, stiffness, or large divergence.

####  Distillation (fewer steps / faster sampling)
Train a strong multi-step model then distill to 1–2 steps:
$$
\text{student} \leftarrow \text{match teacher’s transport in fewer evaluations}.
$$

#### Solver choice
Even with the same $v_\theta$, using Euler vs Heun vs RK4 changes sample quality vs speed.

### Summary

A clean organizing statement:

> Most flow-matching variants can be understood as modifying either  
> **(1)** the *coupling* $\pi(x_0,x_1)$ that determines how start/end points are paired, or  
> **(2)** the *bridge* $x_t=I(x_0,x_1,t)$ that defines where and how we supervise velocities,  
> plus **(3)** practical tweaks (parameterization, regularization, solvers, distillation) that improve stability and efficiency.

## When reading a new paper

When you encounter a “new” flow matching method, ask:

1. **Coupling:** Did they change how $x_0$ pairs with $x_1$? (OT/minibatch OT/learned pairing)
2. **Bridge:** Did they replace $x_t=(1-t)x_0+t x_1$ with a different $I(x_0,x_1,t)$?
3. **Other:** Did they change parameterization, add regularizers, distill, or change the solver?

## A Transport-based Synopsis: VAE, Diffusion, Flow Matching, OTF, and Consistency

### One unifying lens: **transport** (“move probability mass from simple to data”)

All these generative methods can be seen as learning a way to push an easy base distribution into the data distribution:

- Base: $p_0$(often Gaussian noise)
- Target: $p_1 = p_{\text{data}}$
- Goal: learn a transformation (stochastic or deterministic) such that:
$
x_0 \sim p_0 \quad \Rightarrow \quad x_1 = \mathcal{T}(x_0) \sim p_1.
$

They differ mainly in **what transport object they learn**:

- **VAE:** latent transport map through a bottleneck + decoder
- **Diffusion:** score/denoiser that defines reverse-time transport
- **Flow Matching:** velocity (wind) field defining an ODE transport
- **OTF (Optimal Transport Flow):** flow matching + smarter coupling/geometry for transport
- **Consistency:** a direct few-step (sometimes one-step) transport map, often distilled from a teacher

### VAE: transport via latent bottleneck + decoder map

#### Transport view
A VAE learns:
- encoder posterior $q_\phi(z\mid x)$ (inverse/approx posterior),
- decoder likelihood $p_\theta(x\mid z)$ (generative map).

Sampling:
$
z \sim p(z) \quad \Rightarrow \quad x \sim p_\theta(x\mid z).
$

Interpretation: the decoder acts as a **transport map** from a simple latent base $p(z)$ into data space.

#### Best use cases
- **Fast sampling** (one forward pass)
- **Representation learning** (latents are usable/controllable)
- **Backbone for latent generative models** (latent diffusion / latent flow / latent consistency)

#### Limitations
- Likelihood/reconstruction choices can blur fine details
- Posterior collapse / ELBO tradeoffs
- Latent quality bounds downstream sample fidelity

### Diffusion: transport via gradual noising + reverse-time denoising (score-driven)

#### Transport view
Diffusion defines a forward transport (data → noise) and learns the reverse (noise → data).

Forward:
$
x_0 \sim p_{\text{data}} \;\to\; x_T \approx \mathcal{N}(0,I).
$

Reverse sampling uses a learned denoiser (often $\varepsilon_\theta(x_t,t)$), which is equivalent to learning a **score-like** object:
$
s_t(x) = \nabla_x \log p_t(x).
$

#### Best use cases
- **Highest sample quality** and robustness
- Strong controllability (CFG / guidance)
- When compute allows multi-step sampling

#### Limitations
- Many sampling steps (cost)
- Guidance tuning and solver choices
- “Score-like” learning is indirect and time-dependent

### Flow Matching: transport via a learned velocity field (ODE wind)

#### Transport view
Flow matching learns a velocity field:
$$
v_\theta(x,t)\in\mathbb{R}^D,
$$
and sampling integrates the ODE:
$$
\frac{dx}{dt} = v_\theta(x,t),\qquad t\in[0,1].
$$

If learned correctly:
$
x(0)\sim p_0 \quad \Rightarrow \quad x(1)\sim p_1.
$

#### Best use cases
- ODE-based sampling (often fewer steps than diffusion)
- Clear physical interpretation (“wind”)
- Works very well in **latent space** (latent flow matching)

#### Limitations
- Depends on bridge/coupling choice
- ODE solver error vs speed tradeoff
- Poor pairing can cause crossing paths and harder learning

### OTF / Optimal Transport Flow: flow matching with smarter coupling/geometry

#### Transport view
Flow matching requires a coupling $\pi(x_0,x_1)$ between base and data.

Naive coupling:
$
\pi = p_0\otimes p_1 \quad \text{(independent pairing)}.
$

OT-inspired coupling tries to choose $\pi$ that makes transport easier (less crossing, more coherent global rearrangement).

#### Best use cases
- When independent pairing yields messy dynamics
- When you want “straighter” / more efficient transport
- Often improves quality for a fixed number of function evaluations (NFEs)

#### Limitations
- OT approximations add complexity/cost
- Coupling is approximate in minibatches and can be noisy

### Consistency models: direct few-step transport map (fast sampler)

#### Transport view
Consistency models learn a canonical mapper:
$$
f_\theta(x_t,t) \approx x_0,
$$
so sampling can be one-step or a few steps:
$$
x_T\sim \mathcal{N}(0,I),\qquad \hat{x}_0 = f_\theta(x_T,T).
$$

Two regimes:

#### (A) Without teacher (direct CT)
- Risks **collapse** because invariance constraints can admit trivial solutions.

#### (B) With teacher (distillation)
- A strong teacher (diffusion or flow) anchors correct targets, yielding stable training and high quality with few steps.

#### Best use cases
- **Real-time / edge / interactive** generation (1–4 steps)
- Distilling a high-quality teacher into a cheap sampler

#### Limitations
- Teacher-free training can collapse if not anchored strongly
- Teacher-based requires a teacher + distillation pipeline
- One-step is hardest; often multi-step then distill

### How they relate (who plugs into whom)

#### The “stacking” view
- **VAE** (or AE) gives: $x \leftrightarrow z$  
  Then you can run transport in latent:
  - **latent diffusion:** $\varepsilon_\theta(z_t,t)$
  - **latent flow:** $v_\theta(z,t)$
  - **latent consistency:** $f_\theta(z_t,t)$ 
  Finally decode: $\hat{x}=D(z).$

#### Teacher → Student view
- **Diffusion or Flow** can serve as a **teacher**
- **Consistency** can be a **student** distilled to few steps

#### Geometry view
- **OTF** improves the coupling/geometry inside flow-style learning, often reducing solver steps.

### Comparative quick guide (best use cases)

- **VAE alone:** fastest; good representations; may blur
- **Diffusion:** highest quality; slower; guidance is strong
- **Flow matching:** ODE transport; often fewer steps; bridge matters
- **OTF:** improved flow matching via smarter coupling; efficiency gains
- **Consistency:** fastest sampling; best when distilled from a strong teacher
- **Latent versions:** almost always better compute-quality tradeoff for images/audio (depends on latent quality)

### Publishable ideas (research directions)

Below are research stacks you can turn into experiments/papers. Each is phrased as a hypothesis + what to measure. **NFE**= Number of Function Evaluations. So NFE is simply: how many times you call the network during sampling (or during an ODE solve); lower is better. 

####  1. **AE/VAE Latents + OTF Flow Matching + Consistency Distillation**
**Pipeline**
1) Train AE/VAE on images to obtain latents $z$ 
2) Train **OT-guided flow matching** in latent space (smarter coupling)  
3) Distill the latent-flow teacher into a **1–2 step latent consistency** student  
4) Decode to image

**Publishable angle**
- “OT coupling reduces path crossing → fewer NFEs → better distillation”
- Measure: NFE vs FID/precision/recall; distillation stability; collapse frequency

####  2. **Flow Matching vs Diffusion as Teachers for Consistency**
**Pipeline**
- Train a diffusion teacher and a flow-matching teacher of similar quality
- Distill both into equal-capacity consistency students (1–4 steps)

**Publishable angle**
- “Which teacher distills better at the same compute?”
- Measure: student quality vs steps; robustness to guidance scales; training stability

#### 3.  **Guidance-free Flow Matching (CFG-style) in Latent Space**
**Pipeline**
- Train conditional/unconditional latent velocity fields:
$$
v_\theta(z,t,c),\quad v_\theta(z,t,\varnothing)
$$
- Combine at sampling:
$$
v_{\text{CFG}} = v_{\text{uncond}} + w\,(v_{\text{cond}}-v_{\text{uncond}})
$$

**Publishable angle**
- CFG for flow matching: “control vs diversity vs NFE”
- Measure: controllability metrics; diversity; guidance stability; compare to diffusion CFG

#### 4. **Nonlinear bridges (stochastic interpolants) in latent flow matching**
**Pipeline**
- Replace linear bridge $$z_t=(1-t)z_0+t z_1$$ with a nonlinear/stochastic interpolant
- Train and compare to linear bridge FM

**Publishable angle**
- “Bridge choice changes learnability and solver stiffness”
- Measure: stiffness/curvature proxies; NFE required; sample quality; ablations on interpolant parameters

#### 5. **Flow-map matching / shortcut flows for few-step generation**
**Pipeline**
- Learn maps $\Phi_{t\to s}$ directly (global transform), then compose a few maps to sample
- Compare to learning instantaneous velocity

**Publishable angle**
- “Map learning improves few-step performance and stability”
- Measure: quality at 1–4 steps; runtime; stability across seeds/datasets

#### 6. **Discrete (token-like) latent flow / consistency for structured outputs**
**Pipeline**
- Use a VQ-style discrete latent (codebook) or categorical latent
- Train flow-like transport in relaxed space, then distill to few steps

**Publishable angle**
- “Bridging continuous flow tools to discrete generation”
- Measure: discrete validity, quality, and speed; compare to diffusion on discrete spaces

## What makes a strong paper here?
- A clear compute-quality tradeoff curve: **quality vs NFEs vs parameters**
- Ablations showing which lever helps: coupling (OT), bridge (nonlinear), teacher choice, distillation objective
- Stability analysis: collapse/instability rates and mitigation
- Latent bottleneck study: AE vs VAE vs stronger perceptual AEs

## Use of Generative AI

Portions of this material were developed with the assistance of **generative artificial intelligence tools.  
The author reviewed, edited, and validated all content, including explanations, code, and examples, and assumes full responsibility for accuracy and interpretation.